<a href="https://colab.research.google.com/github/MWANIKID/Technical-Indicators-Journal-Article/blob/main/Leakage-Free%20Hybrid%20Stacking%20of%20TGARCH-X%20and%20Transformer%20Volatility%20Forecasts_Technical_Indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# STRICT T4 GPU BOOTSTRAP — MUST BE THE FIRST BLOCK IN A FRESH RUNTIME
# ================================================================
import os
import sys
import subprocess
import random
import gc
import warnings
import time

SEED = 42

# Remove any stale CPU-only restriction left by an earlier script.
_previous_cuda_visible = os.environ.pop("CUDA_VISIBLE_DEVICES", None)
if _previous_cuda_visible is not None:
    print(
        "ℹ️ Removed stale CUDA_VISIBLE_DEVICES="
        f"{_previous_cuda_visible!r} before importing TensorFlow."
    )

# These variables must be defined before TensorFlow is imported.
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# A stale TensorFlow import cannot be repaired safely inside the same
# Python process if it was initialised while the GPU was hidden.
_TF_WAS_ALREADY_IMPORTED = "tensorflow" in sys.modules

# Confirm that Colab actually attached an NVIDIA GPU before importing TF.
try:
    _nvidia_check = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=20,
        check=False
    )
except (FileNotFoundError, subprocess.TimeoutExpired) as error:
    raise RuntimeError(
        "No NVIDIA runtime is available. Select T4 GPU, click Save, then "
        "choose Runtime → Disconnect and delete runtime before rerunning."
    ) from error

if _nvidia_check.returncode != 0 or not _nvidia_check.stdout.strip():
    raise RuntimeError(
        "Colab has not attached an NVIDIA GPU to this runtime.\n"
        f"nvidia-smi output: {_nvidia_check.stderr.strip()}\n"
        "Select T4 GPU, click Save, then choose "
        "Runtime → Disconnect and delete runtime and reconnect."
    )

print("✅ NVIDIA runtime detected:")
print("   " + _nvidia_check.stdout.strip().replace("\n", "\n   "))

if _TF_WAS_ALREADY_IMPORTED:
    import tensorflow as tf

    _already_visible_gpus = tf.config.list_physical_devices("GPU")
    if not _already_visible_gpus:
        raise RuntimeError(
            "TensorFlow was already imported in this Python process while "
            "the GPU was hidden or unavailable. The GPU cannot be restored "
            "safely without replacing the runtime.\n\n"
            "Use Runtime → Disconnect and delete runtime, reconnect with "
            "T4 GPU selected, and run this script as the first TensorFlow code."
        )

    print(
        "⚠️ TensorFlow was already imported. A GPU is visible, so execution "
        "can continue, but use a fresh runtime for the definitive reproducible run."
    )
else:
    import tensorflow as tf

import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit

# Keep this methodological version unchanged so the enhanced code accepts
# the baseline CSV outputs. Record the GPU execution mode separately.
BASELINE_CODE_VERSION = "baseline_light_step5_convex_v1_05_oneday"
REPRODUCIBILITY_VERSION = "deterministic_t4_gpu_float32_v2"

# Require a visible TensorFlow GPU.
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "nvidia-smi can see a GPU, but TensorFlow cannot. This usually means "
        "TensorFlow was initialised earlier with CUDA disabled. Use "
        "Runtime → Disconnect and delete runtime, reconnect, and run this "
        "script first."
    )

# Colab supplies one GPU. Restrict TensorFlow to the first device and enable
# memory growth before creating any tensors or models.
try:
    tf.config.set_visible_devices(gpus[0], "GPU")
except RuntimeError:
    # The device list may already be fixed when TF was imported earlier.
    if not _TF_WAS_ALREADY_IMPORTED:
        raise

try:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    _MEMORY_GROWTH_STATUS = "enabled"
except RuntimeError as error:
    if _TF_WAS_ALREADY_IMPORTED:
        _MEMORY_GROWTH_STATUS = (
            "already initialised; fresh runtime required for strict control"
        )
        print(f"⚠️ GPU memory-growth warning: {error}")
    else:
        raise RuntimeError(
            "TensorFlow initialised the GPU before memory growth was configured. "
            "Use Runtime → Disconnect and delete runtime and run this script first."
        ) from error

# Deterministic seeds and execution.
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as error:
    raise RuntimeError(
        "TensorFlow could not enable deterministic GPU operations."
    ) from error

tf.keras.mixed_precision.set_global_policy("float32")
tf.config.optimizer.set_jit(False)

gpu_details = tf.config.experimental.get_device_details(gpus[0])
GPU_NAME = gpu_details.get("device_name", gpus[0].name)

if "T4" not in GPU_NAME.upper():
    raise RuntimeError(
        f"Expected an NVIDIA T4 for the definitive run, but TensorFlow found "
        f"{GPU_NAME!r}. Select T4 GPU and replace the runtime."
    )

# Force one small operation onto the GPU and verify its placement.
with tf.device("/GPU:0"):
    _gpu_probe = tf.linalg.matmul(
        tf.ones((2, 2), dtype=tf.float32),
        tf.ones((2, 2), dtype=tf.float32)
    )

if "GPU" not in _gpu_probe.device.upper():
    raise RuntimeError(
        "TensorFlow detected a GPU but the verification operation did not run "
        f"on it. Device used: {_gpu_probe.device}"
    )

del _gpu_probe
gc.collect()

print(f"✅ TensorFlow GPU detected: {GPU_NAME}")
print(f"✅ Verification operation device: /GPU:0")
print(f"✅ GPU memory growth: {_MEMORY_GROWTH_STATUS}")
print("✅ Deterministic TensorFlow/cuDNN/cuBLAS settings enabled")
print("✅ Precision policy: float32")
print(f"✅ Reproducibility version: {REPRODUCIBILITY_VERSION}")

# ================================================================
# 📥 Upload CSVs (Master + TGARCH Forecasts)
# ================================================================
from google.colab import files

print("\n📥 Upload exactly one Master CSV and one TGARCH-X forecast CSV.")
uploaded = files.upload()

# ================================================================
# 🎯 QLIKE Loss
# ================================================================
@tf.function
def qlike_loss(y_true, y_pred, epsilon=1e-8):
    y_true = tf.clip_by_value(y_true, epsilon, 1e10)
    y_pred = tf.clip_by_value(y_pred, epsilon, 1e10)
    ratio  = y_true / y_pred
    return tf.reduce_mean(ratio - tf.math.log(ratio) - 1.0)

def qlike_np(y_true, y_pred, epsilon=1e-8):
    y_true = np.clip(np.asarray(y_true, dtype=float), epsilon, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), epsilon, None)
    ratio  = y_true / y_pred
    return np.mean(ratio - np.log(ratio) - 1.0)


# ================================================================
# 📄 Load Data
# ================================================================
print("============================================================")
print("FILE DETECTION")
print("============================================================")
print(f"Files uploaded: {len(uploaded)}")
for filename in uploaded.keys():
    print(f"   - {filename}")
print()

all_uploaded = list(uploaded.keys())
csv_files    = [f for f in all_uploaded if f.lower().endswith('.csv')]

# Master file: prefer names containing 'master'; fallback to largest CSV
main_files = [f for f in all_uploaded if "master" in f.lower()]
if not main_files:
    main_files = [f for f in all_uploaded if "nse" in f.lower() and "data" in f.lower()]
if not main_files:
    main_files = [f for f in all_uploaded if "price" in f.lower() or "stock" in f.lower()]

# TGARCH file: prefer names containing 'tgarch' or 'garch'; fallback to 'forecast'
tgarch_files = [f for f in all_uploaded if "tgarch" in f.lower() or "garch" in f.lower()]
if not tgarch_files:
    tgarch_files = [f for f in all_uploaded if "forecast" in f.lower()]

# Last resort: if exactly two CSVs uploaded, assign by exclusion
if not main_files and not tgarch_files and len(csv_files) == 2:
    tgarch_files = [csv_files[0]]
    main_files   = [csv_files[1]]
elif not main_files and tgarch_files and len(csv_files) == 2:
    main_files   = [f for f in csv_files if f not in tgarch_files]
elif not tgarch_files and main_files and len(csv_files) == 2:
    tgarch_files = [f for f in csv_files if f not in main_files]

print("File Detection Results:")
print(f"   Master files found:  {len(main_files)}")
if main_files:   print(f"      Using: {main_files[0]}")
print(f"   TGARCH files found:  {len(tgarch_files)}")
if tgarch_files: print(f"      Using: {tgarch_files[0]}")
print(f"   All uploaded files:  {all_uploaded}")
print()

if not main_files or not tgarch_files:
    raise FileNotFoundError(
        "❌ Could not identify Master and TGARCH files from uploaded names.\n"
        f"   Uploaded: {all_uploaded}\n"
        "   Please rename your files so the Master data file contains 'Master' or 'master'\n"
        "   and the TGARCH forecast file contains 'TGARCH' or 'tgarch' in the filename."
    )

main_file   = main_files[0]
tgarch_file = tgarch_files[0]

df        = pd.read_csv(main_file)
tgarch_df = pd.read_csv(tgarch_file)

print("✅ All required files loaded successfully!")
print("============================================================\n")

for frame in [df, tgarch_df]:
    frame.columns = (
        frame.columns.str.strip()
        .str.replace(" ", "_")
        .str.replace("(", "")
        .str.replace(")", "")
    )

df = df.rename(columns={
    'Market_Capitalization_KES': 'Market_Cap', 'Category': 'Sector'
})

# Preserve an unfiltered market-cap source for sector weighting.
# This copy is independent of later Return and Trading_Volume filtering.
weight_source = df[
    ['Date', 'Sector', 'Stock', 'Market_Cap']
].copy()

tgarch_fc_candidates = [
    c for c in tgarch_df.columns
    if 'forecast' in c.lower() and 'var' in c.lower()
]
if not tgarch_fc_candidates:
    tgarch_fc_candidates = [
        c for c in tgarch_df.columns if 'forecast' in c.lower()
    ]
tgarch_forecast_col = tgarch_fc_candidates[0]
tgarch_df = tgarch_df.rename(
    columns={tgarch_forecast_col: 'TGARCH_Variance'}
)

print(f"Column Detection:")
print(f"   TGARCH forecast: {tgarch_forecast_col} → 'TGARCH_Variance'")
print()

df['Date']        = pd.to_datetime(df['Date'],        dayfirst=True, errors='coerce')
tgarch_df['Date'] = pd.to_datetime(tgarch_df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values(['Stock', 'Date'])

print("============================================================")
print("STACKING: TRANSFORMER + TGARCH (ALL SECTORS)")
print("Following Wolpert (1992) & Breiman (1996)")
print("============================================================\n")

# ================================================================
# 🔁 Compute Returns
# ================================================================
df['Market_Cap']     = pd.to_numeric(
    df['Market_Cap'].astype(str).str.replace(',', ''), errors='coerce'
)

# Use lagged market-capitalisation weights so weights for day t are known
# at the end of day t-1.
df['Market_Cap_Lag1'] = df.groupby('Stock')['Market_Cap'].shift(1)

# Preserve a target-only source before Trading_Volume and model-feature
# filtering so the observed variance cannot change with the model.
target_source = df[
    ['Date', 'Sector', 'Stock', 'Close', 'Market_Cap']
].copy()
target_source['Close'] = pd.to_numeric(
    target_source['Close'], errors='coerce'
)

# Clean the independent weighting source using only fields required
# for market-capitalisation weighting.
weight_source['Date'] = pd.to_datetime(
    weight_source['Date'], dayfirst=True, errors='coerce'
)
weight_source['Market_Cap'] = pd.to_numeric(
    weight_source['Market_Cap'].astype(str).str.replace(',', ''),
    errors='coerce'
)
weight_source = weight_source.dropna(
    subset=['Date', 'Sector', 'Stock', 'Market_Cap']
)
weight_source = weight_source[
    weight_source['Market_Cap'] > 0
].copy()

df['Trading_Volume'] = pd.to_numeric(
    df['Volume'].astype(str).str.replace(',', ''), errors='coerce'
)
df['Return'] = df.groupby('Stock')['Close'].transform(
    lambda x: np.log(x / x.shift(1))
)
df = df.dropna(
    subset=['Return', 'Market_Cap', 'Market_Cap_Lag1', 'Trading_Volume']
)
df = df[df['Market_Cap_Lag1'] > 0].copy()

# ================================================================
# 📈 Sector-Weighted Returns
# ================================================================
def compute_sector_weighted(group):
    group = group.copy()
    cap_sum = group['Market_Cap_Lag1'].sum()
    if cap_sum <= 0 or not np.isfinite(cap_sum):
        return pd.Series({
            'Sector_Return':     np.nan,
            'Sector_Volume_Sum': group['Trading_Volume'].sum()
        })
    w = group['Market_Cap_Lag1'] / cap_sum
    return pd.Series({
        'Sector_Return':     np.sum(w * group['Return']),
        'Sector_Volume_Sum': group['Trading_Volume'].sum()
    })

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sector_returns = (
        df.groupby(['Date', 'Sector'], group_keys=False)
        .apply(compute_sector_weighted, include_groups=False)
        .reset_index()
        .dropna()
    )

sector_returns['Log_Volume'] = np.log1p(sector_returns['Sector_Volume_Sum'])

# ================================================================
# 📊 Volatility Features
# ================================================================
sector_returns = sector_returns.sort_values(
    ['Sector', 'Date']
).reset_index(drop=True)

sector_returns['Vol_5']  = sector_returns.groupby('Sector')[
    'Sector_Return'
].transform(lambda x: x.rolling(5).std())
sector_returns['Vol_10'] = sector_returns.groupby('Sector')[
    'Sector_Return'
].transform(lambda x: x.rolling(10).std())
sector_returns['Vol_5_Lag1']  = sector_returns.groupby(
    'Sector'
)['Vol_5'].shift(1)
sector_returns['Vol_10_Lag1'] = sector_returns.groupby(
    'Sector'
)['Vol_10'].shift(1)
sector_returns = sector_returns.dropna(
    subset=['Vol_5_Lag1', 'Vol_10_Lag1', 'Log_Volume']
).reset_index(drop=True)

print(f"✅ Sector features computed: {len(sector_returns)} observations\n")

# ================================================================
# 📅 Load TGARCH
# ================================================================
tgarch_clean = tgarch_df[['Date', 'Sector', 'TGARCH_Variance']].copy()
tgarch_clean['TGARCH_Variance'] = pd.to_numeric(
    tgarch_clean['TGARCH_Variance'], errors='coerce'
)
tgarch_clean = tgarch_clean.dropna(subset=['TGARCH_Variance'])
tgarch_clean['TGARCH_Variance'] = tgarch_clean['TGARCH_Variance'].clip(
    lower=1e-8
)

print(f"✅ TGARCH forecasts loaded: {len(tgarch_clean)} observations\n")

# ================================================================
# ✅ FIX 4: Fixed one-day common evaluation target from raw master data
# ================================================================
print(
    "📊 Computing one-day squared-return target "
    "from raw master data..."
)

epsilon = 1e-8

target_source = target_source.dropna(
    subset=['Date', 'Sector', 'Stock', 'Close', 'Market_Cap']
).copy()
target_source = target_source[
    target_source['Market_Cap'] > 0
].copy()

if target_source.duplicated(
    ['Sector', 'Stock', 'Date']
).any():
    raise ValueError(
        "Duplicate Sector-Stock-Date rows detected in target source."
    )

target_source = target_source.sort_values(
    ['Sector', 'Stock', 'Date']
).reset_index(drop=True)

target_source['Stock_Return'] = (
    target_source.groupby(
        ['Sector', 'Stock']
    )['Close'].transform(
        lambda prices: np.log(
            prices / prices.shift(1)
        )
    )
)
target_source['Market_Cap_Lag1'] = (
    target_source.groupby(
        ['Sector', 'Stock']
    )['Market_Cap'].shift(1)
)

target_source = target_source.dropna(
    subset=['Stock_Return', 'Market_Cap_Lag1']
).copy()
target_source = target_source[target_source['Market_Cap_Lag1'] > 0].copy()

target_source['Weighted_Return_Component'] = (
    target_source['Market_Cap_Lag1']
    * target_source['Stock_Return']
)

sector_target_returns = (
    target_source.groupby(
        ['Sector', 'Date'],
        as_index=False
    )
    .agg(
        Weighted_Return_Sum=(
            'Weighted_Return_Component',
            'sum'
        ),
        Sector_Market_Cap_Lag1=(
            'Market_Cap_Lag1',
            'sum'
        )
    )
)

sector_target_returns['Sector_Return'] = (
    sector_target_returns['Weighted_Return_Sum']
    / sector_target_returns['Sector_Market_Cap_Lag1']
)

sector_target_returns = (
    sector_target_returns
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['Sector_Return'])
    .sort_values(['Sector', 'Date'])
    .reset_index(drop=True)
)

sector_target_returns['Eval_Variance'] = (
    sector_target_returns['Sector_Return'] ** 2
)

eval_target_df = (
    sector_target_returns[
        ['Date', 'Sector', 'Eval_Variance']
    ]
    .dropna(subset=['Eval_Variance'])
    .copy()
)

eval_target_df['Eval_Variance'] = (
    eval_target_df['Eval_Variance']
    .clip(lower=epsilon)
)

if eval_target_df.duplicated(
    ['Sector', 'Date']
).any():
    raise ValueError(
        "Duplicate Sector-Date rows detected in common target."
    )

# Use the exact same target for Transformer training.
sector_returns = (
    sector_returns.merge(
        eval_target_df[
            ['Date', 'Sector', 'Eval_Variance']
        ],
        on=['Date', 'Sector'],
        how='inner',
        validate='one_to_one'
    )
    .rename(
        columns={
            'Eval_Variance': 'Realized_Variance'
        }
    )
    .dropna(
        subset=['Realized_Variance']
    )
    .sort_values(['Sector', 'Date'])
    .reset_index(drop=True)
)

print(
    f"✅ Common one-day target merged into Transformer data: "
    f"{len(sector_returns)} observations"
)

print(
    f"✅ Common evaluation target computed: "
    f"{len(eval_target_df)} observations across "
    f"{eval_target_df['Sector'].nunique()} sectors"
)
for s in sorted(eval_target_df['Sector'].unique()):
    n = len(
        eval_target_df[
            eval_target_df['Sector'] == s
        ]
    )
    print(f"   {s}: {n} observations")
print()

common_target_output = eval_target_df.rename(
    columns={
        'Eval_Variance': 'Actual_Variance'
    }
).copy()

common_target_output.to_csv(
    "Common_Actual_Variance_1Day_Squared_Return.csv",
    index=False
)

print(
    "✅ Saved: "
    "Common_Actual_Variance_1Day_Squared_Return.csv\n"
)

del target_source, sector_target_returns

# ================================================================
# 🧠 TRANSFORMER MODEL
# ================================================================
def positional_encoding(seq_len, d_model):
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(
        np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model)
    )
    pos_enc          = np.zeros((seq_len, d_model))
    pos_enc[:, 0::2] = np.sin(position * div_term)
    pos_enc[:, 1::2] = np.cos(position * div_term)
    return tf.cast(pos_enc[np.newaxis, :, :], dtype=tf.float32)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.mha        = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.ffn        = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(d_model)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1   = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2   = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1        = self.layernorm1(inputs + attn_output)
        ffn_output  = self.ffn(out1, training=training)
        ffn_output  = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_transformer_qlike(input_shape,
                             d_model=32,
                             num_heads=4,
                             dff=64,
                             num_layers=1,
                             dropout_rate=0.2):
    tf.keras.utils.set_random_seed(SEED)   # ✅ locks weight initialisers per build
    seq_len, num_features = input_shape
    inputs       = tf.keras.Input(shape=input_shape)
    x            = tf.keras.layers.Dense(d_model)(inputs)
    pos_encoding = positional_encoding(seq_len, d_model)
    x            = x + pos_encoding
    for _ in range(num_layers):
        x = TransformerBlock(d_model, num_heads, dff, dropout_rate)(x)
    x       = tf.keras.layers.GlobalAveragePooling1D()(x)
    x       = tf.keras.layers.Dense(64, activation='relu')(x)
    x       = tf.keras.layers.Dropout(dropout_rate)(x)
    x       = tf.keras.layers.Dense(32, activation='relu')(x)
    x       = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(
        1, activation='softplus', dtype='float32'
    )(x)
    model   = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=qlike_loss
    )
    return model

def fast_predict(model, x):
    return model(x, training=False)

def reset_weights(m):
    for layer in m.layers:
        if hasattr(layer, 'kernel_initializer') and hasattr(layer, 'kernel'):
            layer.kernel.assign(
                layer.kernel_initializer(tf.shape(layer.kernel))
            )
        if hasattr(layer, 'bias_initializer') and hasattr(layer, 'bias'):
            if layer.bias is not None:
                layer.bias.assign(
                    layer.bias_initializer(tf.shape(layer.bias))
                )
        if hasattr(layer, 'layers'):
            for sublayer in layer.layers:
                if hasattr(sublayer, 'kernel_initializer') and \
                        hasattr(sublayer, 'kernel'):
                    sublayer.kernel.assign(
                        sublayer.kernel_initializer(
                            tf.shape(sublayer.kernel)
                        )
                    )
                if hasattr(sublayer, 'bias_initializer') and \
                        hasattr(sublayer, 'bias'):
                    if sublayer.bias is not None:
                        sublayer.bias.assign(
                            sublayer.bias_initializer(
                                tf.shape(sublayer.bias)
                            )
                        )
        if hasattr(layer, 'mha'):
            for w in layer.mha.weights:
                if 'kernel' in w.name:
                    w.assign(
                        tf.keras.initializers.GlorotUniform(
                            seed=SEED
                        )(tf.shape(w))
                    )
                elif 'bias' in w.name:
                    w.assign(tf.zeros_initializer()(tf.shape(w)))

# ================================================================
# ⚙️ Parameters
# ================================================================
target               = 'Realized_Variance'
features             = ['Vol_5_Lag1', 'Vol_10_Lag1', 'Log_Volume']

SEQ_LEN              = 20
WINDOW_SIZE          = 200
STEP_SIZE            = 5
EPOCHS               = 20
BATCH_SIZE           = 64
TRAIN_RATIO          = 0.80
VAL_RATIO            = 0.20
PATIENCE             = 3

CAL_RATIO            = 0.25
STACK_TRAIN_END      = 0.65
CV_SPLITS            = 3
CONVEX_WEIGHT_GRID   = np.linspace(0.0, 1.0, 101)
MIN_SAMPLES_STACKING = 10

# ================================================================
# 🔁 STAGE 1: TRANSFORMER BASE FORECASTING
# ================================================================
print("============================================================")
print("STAGE 1: TRANSFORMER BASE FORECASTING")
print("============================================================")
print(f"   Architecture: 1×TransformerBlock(heads=4, d_model=32, dff=64) →")
print(f"                 GlobalAvgPool → Dense(64) → Dense(32)")
print(f"   ✅ Dropout inactive during inference")
print(f"   ✅ One model fitted per sector")
print(f"   ✅ Scaling from the pre-test sample only")
print(f"   ✅ STEP_SIZE={STEP_SIZE} retained")
print(f"   ✅ One-day squared-return target from raw master data")
print(f"   ✅ Fully reproducible (deterministic ops + float32 + per-build seed)")
print("============================================================\n")

dl_forecasts_all = []

for sector in tqdm(
    sector_returns['Sector'].unique(), desc="Transformer Forecasting"
):
    s_df = sector_returns[
        sector_returns['Sector'] == sector
    ].reset_index(drop=True)

    if len(s_df) < 100:
        print(f"⚠️  Skipping {sector}: insufficient data")
        continue

    X_raw     = s_df[features].values
    y         = s_df[target].values
    dates_all = s_df['Date'].values

    raw_train_end = int(len(X_raw) * TRAIN_RATIO)
    train_mean    = X_raw[:raw_train_end].mean(axis=0)
    train_std     = X_raw[:raw_train_end].std(axis=0) + epsilon
    X_scaled      = (X_raw - train_mean) / train_std

    X_seq = np.array([
        X_scaled[i - SEQ_LEN:i]
        for i in range(SEQ_LEN, len(X_scaled))
    ])
    y_seq     = y[SEQ_LEN:]
    dates_seq = dates_all[SEQ_LEN:]

    T          = len(X_seq)
    test_start = int(T * TRAIN_RATIO)

    val_size   = max(10, int(test_start * VAL_RATIO))
    X_subtrain = X_seq[:test_start - val_size]
    y_subtrain = y_seq[:test_start - val_size]
    X_val      = X_seq[test_start - val_size:test_start]
    y_val      = y_seq[test_start - val_size:test_start]

    if len(X_subtrain) < 30 or len(X_val) < 10:
        print(f"⚠️  Skipping {sector}: train/validation sample too small")
        continue

    tf.keras.backend.clear_session()
    gc.collect()
    model = build_transformer_qlike((SEQ_LEN, len(features)))

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, min_delta=1e-6
    )

    model.fit(
        X_subtrain, y_subtrain,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        verbose=0, shuffle=False, callbacks=[early_stop]
    )

    test_indices = np.arange(test_start, T, STEP_SIZE)
    if len(test_indices) == 0:
        del model
        tf.keras.backend.clear_session()
        gc.collect()
        print(f"⚠️  {sector}: no test indices generated — skipping")
        continue

    forecasts_raw = fast_predict(
        model,
        tf.constant(X_seq[test_indices], dtype=tf.float32)
    ).numpy().reshape(-1).astype(float)

    dates = pd.to_datetime(dates_seq[test_indices])

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    raw_forecast_df = pd.DataFrame({
        'Date':            dates,
        'Sector':          sector,
        'Raw_DL_Forecast': forecasts_raw
    })

    aligned = (
        raw_forecast_df.merge(
            eval_target_df[['Sector', 'Date', 'Eval_Variance']],
            on=['Sector', 'Date'],
            how='inner'
        )
        .sort_values('Date')
        .reset_index(drop=True)
    )

    if len(aligned) == 0:
        print(f"⚠️  {sector}: no matching evaluation dates — skipping")
        continue

    aligned = aligned.rename(
        columns={'Eval_Variance': 'Actual_Variance'}
    )
    aligned['Actual_Variance'] = aligned['Actual_Variance'].clip(
        lower=epsilon
    )

    aligned['DL_Forecast']      = aligned['Raw_DL_Forecast'].clip(lower=epsilon)
    aligned['Calibration_CF']   = np.nan
    aligned['Evaluation_Split'] = 'Pending_Exact_Alignment'

    for _, row in aligned.iterrows():
        dl_forecasts_all.append({
            'Date':             row['Date'],
            'Sector':           sector,
            'DL_Model':         'Transformer',
            'Raw_DL_Forecast':  row['Raw_DL_Forecast'],
            'DL_Forecast':      row['DL_Forecast'],
            'Actual_Variance':  row['Actual_Variance'],
            'Calibration_CF':   row['Calibration_CF'],
            'Evaluation_Split': row['Evaluation_Split']
        })

    print(f"   ✅ {sector}: {len(aligned)} forecasts generated")

    # Release sector-specific host and GPU memory before the next sector.
    del (
        s_df, X_raw, y, dates_all, X_scaled, X_seq, y_seq, dates_seq,
        X_subtrain, y_subtrain, X_val, y_val, test_indices,
        forecasts_raw, dates, raw_forecast_df, aligned
    )
    tf.keras.backend.clear_session()
    gc.collect()

dl_forecasts_df = pd.DataFrame(dl_forecasts_all)
print(f"\n✅ Total Transformer forecasts: {len(dl_forecasts_df)}\n")

# ================================================================
# 🔗 MERGE TRANSFORMER + TGARCH
# ================================================================
print("\n============================================================")
print("MERGING TRANSFORMER + TGARCH FORECASTS")
print("============================================================\n")

hybrid_records = []

for sector in dl_forecasts_df['Sector'].unique():
    dl_sector     = dl_forecasts_df[
        dl_forecasts_df['Sector'] == sector
    ].copy()
    tgarch_sector = tgarch_clean[
        tgarch_clean['Sector'] == sector
    ].copy().rename(columns={'TGARCH_Variance': 'GARCH_Variance'})

    merged = (
        dl_sector.merge(
            tgarch_sector[['Date', 'Sector', 'GARCH_Variance']],
            on=['Date', 'Sector'], how='inner'
        )
        .sort_values('Date')
        .reset_index(drop=True)
    )

    if len(merged) == 0:
        print(f"⚠️  {sector}: No overlapping DL + TGARCH dates")
        continue

    merged = (
        merged.merge(
            eval_target_df[['Date', 'Sector', 'Eval_Variance']],
            on=['Date', 'Sector'], how='inner'
        )
        .sort_values('Date')
        .reset_index(drop=True)
    )

    if len(merged) == 0:
        print(f"⚠️  {sector}: No dates matching common evaluation target")
        continue

    merged['Actual_Variance'] = merged['Eval_Variance'].clip(lower=epsilon)
    merged = merged.drop(columns=['Eval_Variance'])
    merged['GARCH_Model'] = 'TGARCH'

    n_merged  = len(merged)
    cal_end   = max(int(n_merged * CAL_RATIO), 3)
    stack_end = max(int(n_merged * STACK_TRAIN_END), cal_end + 5)

    if stack_end >= n_merged or (n_merged - stack_end) < 5:
        print(f"⚠️  {sector}: split too small after exact alignment "
              f"(n={n_merged}, cal={cal_end}, stack_end={stack_end})")
        continue

    cf = np.mean(
        merged.loc[:cal_end - 1, 'Actual_Variance'].values /
        (merged.loc[:cal_end - 1, 'Raw_DL_Forecast'].values + epsilon)
    )
    merged['DL_Forecast'] = (
        cf * merged['Raw_DL_Forecast']
    ).clip(lower=epsilon)
    merged['Calibration_CF']  = cf
    merged['Evaluation_Split'] = 'Calibration'
    merged.loc[
        cal_end:stack_end - 1, 'Evaluation_Split'
    ] = 'Stacking_Train'
    merged.loc[
        stack_end:, 'Evaluation_Split'
    ] = 'Final_Test'
    merged['Code_Version'] = BASELINE_CODE_VERSION
    merged['Reproducibility_Version'] = REPRODUCIBILITY_VERSION
    merged['Execution_Device'] = GPU_NAME

    hybrid_records.append(merged)
    n_train = len(merged[merged['Evaluation_Split'] == 'Stacking_Train'])
    n_test  = len(merged[merged['Evaluation_Split'] == 'Final_Test'])
    print(f"   ✅ {sector:<35} | {len(merged)} forecasts "
          f"(train={n_train}, test={n_test})")

hybrid_df = pd.concat(hybrid_records, ignore_index=True)
hybrid_df['DL_Forecast']     = hybrid_df['DL_Forecast'].clip(lower=epsilon)
hybrid_df['GARCH_Variance']  = hybrid_df['GARCH_Variance'].clip(lower=epsilon)
hybrid_df['Actual_Variance'] = hybrid_df['Actual_Variance'].clip(lower=epsilon)

total_train = len(hybrid_df[hybrid_df['Evaluation_Split'] == 'Stacking_Train'])
total_test  = len(hybrid_df[hybrid_df['Evaluation_Split'] == 'Final_Test'])
print(f"\n✅ Total merged: {len(hybrid_df)} "
      f"(train={total_train}, test={total_test})\n")

# ================================================================
# ⚖️ Sector Weights — computed from exact Final_Test dates only
# ================================================================
final_test_dates = set(
    hybrid_df.loc[
        hybrid_df['Evaluation_Split'] == 'Final_Test', 'Date'
    ].unique()
)
df_test_period = weight_source[
    weight_source['Date'].isin(final_test_dates)
].copy()

if len(df_test_period) == 0:
    raise ValueError("No master-data rows match the exact Final_Test dates.")

daily_sector_cap = df_test_period.groupby(
    ['Date', 'Sector']
)['Market_Cap'].sum().reset_index()
daily_sector_cap.columns = ['Date', 'Sector', 'Sector_Cap']

daily_total_cap = df_test_period.groupby(
    'Date'
)['Market_Cap'].sum().reset_index()
daily_total_cap.columns = ['Date', 'Total_Cap']

daily_weights = daily_sector_cap.merge(daily_total_cap, on='Date')
daily_weights['W_st'] = (
    daily_weights['Sector_Cap'] / daily_weights['Total_Cap']
)

sector_avg_weights = daily_weights.groupby(
    'Sector'
)['W_st'].mean().to_dict()
all_sectors    = hybrid_df['Sector'].unique()
sector_weights = {s: sector_avg_weights.get(s, 0) for s in all_sectors}
total_w        = sum(sector_weights.values())
if total_w <= 0:
    raise ValueError("Sector weights sum to zero.")
sector_weights = {k: v / total_w for k, v in sector_weights.items()}

print(
    "\n⚖️ Sector Weights "
    "(average daily market-cap share over Final_Test dates):"
)
for s in sorted(sector_weights, key=sector_weights.get, reverse=True):
    print(f"   {s}: {sector_weights[s]:.4f}")


# ================================================================
# 🎯 STAGE 2: CONSTRAINED CONVEX STACKING META-LEARNER
# ================================================================
print("============================================================")
print("STAGE 2: CONSTRAINED CONVEX STACKING META-LEARNER")
print("Chronological QLIKE weight selection")
print("============================================================\n")
print(f"   Min samples:          {MIN_SAMPLES_STACKING}")
print(f"   Calibration period:   0%–{CAL_RATIO:.0%}")
print(f"   Weight-selection CV:  {CAL_RATIO:.0%}–{STACK_TRAIN_END:.0%}")
print(f"   Final test period:    {STACK_TRAIN_END:.0%}–100%")
print(f"   CV splits:            {CV_SPLITS}")
print(
    f"   Weight grid:          {len(CONVEX_WEIGHT_GRID)} values "
    f"from {CONVEX_WEIGHT_GRID[0]:.2f} "
    f"to {CONVEX_WEIGHT_GRID[-1]:.2f}"
)
print("   ✅ Weights are nonnegative")
print("   ✅ Transformer and TGARCH weights sum to one")
print("   ✅ Weight selected using chronological CV QLIKE only")
print("   ✅ Locked Final_Test is used only for final evaluation")
print("   ✅ Existing calibration, target, dates and outputs retained")
print()

sector_decisions = []
hybrid_df['Hybrid_Forecast']           = np.nan
hybrid_df['Norm_Eff_Coef_Share_DL']    = np.nan
hybrid_df['Norm_Eff_Coef_Share_GARCH'] = np.nan
hybrid_df['Fallback_Used']              = False
hybrid_df['Model_Used']                 = ''

for sector in sorted(hybrid_df['Sector'].unique()):
    mask   = hybrid_df['Sector'] == sector
    s_data = (
        hybrid_df.loc[mask]
        .copy()
        .sort_values('Date')
        .reset_index()
        .rename(columns={'index': 'Global_Index'})
    )

    if len(s_data) < MIN_SAMPLES_STACKING:
        print(f"⚠️  Skipping {sector}: insufficient data")
        continue

    cal_mask   = s_data['Evaluation_Split'] == 'Calibration'
    train_mask = s_data['Evaluation_Split'] == 'Stacking_Train'
    test_mask  = s_data['Evaluation_Split'] == 'Final_Test'

    if cal_mask.sum() < 3 or train_mask.sum() < 5 or test_mask.sum() < 5:
        print(
            f"⚠️  Skipping {sector}: split too small "
            f"(cal={cal_mask.sum()}, train={train_mask.sum()}, "
            f"test={test_mask.sum()})"
        )
        continue

    X_all = s_data[['DL_Forecast', 'GARCH_Variance']].to_numpy(dtype=float)
    y_all = s_data['Actual_Variance'].to_numpy(dtype=float)

    X_train_meta = X_all[train_mask.to_numpy()]
    y_train      = y_all[train_mask.to_numpy()]
    y_test       = y_all[test_mask.to_numpy()]

    best_weight_dl   = 0.50
    best_val_qlike   = np.inf
    selection_method = 'Chronological_TimeSeriesSplit'

    actual_cv_splits = min(CV_SPLITS, max(2, len(X_train_meta) // 8))

    try:
        if actual_cv_splits >= 2:
            tscv = TimeSeriesSplit(n_splits=actual_cv_splits)

            chronological_splits = [
                (tr_idx, vl_idx)
                for tr_idx, vl_idx in tscv.split(X_train_meta)
                if len(tr_idx) >= 3 and len(vl_idx) >= 2
            ]

            if not chronological_splits:
                raise ValueError("No valid chronological CV folds.")

            for weight_dl in CONVEX_WEIGHT_GRID:
                fold_qlikes = []

                for _, vl_idx in chronological_splits:
                    validation_prediction = np.clip(
                        weight_dl * X_train_meta[vl_idx, 0]
                        + (1.0 - weight_dl) * X_train_meta[vl_idx, 1],
                        epsilon, None
                    )
                    fold_qlikes.append(
                        qlike_np(y_train[vl_idx], validation_prediction, epsilon)
                    )

                mean_cv_qlike = float(np.mean(fold_qlikes))

                if mean_cv_qlike < best_val_qlike:
                    best_val_qlike = mean_cv_qlike
                    best_weight_dl = float(weight_dl)

        else:
            selection_method = 'Full_Stacking_Train_QLIKE'

            for weight_dl in CONVEX_WEIGHT_GRID:
                training_prediction = np.clip(
                    weight_dl * X_train_meta[:, 0]
                    + (1.0 - weight_dl) * X_train_meta[:, 1],
                    epsilon, None
                )
                training_qlike = qlike_np(y_train, training_prediction, epsilon)

                if training_qlike < best_val_qlike:
                    best_val_qlike = float(training_qlike)
                    best_weight_dl = float(weight_dl)

        best_weight_garch = 1.0 - best_weight_dl

        all_preds = np.clip(
            best_weight_dl * X_all[:, 0]
            + best_weight_garch * X_all[:, 1],
            epsilon, None
        )

        fallback_used = (
            not np.all(np.isfinite(all_preds))
            or np.any(all_preds <= 0)
        )

        if fallback_used:
            best_weight_dl    = 0.50
            best_weight_garch = 0.50
            selection_method  = selection_method + '_Numerical_EqualWeight_Fallback'
            all_preds = np.clip(
                0.50 * X_all[:, 0] + 0.50 * X_all[:, 1],
                epsilon, None
            )

        norm_share_dl    = best_weight_dl
        norm_share_garch = best_weight_garch

        for k, global_idx in enumerate(s_data['Global_Index'].tolist()):
            hybrid_df.loc[global_idx, 'Hybrid_Forecast']           = all_preds[k]
            hybrid_df.loc[global_idx, 'Norm_Eff_Coef_Share_DL']    = norm_share_dl
            hybrid_df.loc[global_idx, 'Norm_Eff_Coef_Share_GARCH'] = norm_share_garch
            hybrid_df.loc[global_idx, 'Fallback_Used']              = fallback_used
            hybrid_df.loc[global_idx, 'Model_Used'] = (
                f'ConstrainedConvex('
                f'w_DL={best_weight_dl:.2f},'
                f'w_TGARCH={best_weight_garch:.2f})'
                + (' [FALLBACK]' if fallback_used else '')
            )

        test_preds       = all_preds[test_mask.to_numpy()]
        dl_test          = X_all[test_mask.to_numpy(), 0]
        garch_test       = X_all[test_mask.to_numpy(), 1]

        test_qlike       = qlike_np(y_test, test_preds,  epsilon)
        test_rmse        = np.sqrt(np.mean((y_test - test_preds) ** 2))
        test_mae         = np.mean(np.abs(y_test - test_preds))
        test_qlike_dl    = qlike_np(y_test, dl_test,    epsilon)
        test_qlike_garch = qlike_np(y_test, garch_test, epsilon)
        best_individual  = min(test_qlike_dl, test_qlike_garch)
        improvement      = best_individual - test_qlike
        improvement_pct  = (
            improvement / best_individual * 100
            if best_individual > 0 else 0.0
        )

        sector_decisions.append({
            'Sector':                    sector,
            'GARCH_Model':               'TGARCH',
            'DL_Model':                  'Transformer',
            'Meta_Learner':              'Constrained_Convex_QLIKE',
            'Convex_Weight_DL':          round(best_weight_dl,    4),
            'Convex_Weight_GARCH':       round(best_weight_garch, 4),
            'Norm_Eff_Coef_Share_DL':    round(norm_share_dl,     4),
            'Norm_Eff_Coef_Share_GARCH': round(norm_share_garch,  4),
            'Weight_Selection_Method':   selection_method,
            'Weight_Grid_Size':          len(CONVEX_WEIGHT_GRID),
            'Weight_Grid_Step':          round(
                float(CONVEX_WEIGHT_GRID[1] - CONVEX_WEIGHT_GRID[0]), 6
            ),
            'Fallback_Used':             fallback_used,
            'CV_QLIKE':                  round(best_val_qlike, 6)
                                         if np.isfinite(best_val_qlike)
                                         else np.nan,
            'Test_Hybrid_QLIKE':         round(test_qlike,       6),
            'Test_Hybrid_RMSE':          round(test_rmse,        6),
            'Test_Hybrid_MAE':           round(test_mae,         6),
            'Test_DL_QLIKE':             round(test_qlike_dl,    6),
            'Test_GARCH_QLIKE':          round(test_qlike_garch, 6),
            'Best_Individual_QLIKE':     round(best_individual,  6),
            'Improvement_QLIKE':         round(improvement,      6),
            'Improvement_Pct':           round(improvement_pct,  2),
            'N_Calibration':             int(cal_mask.sum()),
            'N_Train':                   int(train_mask.sum()),
            'N_Test':                    int(test_mask.sum()),
            'N_Total':                   len(s_data),
            'Code_Version':              BASELINE_CODE_VERSION,
            'Reproducibility_Version':   REPRODUCIBILITY_VERSION,
            'Execution_Device':          GPU_NAME
        })

        status      = "✅" if test_qlike <= best_individual else "⚠️"
        fallback_tag = " [FALLBACK]" if fallback_used else ""
        print(f"{status} {sector:<35}{fallback_tag}")
        print(f"      Test QLIKE: Stacking={test_qlike:.4f}, "
              f"DL={test_qlike_dl:.4f}, TGARCH={test_qlike_garch:.4f}")
        print(f"      Test RMSE={test_rmse:.6f}  MAE={test_mae:.6f}")
        print(f"      Constrained weights: DL={best_weight_dl:.2f}, "
              f"TGARCH={best_weight_garch:.2f}"
              + (" (equal-weight numerical fallback)" if fallback_used else ""))
        print(f"      Weight selection: {selection_method}; "
              f"CV QLIKE={best_val_qlike:.4f}")
        status_text = (
            f"✅ Improvement: {improvement:.4f} ({improvement_pct:.2f}%)"
            if improvement > 0
            else f"⚠️  Underperformance: {abs(improvement):.4f} "
                 f"({abs(improvement_pct):.2f}%)"
        )
        print(f"      {status_text}")
        print(f"      Data: Calibration={int(cal_mask.sum())}, "
              f"Train={int(train_mask.sum())}, "
              f"Test={int(test_mask.sum())}, Total={len(s_data)}")

    except Exception as error:
        print(f"   ❌ Constrained stacking failed for {sector}: {error}")
        import traceback
        traceback.print_exc()
        continue

# ================================================================
# End of Stage 2 loop
# ================================================================
if not sector_decisions:
    print("\n❌ ERROR: No sectors successfully processed!")
    decisions_df   = pd.DataFrame()
    improved_count = 0
    total_count    = 0
    fallback_count = 0
else:
    decisions_df   = pd.DataFrame(sector_decisions)
    improved_count = (decisions_df['Improvement_QLIKE'] > 0).sum()
    total_count    = len(decisions_df)
    fallback_count = int(decisions_df['Fallback_Used'].sum())

    print(f"\n{'='*70}")
    print(f"✅ STACKING SUMMARY (Final-Test split metrics only)")
    print(f"{'='*70}")
    print(f"   Sectors processed:             {total_count}/"
          f"{len(hybrid_df['Sector'].unique())}")
    print(f"   Improved over best individual: {improved_count}/{total_count} "
          f"({improved_count/total_count*100:.1f}%)")
    print(f"   Numerical equal-weight fallback used: {fallback_count}/{total_count}")
    if improved_count > 0:
        avg_imp = decisions_df[
            decisions_df['Improvement_QLIKE'] > 0
        ]['Improvement_Pct'].mean()
        print(f"   Mean improvement (when better): {avg_imp:.2f}%")

# ================================================================
# 📊 Weighted Metrics — Final_Test split only
# ================================================================
print("\n============================================================")
print("STACKING HYBRID — WEIGHTED FINAL-TEST METRICS")
print("(Final_Test split only — not contaminated by training data)")
print("============================================================")

weighted_rmse  = 0.0
weighted_mae   = 0.0
weighted_qlike = 0.0
sector_metrics = []

for sector in sector_weights:
    dec = decisions_df[decisions_df['Sector'] == sector]
    if len(dec) == 0:
        continue

    w                = sector_weights[sector]
    test_qlike_v     = dec['Test_Hybrid_QLIKE'].values[0]
    test_rmse_v      = dec['Test_Hybrid_RMSE'].values[0]
    test_mae_v       = dec['Test_Hybrid_MAE'].values[0]
    norm_share_dl    = dec['Norm_Eff_Coef_Share_DL'].values[0]
    norm_share_garch = dec['Norm_Eff_Coef_Share_GARCH'].values[0]
    fallback_used    = dec['Fallback_Used'].values[0]
    n_test           = dec['N_Test'].values[0]

    weighted_qlike += w * test_qlike_v
    weighted_rmse  += w * test_rmse_v
    weighted_mae   += w * test_mae_v

    sector_metrics.append({
        "Sector":                    sector,
        "GARCH_Model":               "TGARCH",
        "DL_Model":                  "Transformer",
        "Weight":                    w,
        "QLIKE":                     test_qlike_v,
        "RMSE":                      test_rmse_v,
        "MAE":                       test_mae_v,
        "Norm_Eff_Coef_Share_DL":    norm_share_dl,
        "Norm_Eff_Coef_Share_GARCH": norm_share_garch,
        "Fallback_Used":             fallback_used,
        "N_Test":                    n_test,
        "Code_Version":              BASELINE_CODE_VERSION,
        "Reproducibility_Version":   REPRODUCIBILITY_VERSION,
        "Execution_Device":          GPU_NAME
    })

print(f"Weighted RMSE  (Final_Test): {weighted_rmse:.6f}")
print(f"Weighted MAE   (Final_Test): {weighted_mae:.6f}")
print(f"✨ Weighted QLIKE (Final_Test): {weighted_qlike:.6f} ✨")
print("============================================================\n")

print("Per-Sector Breakdown (Final_Test metrics):")

def _fmt_metric(value, width=12):
    value = float(value)
    if abs(value) >= 1000 or (0 < abs(value) < 1e-5):
        return f"{value:>{width}.4e}"
    return f"{value:>{width}.6f}"

header = (
    f"{'Sector':<38}"
    f"{'Weight':>9}"
    f"{'QLIKE':>12}"
    f"{'RMSE':>12}"
    f"{'MAE':>12}"
    f"{'DL':>8}"
    f"{'GARCH':>8}"
    f"{'Fallback':>10}"
    f"{'N':>6}"
)
print(header)
print("-" * len(header))

for m in sorted(sector_metrics, key=lambda x: x['Weight'], reverse=True):
    fb = "yes" if m['Fallback_Used'] else "no"
    print(
        f"{m['Sector']:<38}"
        f"{m['Weight']:>9.4f}"
        f"{_fmt_metric(m['QLIKE'], 12)}"
        f"{_fmt_metric(m['RMSE'], 12)}"
        f"{_fmt_metric(m['MAE'], 12)}"
        f"{m['Norm_Eff_Coef_Share_DL']:>8.2f}"
        f"{m['Norm_Eff_Coef_Share_GARCH']:>8.2f}"
        f"{fb:>10}"
        f"{int(m['N_Test']):>6d}"
    )
print()

# ================================================================
# 💾 Save CSV Outputs
# ================================================================
pd.DataFrame(sector_metrics).to_csv(
    "Stacking_Transformer_TGARCH_Sector_Metrics.csv", index=False
)
decisions_df.to_csv(
    "Stacking_Transformer_TGARCH_Meta_Weights.csv", index=False
)
hybrid_output = hybrid_df[[
    'Date', 'Sector', 'GARCH_Model', 'DL_Model',
    'Actual_Variance', 'Raw_DL_Forecast', 'DL_Forecast',
    'GARCH_Variance', 'Hybrid_Forecast',
    'Norm_Eff_Coef_Share_DL', 'Norm_Eff_Coef_Share_GARCH',
    'Fallback_Used', 'Model_Used',
    'Calibration_CF', 'Evaluation_Split', 'Code_Version',
    'Reproducibility_Version', 'Execution_Device'
]].copy()
hybrid_output.to_csv(
    "Stacking_Transformer_TGARCH_Forecasts.csv", index=False
)

print("✅ CSV Files saved:")
print("   - Stacking_Transformer_TGARCH_Sector_Metrics.csv")
print("   - Stacking_Transformer_TGARCH_Meta_Weights.csv")
print("   - Stacking_Transformer_TGARCH_Forecasts.csv")
print("   - Common_Actual_Variance_1Day_Squared_Return.csv\n")

# ================================================================
# 📈 VISUALIZATIONS
# ================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import shutil

print("============================================================")
print("GENERATING VISUALIZATIONS")
print("============================================================\n")

os.makedirs("Volatility_Graphs",           exist_ok=True)
os.makedirs("Forecast_Error_Graphs",       exist_ok=True)
os.makedirs("Weight_Visualization_Graphs", exist_ok=True)
os.makedirs("Market_Level_Graphs",         exist_ok=True)

plot_df = hybrid_df.copy()
plot_df['Date'] = pd.to_datetime(plot_df['Date'])
plot_df         = plot_df.sort_values(['Sector', 'Date'])
plot_df['Actual_Volatility'] = np.sqrt(
    plot_df['Actual_Variance'].clip(lower=0)
)
plot_df['Hybrid_Volatility'] = np.sqrt(
    plot_df['Hybrid_Forecast'].clip(lower=0)
)

sorted_sectors = sorted(
    sector_weights.keys(), key=lambda s: sector_weights[s], reverse=True
)

# Volatility graphs
print("Generating volatility graphs...")
for sector in sorted_sectors:
    s_data = plot_df[plot_df['Sector'] == sector].sort_values('Date')
    if len(s_data) == 0:
        continue

    w        = sector_weights.get(sector, 0)
    dec      = decisions_df[decisions_df['Sector'] == sector]
    ns_dl    = dec['Norm_Eff_Coef_Share_DL'].values[0] if len(dec) > 0 else 0.5
    fallback = dec['Fallback_Used'].values[0]            if len(dec) > 0 else False

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(s_data['Date'], s_data['Actual_Volatility'],
            color='black', linewidth=2,
            label='Observed One-Day Volatility Proxy',
            marker='o', markersize=4)
    ax.plot(s_data['Date'], s_data['Hybrid_Volatility'],
            color='#1E88E5', linewidth=1.8,
            label='Constrained Convex Stacking (Transformer + TGARCH)',
            linestyle='--', marker='s', markersize=3)
    fb_note = "\n[Equal-weight fallback applied]" if fallback else ""
    ax.set_title(
        f'One-Day Squared-Return Variance Proxy Forecasts: {sector}{fb_note}\n'
        f'(DL convex weight={ns_dl:.2f}, Market Weight={w:.4f})',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Volatility (√Squared Return = |Return|)', fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    safe_name = sector.replace('/', '_').replace('\\', '_').replace(' ', '_')
    plt.savefig(
        f"Volatility_Graphs/{safe_name}_Volatility.png",
        dpi=150, bbox_inches='tight'
    )
    plt.close()

print(f"   ✅ Generated {len(sorted_sectors)} volatility graphs\n")

# Forecast error graphs
print("Generating forecast error graphs...")
for sector in sorted_sectors:
    s_data = plot_df[plot_df['Sector'] == sector].sort_values('Date')
    if len(s_data) == 0:
        continue
    error = (
        s_data['Actual_Volatility'].values
        - s_data['Hybrid_Volatility'].values
    )
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(s_data['Date'], error, width=2.5,
           color=np.where(error >= 0, '#4CAF50', '#F44336'), alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=1.5)
    ax.set_title(
        f'Forecast Errors: {sector} (Constrained Convex Stacking)',
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Error (Observed − Forecast)', fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    safe_name = sector.replace('/', '_').replace('\\', '_').replace(' ', '_')
    plt.savefig(
        f"Forecast_Error_Graphs/{safe_name}_Errors.png",
        dpi=150, bbox_inches='tight'
    )
    plt.close()

print(f"   ✅ Generated {len(sorted_sectors)} error graphs\n")

# Constrained-weight graphs
print("Generating constrained-weight graphs...")
for sector in sorted_sectors:
    dec = decisions_df[decisions_df['Sector'] == sector]
    if len(dec) == 0:
        continue

    ns_dl    = dec['Norm_Eff_Coef_Share_DL'].values[0]
    ns_garch = dec['Norm_Eff_Coef_Share_GARCH'].values[0]
    fallback = dec['Fallback_Used'].values[0]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(
        ['Transformer', 'TGARCH'],
        [ns_dl, ns_garch],
        color=['#1976D2', '#D32F2F'],
        alpha=0.85, edgecolor='black', linewidth=1.5
    )
    fb_note = "\n[Equal-weight fallback applied]" if fallback else ""
    ax.set_title(
        f'Constrained Convex Weights: {sector}{fb_note}',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Constrained Convex Weight', fontsize=12)
    ax.set_ylim([0, 1])
    ax.axhline(y=0.5, color='gray', linewidth=1.5,
               linestyle='--', alpha=0.6, label='Equal Share (0.50)')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h,
                f'{h:.2f}', ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    safe_name = sector.replace('/', '_').replace('\\', '_').replace(' ', '_')
    plt.savefig(
        f"Weight_Visualization_Graphs/{safe_name}_CoefShares.png",
        dpi=150, bbox_inches='tight'
    )
    plt.close()

print(f"   ✅ Generated {len(sorted_sectors)} constrained-weight graphs\n")

# Market-level graph
print("Generating market-level graph...")
fig, ax = plt.subplots(figsize=(14, 6))
plot_df['Sector_Weight'] = plot_df['Sector'].map(sector_weights)
market_vol = plot_df.groupby('Date').apply(
    lambda g: pd.Series({
        'Actual': np.sum(g['Sector_Weight'] * g['Actual_Volatility']),
        'Hybrid': np.sum(g['Sector_Weight'] * g['Hybrid_Volatility'])
    }), include_groups=False
).reset_index().sort_values('Date')

ax.plot(market_vol['Date'], market_vol['Actual'],
        color='black', linewidth=2.5,
        label='Observed One-Day Volatility Proxy',
        marker='o', markersize=5)
ax.plot(market_vol['Date'], market_vol['Hybrid'],
        color='#1E88E5', linewidth=2,
        label='Constrained Convex Stacking (Transformer + TGARCH)',
        linestyle='--', marker='s', markersize=4)
ax.set_title(
    'Market-Level Weighted Volatility\n'
    '(Constrained Convex Stacking: Transformer + TGARCH)',
    fontsize=16, fontweight='bold'
)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Market-Cap-Weighted Volatility', fontsize=13)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    'Market_Level_Graphs/Market_Weighted_Volatility_Stacking.png',
    dpi=150, bbox_inches='tight'
)
plt.close()
print("   ✅ Saved market-level graph\n")

# ================================================================
# 📦 ZIP and Download
# ================================================================
print("Creating ZIP files...")
shutil.make_archive('Volatility_Graphs',           'zip', 'Volatility_Graphs')
shutil.make_archive('Forecast_Error_Graphs',       'zip', 'Forecast_Error_Graphs')
shutil.make_archive('Weight_Visualization_Graphs', 'zip', 'Weight_Visualization_Graphs')
shutil.make_archive('Market_Level_Graphs',         'zip', 'Market_Level_Graphs')
print("✅ ZIP files created\n")

# ================================================================
# 📥 Download All Files
# ================================================================
from google.colab import files as colab_files

print("Downloading files...")
download_list = [
    "Stacking_Transformer_TGARCH_Sector_Metrics.csv",
    "Stacking_Transformer_TGARCH_Meta_Weights.csv",
    "Stacking_Transformer_TGARCH_Forecasts.csv",
    "Common_Actual_Variance_1Day_Squared_Return.csv",
    "Volatility_Graphs.zip",
    "Forecast_Error_Graphs.zip",
    "Weight_Visualization_Graphs.zip",
    "Market_Level_Graphs.zip"
]

for f in download_list:
    try:
        colab_files.download(f)
        print(f"   ✅ Downloaded: {f}")
        time.sleep(1.5)
    except Exception as e:
        print(f"   ⚠️ Failed: {f} — {e}")

# ================================================================
# 📊 Summary
# ================================================================
print("\n============================================================")
print("ANALYSIS COMPLETE")
print("============================================================")
print(f"✨ Weighted QLIKE (Final_Test): {weighted_qlike:.6f}")
print(f"✨ Weighted RMSE  (Final_Test): {weighted_rmse:.6f}")
print(f"✨ Weighted MAE   (Final_Test): {weighted_mae:.6f}")
print(f"✅ Methodology: Constrained Convex Stacking Meta-Learner")
print(f"   - Stage 1: Transformer built once per sector,")
print(f"              and trained once on the pre-test sample")
print(f"   - Stage 2: Constrained convex weights selected by chronological QLIKE CV")
print(f"   - Stage 3: Final stacked forecasts")
print(f"   - ✅ FIX 1:          Test-split metrics only")
print(f"   - ✅ FIX 2:          Merge by Sector+Date")
print(f"   - ✅ FIX 3:          Calibration from training split only")
print(f"   - ✅ FIX 4:          One-day squared-return target from raw master data")
print(f"   - ✅ FIX 5:          Architecture matches manuscript")
print(f"   - ✅ FIX 6:          Weight selection uses Stacking_Train only")
print(f"   - ✅ FIX 7:          Fallback metrics consistent")
print(f"   - ✅ FIX 8:          Nonnegative weights sum to one")
print(f"   - ✅ Remaining 2:    Dropout inactive during inference")
print(f"   - ✅ Remaining 3:    One Transformer fit per sector")
print(f"   - ✅ Remaining 4:    Pre-test scaling only")
print(f"   - ✅ Remaining 5:    Market weights from Final_Test dates")
print(f"   - ✅ Remaining 6:    Evaluation_Split exported in CSV")
print(f"   - ✅ Remaining 7:    Label = Observed One-Day Volatility Proxy")
print(f"   - ✅ REPRODUCIBILITY: TF_DETERMINISTIC_OPS + float32 + per-build seed")
print(f"   - ✅ EXECUTION:       {GPU_NAME} + GPU memory growth")
print(f"   - ✅ REPRO VERSION:   {REPRODUCIBILITY_VERSION}")
print(f"✅ Improved in {improved_count}/{total_count} sectors")
print(f"✅ Fallback used in {fallback_count}/{total_count} sectors")
print(f"✅ Generated {len(sorted_sectors) * 3 + 1} graphs")
print("============================================================\n")


✅ NVIDIA runtime detected:
   Tesla T4, 15360 MiB, 580.82.07
⚠️ TensorFlow was already imported. A GPU is visible, so execution can continue, but use a fresh runtime for the definitive reproducible run.
✅ TensorFlow GPU detected: Tesla T4
✅ Verification operation device: /GPU:0
✅ GPU memory growth: enabled
✅ Deterministic TensorFlow/cuDNN/cuBLAS settings enabled
✅ Precision policy: float32
✅ Reproducibility version: deterministic_t4_gpu_float32_v2

📥 Upload exactly one Master CSV and one TGARCH-X forecast CSV.


Saving Master File_All variables 2014-2024-v2.csv to Master File_All variables 2014-2024-v2 (2).csv
Saving TGARCH_X_Forecasts_CLEAN.csv to TGARCH_X_Forecasts_CLEAN (2).csv
FILE DETECTION
Files uploaded: 2
   - Master File_All variables 2014-2024-v2 (2).csv
   - TGARCH_X_Forecasts_CLEAN (2).csv

File Detection Results:
   Master files found:  1
      Using: Master File_All variables 2014-2024-v2 (2).csv
   TGARCH files found:  1
      Using: TGARCH_X_Forecasts_CLEAN (2).csv
   All uploaded files:  ['Master File_All variables 2014-2024-v2 (2).csv', 'TGARCH_X_Forecasts_CLEAN (2).csv']

✅ All required files loaded successfully!

Column Detection:
   TGARCH forecast: Forecast_Variance → 'TGARCH_Variance'

STACKING: TRANSFORMER + TGARCH (ALL SECTORS)
Following Wolpert (1992) & Breiman (1996)



/tmp/ipykernel_2059/1154221925.py:297: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  tgarch_df['Date'] = pd.to_datetime(tgarch_df['Date'], dayfirst=True, errors='coerce')


✅ Sector features computed: 25836 observations

✅ TGARCH forecasts loaded: 23916 observations

📊 Computing one-day squared-return target from raw master data...
✅ Common one-day target merged into Transformer data: 25836 observations
✅ Common evaluation target computed: 25936 observations across 10 sectors
   Agriculture: 2725 observations
   Automobiles and Accessories: 1269 observations
   Banking: 2743 observations
   Commercial and services: 2743 observations
   Construction and Allied: 2741 observations
   Energy and Petroleum: 2743 observations
   Insurance: 2743 observations
   Investment: 2743 observations
   Manufacturing and Allied: 2743 observations
   Telecommunication and Technology: 2743 observations

✅ Saved: Common_Actual_Variance_1Day_Squared_Return.csv

STAGE 1: TRANSFORMER BASE FORECASTING
   Architecture: 1×TransformerBlock(heads=4, d_model=32, dff=64) →
                 GlobalAvgPool → Dense(64) → Dense(32)
   ✅ Dropout inactive during inference
   ✅ One model fitt

Transformer Forecasting:   0%|          | 0/10 [00:00<?, ?it/s]

...

<IPython.core.display.Javascript object>

   ✅ Downloaded: Weight_Visualization_Graphs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✅ Downloaded: Market_Level_Graphs.zip

ANALYSIS COMPLETE
✨ Weighted QLIKE (Final_Test): 1.835549
✨ Weighted RMSE  (Final_Test): 0.000529
✨ Weighted MAE   (Final_Test): 0.000303
✅ Methodology: Constrained Convex Stacking Meta-Learner
   - Stage 1: Transformer built once per sector,
              and trained once on the pre-test sample
   - Stage 2: Constrained convex weights selected by chronological QLIKE CV
   - Stage 3: Final stacked forecasts
   - ✅ FIX 1:          Test-split metrics only
   - ✅ FIX 2:          Merge by Sector+Date
   - ✅ FIX 3:          Calibration from training split only
   - ✅ FIX 4:          One-day squared-return target from raw master data
   - ✅ FIX 5:          Architecture matches manuscript
   - ✅ FIX 6:          Weight selection uses Stacking_Train only
   - ✅ FIX 7:          Fallback metrics consistent
   - ✅ FIX 8:          Nonnegative weights sum to one
   - ✅ Remaining 2:    Dropout inactive during inference
   - ✅ Remaining 3:    One Transformer f

In [ ]:
# Verify all sectors have enough observations
baseline_forecasts_raw = pd.read_csv(
    'Stacking_Transformer_TGARCH_Forecasts.csv'
)
counts = baseline_forecasts_raw.groupby('Sector').size()
print("Forecast counts per sector:")
print(counts.sort_values())
print(f"\nMinimum: {counts.min()}")
print(f"All sectors ≥ 20: {(counts >= 20).all()}")

Forecast counts per sector:
Sector
Automobiles and Accessories          50
Agriculture                         108
Banking                             109
Commercial and services             109
Construction and Allied             109
Energy and Petroleum                109
Insurance                           109
Investment                          109
Manufacturing and Allied            109
Telecommunication and Technology    109
dtype: int64

Minimum: 50
All sectors ≥ 20: True


In [ ]:
# ================================================================
# 🗑️ CLEAR ALL CHECKPOINTS AND CACHED VARIABLES
# Forces a completely fresh run with the date-alignment fix active
# ================================================================
import os
import gc
import tensorflow as tf

print("="*90)
print("🗑️  CLEARING CHECKPOINTS AND SESSION MEMORY")
print("="*90)

# ----------------------------------------------------------------
# 1. Delete checkpoint files from disk
# ----------------------------------------------------------------
checkpoint_files = [
    'Enhanced_Stacking_CHECKPOINT.csv',
    'Enhanced_Forecasts_CHECKPOINT.csv',
]
for f in checkpoint_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"   🗑️  Deleted: {f}")
    else:
        print(f"   ℹ️  Not found (already clean): {f}")

# ----------------------------------------------------------------
# 2. Clear TensorFlow session and Python memory
# ----------------------------------------------------------------
tf.keras.backend.clear_session()
gc.collect()
gc.collect()
print("\n   ✅ TensorFlow session cleared")
print("   ✅ Python garbage collected")

# ----------------------------------------------------------------
# 3. Delete in-memory variables from the previous run
#    so Python does not reuse stale data
# ----------------------------------------------------------------
vars_to_clear = [
    'enhanced_hybrid_results',
    'enhanced_forecasts_all',
    'already_completed_sectors',
    'best_per_sector',
    'enhanced_results_df',
    'best_enhanced_forecasts',
    'all_enhanced_forecasts',
    'model_enhanced',
    'model_counter',
    'training_diagnostics',
]
cleared = []
for v in vars_to_clear:
    if v in dir():
        exec(f"del {v}")
        cleared.append(v)
    elif v in globals():
        del globals()[v]
        cleared.append(v)

if cleared:
    print(f"\n   ✅ Cleared in-memory variables: {cleared}")
else:
    print("\n   ℹ️  No stale in-memory variables found")

gc.collect()

print("\n" + "="*90)
print("✅ SESSION CLEAN — safe to run the full enhanced stacking script now")
print("   The script will process all 10 sectors from scratch with the")
print("   TGARCH date alignment fix active throughout.")
print("="*90)

🗑️  CLEARING CHECKPOINTS AND SESSION MEMORY
   🗑️  Deleted: Enhanced_Stacking_CHECKPOINT.csv
   🗑️  Deleted: Enhanced_Forecasts_CHECKPOINT.csv

   ✅ TensorFlow session cleared
   ✅ Python garbage collected

   ✅ Cleared in-memory variables: ['enhanced_hybrid_results', 'enhanced_forecasts_all', 'already_completed_sectors', 'best_per_sector', 'enhanced_results_df', 'best_enhanced_forecasts', 'all_enhanced_forecasts', 'training_diagnostics']

✅ SESSION CLEAN — safe to run the full enhanced stacking script now
   The script will process all 10 sectors from scratch with the
   TGARCH date alignment fix active throughout.


In [ ]:
# ================================================================
# 🧠 T4 GPU + reproducibility configuration
# Must appear before TensorFlow is imported
# ================================================================
import os
import gc
import hashlib

SEED = 42

os.environ['TF_CPP_MIN_LOG_LEVEL']       = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH']  = 'true'
os.environ['PYTHONHASHSEED']             = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']       = '1'
os.environ['TF_CUDNN_DETERMINISTIC']     = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG']    = ':4096:8'

# IMPORTANT: do not set CUDA_VISIBLE_DEVICES="-1"; that would disable the GPU.
print("✅ Deterministic T4-GPU environment variables set")

# ================================================================
# 📥 LOAD PRE-COMPUTED BASELINE RESULTS
# ================================================================
from google.colab import files
import pandas as pd
import numpy as np
import zipfile

CODE_VERSION = "enhanced_light_16ind_convex_indicator_v1_06_t4gpu_oneday_conservative_check"
REPRODUCIBILITY_VERSION = "deterministic_t4_gpu_float32_v1"
REQUIRED_BASELINE_VERSION = "baseline_light_step5_convex_v1_05_oneday"

print("="*90)
print("🔄 LOADING PRE-COMPUTED BASELINE RESULTS")
print("="*90)

required_baseline_files = {
    "metrics": "Stacking_Transformer_TGARCH_Sector_Metrics.csv",
    "forecasts": "Stacking_Transformer_TGARCH_Forecasts.csv",
}


def _normalise_filename(name):
    """Normalise Colab duplicate suffixes such as ' (1)' or ' (2)'."""
    import re
    base = os.path.basename(str(name))
    stem, extension = os.path.splitext(base)
    stem = re.sub(r"\s*\(\d+\)$", "", stem)
    return stem.lower(), extension.lower()


def _classify_baseline_csv(path):
    """Classify a CSV from its columns without loading the full file."""
    try:
        sample = pd.read_csv(path, nrows=5)
    except Exception:
        return None

    columns = {
        str(column).strip().replace(" ", "_").replace("(", "").replace(")", "")
        for column in sample.columns
    }

    metrics_required = {"Sector", "Weight", "QLIKE", "RMSE", "MAE"}
    forecast_core = {"Date", "Sector"}
    forecast_names = {
        "Hybrid_Forecast",
        "Baseline_Hybrid_Variance",
        "Hybrid_Variance",
        "Stacked_Forecast",
    }

    if metrics_required.issubset(columns):
        return "metrics"

    if forecast_core.issubset(columns) and (
        columns.intersection(forecast_names)
        or any("hybrid" in column.lower() for column in columns)
    ):
        return "forecasts"

    return None


def _resolve_baseline_file(role, canonical_name, uploaded_names):
    """Resolve exact, Colab-renamed, or column-compatible baseline files."""
    if os.path.exists(canonical_name):
        return canonical_name

    canonical_stem, canonical_extension = _normalise_filename(canonical_name)

    search_names = []
    for name in list(uploaded_names) + [
        name for name in os.listdir(".") if name.lower().endswith(".csv")
    ]:
        if name not in search_names and os.path.exists(name):
            search_names.append(name)

    filename_matches = []
    role_matches = []

    for name in search_names:
        stem, extension = _normalise_filename(name)
        if extension != ".csv":
            continue

        if stem == canonical_stem:
            filename_matches.append(name)

        if _classify_baseline_csv(name) == role:
            role_matches.append(name)

    candidates = filename_matches or role_matches

    if not candidates:
        print("\nUploaded/current CSV inspection:")
        for name in search_names:
            try:
                columns = list(pd.read_csv(name, nrows=0).columns)
                print(f"   - {name}: {columns}")
            except Exception as error:
                print(f"   - {name}: unreadable ({error})")
        raise FileNotFoundError(
            f"Could not identify the baseline {role} CSV. "
            f"Expected output: {canonical_name}"
        )

    unique_candidates = list(dict.fromkeys(candidates))
    if len(unique_candidates) > 1:
        exact = [
            candidate for candidate in unique_candidates
            if os.path.basename(candidate) == canonical_name
        ]
        if len(exact) == 1:
            selected = exact[0]
        else:
            raise ValueError(
                f"Multiple possible baseline {role} CSVs were detected: "
                f"{unique_candidates}. Keep only the files from one baseline run."
            )
    else:
        selected = unique_candidates[0]

    if os.path.abspath(selected) != os.path.abspath(canonical_name):
        import shutil
        shutil.copyfile(selected, canonical_name)
        print(f"✅ Resolved {selected} → {canonical_name}")

    return canonical_name


missing = [
    canonical_name
    for canonical_name in required_baseline_files.values()
    if not os.path.exists(canonical_name)
]

baseline_upload = {}
if missing:
    print(f"⚠️ Missing baseline files: {missing}")
    print("📥 Upload ONLY the two baseline output CSVs now:")
    print("   - Stacking_Transformer_TGARCH_Sector_Metrics.csv")
    print("   - Stacking_Transformer_TGARCH_Forecasts.csv")
    baseline_upload = files.upload()
    print(f"✅ Uploaded: {list(baseline_upload.keys())}")

uploaded_names = list(baseline_upload.keys())

metrics_path = _resolve_baseline_file(
    "metrics",
    required_baseline_files["metrics"],
    uploaded_names,
)
forecasts_path = _resolve_baseline_file(
    "forecasts",
    required_baseline_files["forecasts"],
    uploaded_names,
)

baseline_sector_metrics = pd.read_csv(metrics_path)
baseline_forecasts_raw = pd.read_csv(forecasts_path)

print("✅ Loaded baseline results:")
print(f"   - Sector metrics: {len(baseline_sector_metrics)} sectors")
print(f"   - Forecasts:      {len(baseline_forecasts_raw)} observations")

baseline_versions = set()
for _baseline_df in [baseline_sector_metrics, baseline_forecasts_raw]:
    if 'Code_Version' in _baseline_df.columns:
        baseline_versions.update(
            _baseline_df['Code_Version'].dropna().astype(str).unique()
        )

if baseline_versions and baseline_versions != {REQUIRED_BASELINE_VERSION}:
    raise ValueError(
        "Baseline files were not generated by the required one-day "
        f"baseline version {REQUIRED_BASELINE_VERSION}. "
        f"Found: {sorted(baseline_versions)}"
    )

print(f"✅ Baseline version confirmed: {REQUIRED_BASELINE_VERSION}")

# ================================================================
# 🔄 CHECK FOR EXISTING CHECKPOINT
# ================================================================
CHECKPOINT_FILE          = 'Enhanced_Stacking_CHECKPOINT.csv'
FORECAST_CHECKPOINT_FILE = 'Enhanced_Forecasts_CHECKPOINT.csv'
DM_LOSS_CHECKPOINT_FILE  = 'DM_LossDiffs_CHECKPOINT.csv'

def check_checkpoint_version(filepath):
    if not os.path.exists(filepath):
        return False
    try:
        df_cp = pd.read_csv(filepath, nrows=1)
        if 'Code_Version' not in df_cp.columns:
            print(f"   ⚠️  {filepath}: no version field — treating as stale")
            return False
        if df_cp['Code_Version'].iloc[0] != CODE_VERSION:
            print(f"   ⚠️  {filepath}: version mismatch — treating as stale")
            return False
        return True
    except Exception:
        return False

checkpoint_valid = check_checkpoint_version(CHECKPOINT_FILE)

if checkpoint_valid:
    print("\n" + "="*90)
    print(f"🔄 CHECKPOINT FOUND (version={CODE_VERSION}) — RESUME AVAILABLE")
    print("="*90)
    _cp = pd.read_csv(CHECKPOINT_FILE)
    print(f"   Completed sectors: {sorted(_cp['Sector'].unique())}")
else:
    if os.path.exists(CHECKPOINT_FILE):
        print(f"\n⚠️  Stale/mismatched checkpoint found.")
        print(f"   For definitive run delete checkpoints and restart.")
    print("\nℹ️  Starting fresh (no valid checkpoint)")

print(f"\n📥 UPLOAD DATA FILES (Master + TGARCH-X)")
print(f"   If resuming, also upload checkpoint files.")
uploaded = files.upload()

# ================================================================
# 📦 Imports & Setup
# ================================================================
import random
import warnings
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import display, Image
import psutil
from scipy import stats

def clear_memory():
    tf.keras.backend.clear_session()
    gc.collect()
    gc.collect()
    gc.collect()

def get_memory_usage():
    return psutil.virtual_memory().used / (1024**3)

def log_memory(stage=""):
    mem_gb = get_memory_usage()
    print(f"   💾 Memory [{stage}]: {mem_gb:.2f} GB")
    return mem_gb

# ================================================================
# 🎮 Require and configure a T4 GPU
# ================================================================
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError(
        "No GPU detected. In Colab select Runtime → Change runtime type "
        "→ T4 GPU, restart the runtime, and run this script first."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        raise RuntimeError(
            "GPU memory growth could not be configured. Restart the Colab "
            "runtime and run this script before any other TensorFlow cell."
        ) from error

gpu_details = tf.config.experimental.get_device_details(gpus[0])
GPU_NAME = gpu_details.get('device_name', 'Unknown GPU')

if 'T4' not in GPU_NAME.upper():
    raise RuntimeError(
        f"This definitive script requires a T4 GPU, but detected: {GPU_NAME}. "
        "Select T4 GPU in Colab and restart the runtime."
    )

try:
    tf.config.experimental.enable_op_determinism()
except Exception as error:
    raise RuntimeError(
        f"TensorFlow deterministic operations could not be enabled: {error}"
    ) from error

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

tf.keras.mixed_precision.set_global_policy('float32')
tf.config.optimizer.set_jit(False)

print(f"✅ T4 GPU detected: {GPU_NAME}")
print("✅ GPU memory growth enabled")
print("✅ Deterministic TensorFlow operations enabled")
print(f"✅ Reproducibility version: {REPRODUCIBILITY_VERSION}")

print("============================================================")
print("ENHANCED STACKING: TRANSFORMER + TGARCH (WITH INDICATORS)")
print(f"Code version: {CODE_VERSION}")
print(f"Execution device: {GPU_NAME}")
print(f"Reproducibility: {REPRODUCIBILITY_VERSION}")
print("Following Wolpert 1992, Breiman 1996")
print("Target: one-day squared market-cap-weighted sector return")
print("============================================================")
log_memory("Initial")

# ================================================================
# 🎯 QLIKE Loss
# ================================================================
@tf.function
def qlike_loss(y_true, y_pred, epsilon=1e-8):
    y_true = tf.clip_by_value(y_true, epsilon, 1e10)
    y_pred = tf.clip_by_value(y_pred, epsilon, 1e10)
    ratio  = y_true / y_pred
    return tf.reduce_mean(ratio - tf.math.log(ratio) - 1.0)

def qlike_np(y_true, y_pred, epsilon=1e-8):
    y_true = np.clip(np.asarray(y_true, dtype=float), epsilon, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), epsilon, None)
    ratio  = y_true / y_pred
    return np.mean(ratio - np.log(ratio) - 1.0)


def indicator_scaling_parameters(
    indicator_train, clip_low=0.01, clip_high=0.99, epsilon=1e-8
):
    values = np.asarray(indicator_train, dtype=float).reshape(-1)
    if len(values) == 0 or not np.all(np.isfinite(values)):
        raise ValueError("Indicator training values are empty or non-finite.")
    lower = float(np.quantile(values, clip_low))
    upper = float(np.quantile(values, clip_high))
    if upper < lower:
        lower, upper = upper, lower
    clipped = np.clip(values, lower, upper)
    center  = float(np.mean(clipped))
    scale   = float(np.std(clipped))
    if not np.isfinite(scale) or scale < epsilon:
        scale = 1.0
    return lower, upper, center, scale


def standardize_indicator(values, parameters):
    lower, upper, center, scale = parameters
    values  = np.asarray(values, dtype=float).reshape(-1)
    clipped = np.clip(values, lower, upper)
    return (clipped - center) / scale


def bounded_indicator_multiplier(
    standardized_indicator, beta, lower=0.50, upper=2.00
):
    log_lower      = np.log(lower)
    log_upper      = np.log(upper)
    log_adjustment = np.clip(
        beta * np.asarray(standardized_indicator, dtype=float),
        log_lower, log_upper
    )
    return np.exp(log_adjustment)


def qlike_series(y_true, y_pred, epsilon=1e-8):
    y_true = np.clip(np.asarray(y_true, dtype=float), epsilon, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), epsilon, None)
    ratio  = y_true / y_pred
    return ratio - np.log(ratio) - 1.0

# ================================================================
# 📊 Diebold-Mariano Test (Harvey, Leybourne & Newbold 1997)
# ================================================================
def diebold_mariano_test(loss_baseline, loss_enhanced, h=1):
    d      = np.asarray(loss_baseline) - np.asarray(loss_enhanced)
    n      = len(d)
    if n < 10:
        return np.nan, np.nan, np.mean(d)
    d_bar  = np.mean(d)
    gamma0 = np.var(d, ddof=1)
    nw_var = gamma0
    for lag in range(1, h):
        gamma_lag = np.cov(d[lag:], d[:-lag], ddof=1)[0, 1]
        nw_var   += 2 * (1 - lag / h) * gamma_lag
    nw_var      = max(nw_var, 1e-16)
    dm_stat     = d_bar / np.sqrt(nw_var / n)
    hln_corr    = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    dm_stat_adj = dm_stat * hln_corr
    p_value     = 2 * (1 - stats.t.cdf(abs(dm_stat_adj), df=n - 1))
    return dm_stat_adj, p_value, d_bar

# ================================================================
# 📄 Load Data
# ================================================================
main_files   = [f for f in uploaded if "Master" in f or "master" in f]
tgarch_files = [f for f in uploaded if "TGARCH" in f and "Forecast" in f]
if not tgarch_files:
    tgarch_files = [f for f in uploaded if "TGARCH" in f or "tgarch" in f]

if not main_files:   raise FileNotFoundError("No Master file uploaded.")
if not tgarch_files: raise FileNotFoundError("No TGARCH-X file uploaded.")

main_file   = main_files[0]
tgarch_file = tgarch_files[0]

df        = pd.read_csv(main_file)
tgarch_df = pd.read_csv(tgarch_file)

for frame in [df, tgarch_df]:
    frame.columns = (frame.columns.str.strip()
                     .str.replace(" ", "_")
                     .str.replace("(", "")
                     .str.replace(")", ""))

df = df.rename(columns={
    'Market_Capitalization_KES': 'Market_Cap', 'Category': 'Sector'
})

tgarch_fc_col = [
    c for c in tgarch_df.columns
    if 'forecast' in c.lower() and 'var' in c.lower()
]
if not tgarch_fc_col:
    raise ValueError("Could not detect TGARCH-X forecast variance column.")
tgarch_df = tgarch_df.rename(columns={tgarch_fc_col[0]: 'TGARCH_Variance'})

df['Date']        = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
tgarch_df['Date'] = pd.to_datetime(tgarch_df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values(['Stock', 'Date']).reset_index(drop=True)

print(f"Master:   {main_file}")
print(f"TGARCH-X: {tgarch_file}")

# ================================================================
# 🔁 Compute Returns
# ================================================================
df['Market_Cap'] = pd.to_numeric(
    df['Market_Cap'].astype(str).str.replace(',', ''), errors='coerce'
)

# Use lagged market-capitalisation weights so weights for day t are known
# at the end of day t-1.
df['Market_Cap_Lag1'] = df.groupby('Stock')['Market_Cap'].shift(1)

target_source = df[
    ['Date', 'Sector', 'Stock', 'Close', 'Market_Cap']
].copy()
target_source['Close'] = pd.to_numeric(target_source['Close'], errors='coerce')

df['Trading_Volume'] = pd.to_numeric(
    df['Volume'].astype(str).str.replace(',', ''), errors='coerce'
)
df['Return'] = df.groupby('Stock')['Close'].transform(
    lambda x: np.log(x / x.shift(1))
)
df = df.dropna(
    subset=['Return', 'Market_Cap', 'Market_Cap_Lag1', 'Trading_Volume']
).copy()
df = df[df['Market_Cap_Lag1'] > 0].copy()

# ================================================================
# 📈 Sector-Weighted Returns
# ================================================================
def compute_sector_weighted(group):
    group = group.copy()
    cap_sum = group['Market_Cap_Lag1'].sum()
    if cap_sum <= 0 or not np.isfinite(cap_sum):
        return pd.Series({
            'Sector_Return':     np.nan,
            'Sector_Volume_Sum': group['Trading_Volume'].sum()
        })
    w = group['Market_Cap_Lag1'] / cap_sum
    return pd.Series({
        'Sector_Return':     np.sum(w * group['Return']),
        'Sector_Volume_Sum': group['Trading_Volume'].sum()
    })

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sector_returns = (
        df.groupby(['Date', 'Sector'], group_keys=False)
        .apply(compute_sector_weighted, include_groups=False)
        .reset_index().dropna()
    )

sector_returns['Log_Volume']  = np.log1p(sector_returns['Sector_Volume_Sum'])
sector_returns                = sector_returns.sort_values(
    ['Sector', 'Date']
).reset_index(drop=True)
sector_returns['Vol_5']       = sector_returns.groupby(
    'Sector')['Sector_Return'].transform(lambda x: x.rolling(5).std())
sector_returns['Vol_10']      = sector_returns.groupby(
    'Sector')['Sector_Return'].transform(lambda x: x.rolling(10).std())
sector_returns['Vol_5_Lag1']  = sector_returns.groupby('Sector')['Vol_5'].shift(1)
sector_returns['Vol_10_Lag1'] = sector_returns.groupby('Sector')['Vol_10'].shift(1)
sector_returns = sector_returns.dropna(
    subset=['Vol_5_Lag1', 'Vol_10_Lag1', 'Log_Volume']
).reset_index(drop=True)

# ================================================================
# ✅ Fixed One-Day Common Evaluation Target from Raw Master Data
# ================================================================
epsilon = 1e-8

print("\n📊 Computing one-day squared-return target from raw master data...")

target_source = target_source.dropna(
    subset=['Date', 'Sector', 'Stock', 'Close', 'Market_Cap']
).copy()
target_source = target_source[target_source['Market_Cap'] > 0].copy()

if target_source.duplicated(['Sector', 'Stock', 'Date']).any():
    raise ValueError("Duplicate Sector-Stock-Date rows detected in target source.")

target_source = target_source.sort_values(
    ['Sector', 'Stock', 'Date']
).reset_index(drop=True)

target_source['Stock_Return'] = (
    target_source.groupby(['Sector', 'Stock'])['Close'].transform(
        lambda prices: np.log(prices / prices.shift(1))
    )
)
target_source['Market_Cap_Lag1'] = (
    target_source.groupby(['Sector', 'Stock'])['Market_Cap'].shift(1)
)

target_source = target_source.dropna(
    subset=['Stock_Return', 'Market_Cap_Lag1']
).copy()
target_source = target_source[target_source['Market_Cap_Lag1'] > 0].copy()
target_source['Weighted_Return_Component'] = (
    target_source['Market_Cap_Lag1'] * target_source['Stock_Return']
)

sector_target_returns = (
    target_source.groupby(['Sector', 'Date'], as_index=False)
    .agg(
        Weighted_Return_Sum=('Weighted_Return_Component', 'sum'),
        Sector_Market_Cap_Lag1=('Market_Cap_Lag1', 'sum'),
        N_Stocks=('Stock', 'nunique')
    )
)

sector_target_returns['Sector_Return'] = (
    sector_target_returns['Weighted_Return_Sum']
    / sector_target_returns['Sector_Market_Cap_Lag1']
)

sector_target_returns = (
    sector_target_returns
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['Sector_Return'])
    .sort_values(['Sector', 'Date'])
    .reset_index(drop=True)
)

sector_target_returns['Actual_Variance_Raw'] = (
    sector_target_returns['Sector_Return'] ** 2
)

common_actual = (
    sector_target_returns[
        ['Date', 'Sector', 'Sector_Return', 'N_Stocks', 'Actual_Variance_Raw']
    ]
    .dropna(subset=['Actual_Variance_Raw'])
    .copy()
)

common_actual['Actual_Variance'] = common_actual['Actual_Variance_Raw'].clip(lower=epsilon)
common_actual = common_actual.sort_values(['Sector', 'Date']).reset_index(drop=True)

if common_actual.duplicated(['Sector', 'Date']).any():
    raise ValueError("Duplicate Sector-Date rows detected in common target.")

target_hash_frame        = common_actual[['Date', 'Sector', 'Actual_Variance']].copy()
target_hash_frame['Date'] = pd.to_datetime(
    target_hash_frame['Date']
).dt.strftime('%Y-%m-%d')

target_bytes  = target_hash_frame.to_csv(
    index=False, float_format='%.17g', lineterminator='\n'
).encode('utf-8')
TARGET_SHA256 = hashlib.sha256(target_bytes).hexdigest()

common_actual.to_csv('Common_Actual_Variance_1Day_Squared_Return.csv', index=False)
with open('Common_Actual_Variance_1Day_Squared_Return_SHA256.txt', 'w',
          encoding='utf-8') as target_hash_file:
    target_hash_file.write(TARGET_SHA256 + '\n')

eval_target_df = common_actual[
    ['Date', 'Sector', 'Actual_Variance']
].rename(columns={'Actual_Variance': 'Eval_Variance'})

sector_returns = (
    sector_returns.merge(
        eval_target_df[['Date', 'Sector', 'Eval_Variance']],
        on=['Date', 'Sector'], how='inner', validate='one_to_one'
    )
    .rename(columns={'Eval_Variance': 'Realized_Variance'})
    .dropna(subset=['Realized_Variance'])
    .sort_values(['Sector', 'Date'])
    .reset_index(drop=True)
)

print(f"✅ Common one-day target merged into enhanced Transformer data: "
      f"{len(sector_returns)} observations")
print(f"✅ Common target: {len(eval_target_df)} obs across "
      f"{eval_target_df['Sector'].nunique()} sectors")
print(f"✅ Common target SHA-256: {TARGET_SHA256}")

del target_source, sector_target_returns, target_hash_frame
gc.collect()
log_memory("After eval target")

# ================================================================
# 📅 Clean TGARCH-X Forecasts
# ================================================================
tgarch_clean = tgarch_df[['Date', 'Sector', 'TGARCH_Variance']].copy()
tgarch_clean['TGARCH_Variance'] = pd.to_numeric(
    tgarch_clean['TGARCH_Variance'], errors='coerce'
)
tgarch_clean = tgarch_clean.dropna(
    subset=['Date', 'Sector', 'TGARCH_Variance']
).copy()
tgarch_clean['TGARCH_Variance'] = tgarch_clean['TGARCH_Variance'].clip(lower=epsilon)

if tgarch_clean.duplicated(['Sector', 'Date']).any():
    raise ValueError("Duplicate TGARCH-X forecast dates detected.")

tgarch_dates_by_sector = tgarch_clean.groupby('Sector')['Date'].apply(set).to_dict()
print("TGARCH-X Coverage:")
for s in sorted(tgarch_dates_by_sector):
    print(f"   {s}: {len(tgarch_dates_by_sector[s])} dates")

del tgarch_df
gc.collect()
log_memory("After TGARCH clean")

# ================================================================
# 🔄 Load Baseline Results
# ================================================================
print("\n" + "="*90)
print("🔄 USING PRE-COMPUTED BASELINE RESULTS")
print("="*90)

baseline_results = baseline_sector_metrics[
    ['Sector', 'Weight', 'QLIKE', 'RMSE', 'MAE']
].copy().rename(columns={
    'QLIKE': 'Baseline_Hybrid_QLIKE',
    'RMSE':  'Baseline_Hybrid_RMSE',
    'MAE':   'Baseline_Hybrid_MAE'
})

weighted_baseline_qlike = (
    baseline_results['Weight'] * baseline_results['Baseline_Hybrid_QLIKE']
).sum()
weighted_baseline_rmse = (
    baseline_results['Weight'] * baseline_results['Baseline_Hybrid_RMSE']
).sum()
weighted_baseline_mae = (
    baseline_results['Weight'] * baseline_results['Baseline_Hybrid_MAE']
).sum()
sector_weights = baseline_results.set_index('Sector')['Weight'].to_dict()

print(f"   Weighted Baseline QLIKE (original): {weighted_baseline_qlike:.6f}")
print(f"   Weighted Baseline RMSE  (original): {weighted_baseline_rmse:.6f}")
print(f"   Weighted Baseline MAE   (original): {weighted_baseline_mae:.6f}")
log_memory("After Baseline Load")

bfc         = baseline_forecasts_raw.copy()
bfc.columns = (bfc.columns.str.strip()
               .str.replace(" ", "_")
               .str.replace("(", "")
               .str.replace(")", ""))

preferred_hybrid_columns = [
    "Hybrid_Forecast", "Baseline_Hybrid_Variance",
    "Hybrid_Variance", "Stacked_Forecast"
]
hybrid_col = next((c for c in preferred_hybrid_columns if c in bfc.columns), None)
if hybrid_col is None:
    hybrid_candidates = [c for c in bfc.columns if "hybrid" in c.lower()]
    if len(hybrid_candidates) != 1:
        raise ValueError(
            "Unable to identify one unique baseline hybrid forecast column. "
            f"Candidates: {hybrid_candidates}"
        )
    hybrid_col = hybrid_candidates[0]

print(f"   Baseline hybrid column detected: '{hybrid_col}'")

baseline_forecasts = bfc.copy()
if hybrid_col != 'Baseline_Hybrid_Variance':
    baseline_forecasts = baseline_forecasts.rename(
        columns={hybrid_col: 'Baseline_Hybrid_Variance'}
    )
baseline_forecasts['Date'] = pd.to_datetime(
    baseline_forecasts['Date'], format='%Y-%m-%d', errors='coerce'
)
baseline_forecasts = baseline_forecasts.merge(
    eval_target_df[['Date', 'Sector', 'Eval_Variance']],
    on=['Date', 'Sector'], how='left'
)
if 'Actual_Variance' in baseline_forecasts.columns:
    baseline_forecasts['Actual_Variance'] = np.where(
        baseline_forecasts['Eval_Variance'].notna(),
        baseline_forecasts['Eval_Variance'],
        baseline_forecasts['Actual_Variance']
    )
else:
    baseline_forecasts['Actual_Variance'] = baseline_forecasts['Eval_Variance']

baseline_forecasts = baseline_forecasts.drop(columns=['Eval_Variance'], errors='ignore')
baseline_forecasts['Actual_Variance'] = baseline_forecasts[
    'Actual_Variance'
].clip(lower=epsilon)

if baseline_forecasts.duplicated(['Sector', 'Date']).any():
    raise ValueError("Duplicate baseline forecast dates detected.")

print(f"✅ Baseline forecasts aligned: {len(baseline_forecasts)} rows")

del bfc, baseline_forecasts_raw, baseline_sector_metrics
gc.collect()
log_memory("After baseline processing")

# ================================================================
# 🧠 Transformer Model
# ================================================================
def positional_encoding(seq_len, d_model):
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(
        np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model)
    )
    pos_enc          = np.zeros((seq_len, d_model))
    pos_enc[:, 0::2] = np.sin(position * div_term)
    pos_enc[:, 1::2] = np.cos(position * div_term)
    return tf.cast(pos_enc[np.newaxis, :, :], dtype=tf.float32)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.mha        = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.ffn        = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(d_model)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1   = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2   = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1        = self.layernorm1(inputs + attn_output)
        ffn_output  = self.ffn(out1, training=training)
        ffn_output  = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_transformer_qlike(input_shape,
                             d_model=32, num_heads=4,
                             dff=64, num_layers=1,
                             dropout_rate=0.2):
    tf.keras.utils.set_random_seed(SEED)   # ✅ locks weight initialisers per build
    seq_len, num_features = input_shape
    inputs  = tf.keras.Input(shape=input_shape)
    x       = tf.keras.layers.Dense(d_model)(inputs)
    x       = x + positional_encoding(seq_len, d_model)
    for _   in range(num_layers):
        x   = TransformerBlock(d_model, num_heads, dff, dropout_rate)(x)
    x       = tf.keras.layers.GlobalAveragePooling1D()(x)
    x       = tf.keras.layers.Dense(64, activation='relu')(x)
    x       = tf.keras.layers.Dropout(dropout_rate)(x)
    x       = tf.keras.layers.Dense(32, activation='relu')(x)
    x       = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(
        1, activation='softplus', dtype='float32'
    )(x)
    model   = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=qlike_loss
    )
    return model

def fast_predict(model, x):
    return model(x, training=False)

# ================================================================
# ⚙️ Parameters
# ================================================================
target      = 'Realized_Variance'
SEQ_LEN     = 20
WINDOW_SIZE = 200
EPOCHS      = 20
BATCH_SIZE  = 64
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.20
PATIENCE    = 3

CAL_RATIO       = 0.25
STACK_TRAIN_END = 0.65

CV_SPLITS                 = 3
CONVEX_WEIGHT_GRID        = np.linspace(0.0, 1.0, 101)
INDICATOR_BETA_GRID       = np.linspace(-0.50, 0.50, 41)
MIN_SAMPLES_STACKING      = 10
MIN_CV_IMPROVEMENT        = 2.0
MIN_WINNING_FOLDS         = 2
MIN_ABS_INDICATOR_BETA    = 0.025
INDICATOR_CLIP_LOW        = 0.01
INDICATOR_CLIP_HIGH       = 0.99
INDICATOR_ADJUSTMENT_LOW  = 0.50
INDICATOR_ADJUSTMENT_HIGH = 2.00

training_diagnostics = []
log_memory("After Setup")

# ================================================================
# 📊 Sector Price Indices
# ================================================================
print("\n📊 Step 1: Building sector price indices...")
sector_indices_list = []
for sector in tqdm(df['Sector'].unique(), desc="Price indices"):
    s_stocks     = df[df['Sector'] == sector].copy()
    sector_index = s_stocks.groupby('Date').apply(
        lambda x: pd.Series({
            'Sector':       sector,
            'Sector_Close': (
                np.average(x['Close'], weights=x['Market_Cap_Lag1'])
                if x['Market_Cap_Lag1'].sum() > 0 else x['Close'].mean()
            ),
            'Sector_High':  (
                np.average(x['High'], weights=x['Market_Cap_Lag1'])
                if x['Market_Cap_Lag1'].sum() > 0 else x['High'].mean()
            ),
            'Sector_Low':   (
                np.average(x['Low'], weights=x['Market_Cap_Lag1'])
                if x['Market_Cap_Lag1'].sum() > 0 else x['Low'].mean()
            )
        }), include_groups=False
    ).reset_index()
    sector_indices_list.append(sector_index)

sector_indices     = pd.concat(sector_indices_list, ignore_index=True)
sector_indices     = sector_indices.sort_values(['Sector', 'Date']).dropna()
sector_data_merged = sector_indices.merge(
    sector_returns[[
        'Date', 'Sector', 'Sector_Return', 'Sector_Volume_Sum',
        'Log_Volume', 'Vol_5', 'Vol_10', 'Vol_5_Lag1',
        'Vol_10_Lag1', 'Realized_Variance'
    ]],
    on=['Date', 'Sector'], how='inner'
)
print(f"✅ Merged price indices + features: {sector_data_merged.shape}")

del sector_indices_list, sector_indices, sector_returns
gc.collect()
log_memory("After price indices")

# ================================================================
# 📊 Compute Technical Indicators (16 indicators, all lagged by 1 day)
# ================================================================
print("\n📊 Step 2: Computing technical indicators (with Lag1)...")

try:
    import ta
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'ta', '--quiet'])
    import ta

from ta.momentum   import RSIIndicator
from ta.trend      import MACD
from ta.volatility import BollingerBands, AverageTrueRange

def compute_indicators(sector_df):
    df_ind = sector_df.copy().sort_values('Date').reset_index(drop=True)
    if len(df_ind) < 50:
        return None

    close = pd.Series(df_ind['Sector_Close'].values, dtype=float)
    high  = pd.Series(df_ind['Sector_High'].values, dtype=float)
    low   = pd.Series(df_ind['Sector_Low'].values, dtype=float)

    try:
        ema10 = close.ewm(span=10, adjust=False).mean()
        ema14 = close.ewm(span=14, adjust=False).mean()
        ema20 = close.ewm(span=20, adjust=False).mean()
        sma10 = close.rolling(10).mean()
        sma20 = close.rolling(20).mean()
        sma50 = close.rolling(50).mean()

        bb        = BollingerBands(close=close, window=20, window_dev=2)
        bb_high   = bb.bollinger_hband()
        bb_low    = bb.bollinger_lband()
        bb_middle = bb.bollinger_mavg()

        rsi      = RSIIndicator(close=close, window=14).rsi()
        macd_ind = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
        macd        = macd_ind.macd()
        macd_signal = macd_ind.macd_signal()
        macd_diff   = macd_ind.macd_diff()
        atr         = AverageTrueRange(
            high=high, low=low, close=close, window=14
        ).average_true_range()

        safe_close = close.replace(0, np.nan)
        safe_mid   = bb_middle.replace(0, np.nan)
        safe_high  = bb_high.replace(0, np.nan)
        safe_low   = bb_low.replace(0, np.nan)

        df_ind['Sector_EMA_10_Gap']           = close / ema10 - 1.0
        df_ind['Sector_EMA_14_Gap']           = close / ema14 - 1.0
        df_ind['Sector_EMA_20_Gap']           = close / ema20 - 1.0
        df_ind['Sector_SMA_10_Gap']           = close / sma10 - 1.0
        df_ind['Sector_SMA_20_Gap']           = close / sma20 - 1.0
        df_ind['Sector_SMA_50_Gap']           = close / sma50 - 1.0
        df_ind['Sector_BB_High_Distance']     = close / safe_high - 1.0
        df_ind['Sector_BB_Low_Distance']      = close / safe_low - 1.0
        df_ind['Sector_BB_Middle_Distance']   = close / safe_mid - 1.0
        df_ind['Sector_BB_Width_Normalized']  = (bb_high - bb_low) / safe_mid
        df_ind['Sector_BB_Pct']               = bb.bollinger_pband().values
        df_ind['Sector_RSI_14']               = rsi.values
        df_ind['Sector_MACD_Normalized']      = macd / safe_close
        df_ind['Sector_MACD_Signal_Normalized'] = macd_signal / safe_close
        df_ind['Sector_MACD_Diff_Normalized'] = macd_diff / safe_close
        df_ind['Sector_ATR_Normalized']       = atr / safe_close

        raw_ind_cols = [
            'Sector_EMA_10_Gap', 'Sector_EMA_14_Gap', 'Sector_EMA_20_Gap',
            'Sector_SMA_10_Gap', 'Sector_SMA_20_Gap', 'Sector_SMA_50_Gap',
            'Sector_BB_High_Distance', 'Sector_BB_Low_Distance',
            'Sector_BB_Middle_Distance', 'Sector_BB_Width_Normalized',
            'Sector_BB_Pct', 'Sector_RSI_14',
            'Sector_MACD_Normalized', 'Sector_MACD_Signal_Normalized',
            'Sector_MACD_Diff_Normalized', 'Sector_ATR_Normalized'
        ]

        for col in raw_ind_cols:
            df_ind[f"{col}_Lag1"] = df_ind[col].shift(1)

        keep_cols = (
            ['Date', 'Sector', 'Sector_Close', 'Sector_High', 'Sector_Low',
             'Sector_Return', 'Sector_Volume_Sum', 'Log_Volume',
             'Vol_5', 'Vol_10', 'Vol_5_Lag1', 'Vol_10_Lag1', 'Realized_Variance']
            + [f"{col}_Lag1" for col in raw_ind_cols]
        )
        return df_ind[keep_cols]

    except Exception as e:
        print(f"   ⚠️ Indicator computation failed: {e}")
        return None


all_sectors_indicators = []
for sector in tqdm(sector_data_merged['Sector'].unique(), desc="Indicators"):
    result = compute_indicators(
        sector_data_merged[sector_data_merged['Sector'] == sector].copy()
    )
    if result is not None:
        all_sectors_indicators.append(result)

final_data_with_indicators = pd.concat(
    all_sectors_indicators, ignore_index=True
).dropna()
print(f"✅ Indicators (with Lag1): {final_data_with_indicators.shape}")

del all_sectors_indicators, sector_data_merged, df
gc.collect()
log_memory("After indicators — freed all pre-training data")

# ================================================================
# ✅ STACKING META-LEARNER
# ================================================================
def stacking_meta_learner_with_indicator(
    dl_preds, garch_preds, indicator_vals, baseline_preds, actuals,
    cal_end, stack_end, epsilon=1e-8
):
    dl_flat       = np.clip(np.asarray(dl_preds, dtype=float).flatten(), epsilon, None)
    garch_flat    = np.clip(np.asarray(garch_preds, dtype=float).flatten(), epsilon, None)
    ind_flat      = np.asarray(indicator_vals, dtype=float).flatten()
    baseline_flat = np.clip(np.asarray(baseline_preds, dtype=float).flatten(), epsilon, None)
    actuals       = np.clip(np.asarray(actuals, dtype=float).flatten(), epsilon, None)
    n             = len(dl_flat)

    if not (len(garch_flat) == n and len(ind_flat) == n
            and len(baseline_flat) == n and len(actuals) == n):
        raise ValueError("Meta-learner inputs do not have identical lengths.")

    def _fallback():
        hybrid      = baseline_flat.copy()
        train_qlike = (
            qlike_np(actuals[cal_end:stack_end], hybrid[cal_end:stack_end], epsilon)
            if stack_end > cal_end else np.nan
        )
        test_qlike  = (
            qlike_np(actuals[stack_end:], hybrid[stack_end:], epsilon)
            if stack_end < n else np.nan
        )
        return (hybrid, np.nan, np.nan, 0.0, np.nan, np.nan, np.nan, 0, 0,
                train_qlike, test_qlike, True)

    if ((stack_end - cal_end) < 5 or (n - stack_end) < 5
            or not np.all(np.isfinite(ind_flat))):
        return _fallback()

    dl_train        = dl_flat[cal_end:stack_end]
    garch_train     = garch_flat[cal_end:stack_end]
    indicator_train = ind_flat[cal_end:stack_end]
    y_train         = actuals[cal_end:stack_end]
    baseline_train  = baseline_flat[cal_end:stack_end]

    actual_cv_splits = min(CV_SPLITS, max(2, len(y_train) // 8))
    if actual_cv_splits < 2:
        return _fallback()

    try:
        tscv = TimeSeriesSplit(n_splits=actual_cv_splits)
        chronological_folds = []

        for train_index, validation_index in tscv.split(y_train):
            if len(train_index) < 3 or len(validation_index) < 2:
                continue
            fold_parameters = indicator_scaling_parameters(
                indicator_train[train_index],
                clip_low=INDICATOR_CLIP_LOW, clip_high=INDICATOR_CLIP_HIGH,
                epsilon=epsilon
            )
            validation_z = standardize_indicator(
                indicator_train[validation_index], fold_parameters
            )
            chronological_folds.append({
                "DL":          dl_train[validation_index],
                "GARCH":       garch_train[validation_index],
                "Indicator_Z": validation_z,
                "Actual":      y_train[validation_index],
                "Baseline":    baseline_train[validation_index]
            })

        if not chronological_folds:
            return _fallback()

        cv_total_folds = len(chronological_folds)

        baseline_fold_qlikes   = [
            qlike_np(fold["Actual"], fold["Baseline"], epsilon)
            for fold in chronological_folds
        ]
        mean_baseline_cv_qlike = float(np.mean(baseline_fold_qlikes))

        best_weight_dl     = 0.50
        best_beta          = 0.0
        best_cv_qlike      = np.inf
        best_winning_folds = 0

        for weight_dl in CONVEX_WEIGHT_GRID:
            weight_garch = 1.0 - float(weight_dl)
            for beta in INDICATOR_BETA_GRID:
                fold_qlikes   = []
                winning_folds = 0
                for fold, baseline_q in zip(chronological_folds, baseline_fold_qlikes):
                    base_prediction = np.clip(
                        float(weight_dl) * fold["DL"] + weight_garch * fold["GARCH"],
                        epsilon, None
                    )
                    adjustment = bounded_indicator_multiplier(
                        fold["Indicator_Z"], float(beta),
                        lower=INDICATOR_ADJUSTMENT_LOW,
                        upper=INDICATOR_ADJUSTMENT_HIGH
                    )
                    enhanced_prediction = np.clip(
                        base_prediction * adjustment, epsilon, None
                    )
                    candidate_qlike = qlike_np(
                        fold["Actual"], enhanced_prediction, epsilon
                    )
                    fold_qlikes.append(candidate_qlike)
                    if candidate_qlike < baseline_q:
                        winning_folds += 1

                mean_candidate_qlike       = float(np.mean(fold_qlikes))
                is_better                  = mean_candidate_qlike < best_cv_qlike - 1e-12
                is_tie_with_smaller_beta   = (
                    abs(mean_candidate_qlike - best_cv_qlike) <= 1e-12
                    and abs(float(beta)) < abs(best_beta)
                )
                if is_better or is_tie_with_smaller_beta:
                    best_cv_qlike      = mean_candidate_qlike
                    best_weight_dl     = float(weight_dl)
                    best_beta          = float(beta)
                    best_winning_folds = int(winning_folds)

        best_weight_garch   = 1.0 - best_weight_dl
        best_cv_improvement = (
            (mean_baseline_cv_qlike - best_cv_qlike)
            / mean_baseline_cv_qlike * 100.0
            if mean_baseline_cv_qlike > 0 else -np.inf
        )

        full_train_parameters = indicator_scaling_parameters(
            indicator_train, clip_low=INDICATOR_CLIP_LOW,
            clip_high=INDICATOR_CLIP_HIGH, epsilon=epsilon
        )
        all_indicator_z = standardize_indicator(ind_flat, full_train_parameters)
        all_base        = np.clip(
            best_weight_dl * dl_flat + best_weight_garch * garch_flat,
            epsilon, None
        )
        all_adjustment  = bounded_indicator_multiplier(
            all_indicator_z, best_beta,
            lower=INDICATOR_ADJUSTMENT_LOW, upper=INDICATOR_ADJUSTMENT_HIGH
        )
        all_hybrid = np.clip(all_base * all_adjustment, epsilon, None)

        if not np.all(np.isfinite(all_hybrid)) or np.any(all_hybrid <= 0):
            return _fallback()

        train_qlike = qlike_np(y_train, all_hybrid[cal_end:stack_end], epsilon)
        test_qlike  = qlike_np(actuals[stack_end:], all_hybrid[stack_end:], epsilon)

        return (
            all_hybrid, best_weight_dl, best_weight_garch, best_beta,
            best_cv_qlike, mean_baseline_cv_qlike, best_cv_improvement,
            best_winning_folds, cv_total_folds, train_qlike, test_qlike, False
        )

    except Exception as error:
        print(f"   ⚠️ Constrained indicator meta-learner failed: {error}")
        return _fallback()

# ================================================================
# 🔬 ENHANCED STACKING LOOP
# ================================================================
print("\n📊 Step 3: Enhanced Stacking...")
print(f"   ✅ Issue 1:  Three-period split: cal[:{int(CAL_RATIO*100)}%] "
      f"→ constrained CV[{int(CAL_RATIO*100)}%:{int(STACK_TRAIN_END*100)}%] "
      f"→ test[{int(STACK_TRAIN_END*100)}%:]")
print("   ✅ Issue 2:  Overall weighted baseline on matched dates")
print("   ✅ Issue 3:  Baseline RMSE/MAE computed from comp_test")
print("   ✅ Issue 4:  Explicit hybrid column detection")
print("   ✅ Issue 5:  Visualizations use Baseline_QLIKE_Same_Dates")
print(f"   ✅ Issue 6:  Code version guard ({CODE_VERSION})")
print("   ✅ Efficiency: Transformer trained once per sector-indicator")
print("   ✅ QLIKE stability: constrained convex base + bounded indicator adjustment")
print("   ✅ CV gate: indicator must improve baseline in training folds")
print("   ✅ Conservative robustness: enhanced model must win all CV folds")
print("   ✅ Indicator clipping: training-only 1st-99th percentiles")
print("   ✅ Base weights: nonnegative and constrained to sum to one")
print("   ✅ Indicator beta: bounded to [-0.50, 0.50]")
print("   ✅ Indicator multiplier: bounded to [0.50, 2.00]")
print("   ✅ Indicators: 16 candidates, each evaluated separately")
print("   ✅ Memory:    pre-training data freed before loop")
print("   ✅ REPRODUCIBILITY: deterministic T4 GPU + float32 + fixed per-build seed")

baseline_features  = ['Vol_5_Lag1', 'Vol_10_Lag1', 'Log_Volume']
all_indicators_raw = [
    'Sector_EMA_10_Gap', 'Sector_EMA_14_Gap', 'Sector_EMA_20_Gap',
    'Sector_SMA_10_Gap', 'Sector_SMA_20_Gap', 'Sector_SMA_50_Gap',
    'Sector_BB_High_Distance', 'Sector_BB_Low_Distance',
    'Sector_BB_Middle_Distance', 'Sector_BB_Width_Normalized',
    'Sector_BB_Pct', 'Sector_RSI_14',
    'Sector_MACD_Normalized', 'Sector_MACD_Signal_Normalized',
    'Sector_MACD_Diff_Normalized', 'Sector_ATR_Normalized'
]
all_indicators       = [f"{ind}_Lag1" for ind in all_indicators_raw]
available_indicators = [
    ind for ind in all_indicators
    if ind in final_data_with_indicators.columns
]
print(f"   Available lagged indicators: {len(available_indicators)}")

enhanced_hybrid_results   = []
enhanced_forecasts_all    = []
dm_test_records           = []
already_completed_sectors = set()

if checkpoint_valid:
    _resumed = pd.read_csv(CHECKPOINT_FILE)
    enhanced_hybrid_results   = _resumed.to_dict('records')
    already_completed_sectors = set(_resumed['Sector'].unique())
    print(f"\n🔄 RESUMING: {len(enhanced_hybrid_results)} results, "
          f"done: {sorted(already_completed_sectors)}")

    if os.path.exists(FORECAST_CHECKPOINT_FILE):
        _rf         = pd.read_csv(FORECAST_CHECKPOINT_FILE)
        _rf['Date'] = pd.to_datetime(_rf['Date'], dayfirst=True, errors='coerce')
        enhanced_forecasts_all = [_rf]
        print(f"   Resumed {len(_rf)} forecast rows")

    if os.path.exists(DM_LOSS_CHECKPOINT_FILE):
        _dm_cp = pd.read_csv(DM_LOSS_CHECKPOINT_FILE)
        for (sec, ind), grp in _dm_cp.groupby(['Sector', 'Indicator']):
            dm_test_records.append({
                'Sector':      sec,
                'Indicator':   ind,
                'Dates':       grp['Date'].tolist(),
                'Base_Losses': grp['Base_QLIKE'].tolist(),
                'Enh_Losses':  grp['Enh_QLIKE'].tolist(),
                'Loss_Diffs':  grp['Loss_Diff_dt'].tolist()
            })
        print(f"   Resumed {len(dm_test_records)} DM records")

log_memory("Before Enhanced Training")

sectors_to_process = [
    s for s in baseline_results['Sector'].unique()
    if s not in already_completed_sectors
]
print(f"\n   Sectors remaining: {len(sectors_to_process)} / "
      f"{len(baseline_results['Sector'].unique())}")

n_sectors = len(sectors_to_process)
for sector_idx, sector in enumerate(sectors_to_process, 1):
    pct = sector_idx / n_sectors * 100
    print(f"\n{'='*70}")
    print(f"🔄 Sector {sector_idx}/{n_sectors} ({pct:.0f}%) — {sector}")
    print(f"{'='*70}")

    sector_data = final_data_with_indicators[
        final_data_with_indicators['Sector'] == sector
    ].copy().sort_values('Date').reset_index(drop=True)

    if len(sector_data) < 200:
        print(f"   ⚠️ Skipping — insufficient data ({len(sector_data)} rows)")
        continue

    baseline_row = baseline_results[baseline_results['Sector'] == sector]
    if len(baseline_row) == 0:
        print(f"   ⚠️ Skipping — no baseline row found")
        continue

    weight                = baseline_row['Weight'].values[0]
    tgarch_date_set       = tgarch_dates_by_sector.get(sector, set())
    baseline_sector_df    = baseline_forecasts[
        baseline_forecasts['Sector'] == sector
    ].copy()
    baseline_sector_dates = set(pd.to_datetime(baseline_sector_df['Date']))

    target_dates = sorted([
        d for d in baseline_sector_dates if d in tgarch_date_set
    ])
    print(f"   Target forecast dates: {len(target_dates)}")

    if len(target_dates) == 0:
        print(f"   ⚠️ Skipping — no overlapping baseline+TGARCH dates")
        continue

    n_indicators = len(available_indicators)
    for indicator_idx, indicator in enumerate(available_indicators, 1):
        ind_label = indicator.replace('Sector_', '').replace('_Lag1', '')
        print(f"   [{indicator_idx:2d}/{n_indicators}] "
              f"({indicator_idx/n_indicators*100:.0f}%) "
              f"Testing: {ind_label}", end='')

        enhanced_features = baseline_features + [indicator]
        s_df_clean = sector_data.dropna(
            subset=enhanced_features + [target]
        ).reset_index(drop=True)

        if len(s_df_clean) < 200:
            print(f" — skipped (insufficient rows)")
            gc.collect()
            continue

        X_raw  = s_df_clean[enhanced_features].values
        y_raw  = s_df_clean[target].values
        dates  = pd.to_datetime(s_df_clean['Date'].values)
        n_rows = len(X_raw)

        train_end = int(n_rows * TRAIN_RATIO)
        X_mean    = X_raw[:train_end].mean(axis=0)
        X_std     = X_raw[:train_end].std(axis=0) + epsilon
        X_scaled  = (X_raw - X_mean) / X_std

        X_seq = np.array([X_scaled[i:i + SEQ_LEN] for i in range(n_rows - SEQ_LEN)])
        y_seq = y_raw[SEQ_LEN:]
        d_seq = dates[SEQ_LEN:]

        if len(X_seq) < 100:
            print(f" — skipped (too short)")
            del X_raw, y_raw, dates, X_scaled, X_seq, y_seq, d_seq
            gc.collect()
            continue

        split    = int(len(X_seq) * TRAIN_RATIO)
        val_size = max(10, int(split * VAL_RATIO))

        X_tr = X_seq[:split - val_size]
        y_tr = y_seq[:split - val_size]
        X_vl = X_seq[split - val_size:split]
        y_vl = y_seq[split - val_size:split]

        if len(X_tr) < 30 or len(X_vl) < 10:
            print(f" — skipped (train/val too small)")
            del X_raw, y_raw, dates, X_scaled, X_seq, y_seq, d_seq
            gc.collect()
            continue

        tf.keras.backend.clear_session()
        gc.collect()

        model      = build_transformer_qlike((SEQ_LEN, len(enhanced_features)))
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE, restore_best_weights=True
        )
        history = model.fit(
            X_tr, y_tr, validation_data=(X_vl, y_vl),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            verbose=0, shuffle=False, callbacks=[early_stop]
        )

        training_diagnostics.append({
            "Sector":           sector,
            "Indicator":        indicator,
            "Epochs_Run":       len(history.history["loss"]),
            "Final_Train_Loss": float(history.history["loss"][-1]),
            "Best_Val_Loss":    float(np.min(history.history["val_loss"]))
        })

        X_test_all = X_seq[split:]
        d_test_all = d_seq[split:]

        raw_preds = fast_predict(
            model, tf.constant(X_test_all, dtype=tf.float32)
        ).numpy().flatten()

        del model, X_tr, y_tr, X_vl, y_vl, X_seq, X_scaled
        tf.keras.backend.clear_session()
        gc.collect()

        forecasts_raw  = []
        dates_out      = []
        indicator_vals = []

        for j, fd in enumerate(d_test_all):
            if fd in baseline_sector_dates and fd in tgarch_date_set:
                forecasts_raw.append(float(raw_preds[j]))
                dates_out.append(fd)
                target_row = SEQ_LEN + split + j
                if target_row >= len(X_raw):
                    continue
                indicator_vals.append(
                    float(X_raw[target_row, len(baseline_features)])
                )

        del X_raw, y_raw, dates, d_test_all, raw_preds
        gc.collect()

        if not forecasts_raw:
            print(f" — no matching dates")
            gc.collect()
            continue

        print(f" — {len(forecasts_raw)} forecasts")

        raw_fc_df = pd.DataFrame({
            'Date': pd.to_datetime(dates_out), 'Sector': sector, 'Raw_DL': forecasts_raw
        })
        aligned = raw_fc_df.merge(
            eval_target_df[['Sector', 'Date', 'Eval_Variance']],
            on=['Sector', 'Date'], how='inner'
        ).sort_values('Date').reset_index(drop=True)
        del raw_fc_df
        gc.collect()

        if len(aligned) == 0:
            del aligned
            gc.collect()
            continue

        actuals       = aligned['Eval_Variance'].clip(lower=epsilon).values
        dates_aligned = aligned['Date'].tolist()
        del aligned
        gc.collect()

        n_aligned = len(actuals)
        cal_end   = max(int(n_aligned * CAL_RATIO), 3)
        stack_end = max(int(n_aligned * STACK_TRAIN_END), cal_end + 5)

        if stack_end >= n_aligned or (n_aligned - stack_end) < 5:
            gc.collect()
            continue

        keep_mask   = [d in set(dates_aligned) for d in pd.to_datetime(dates_out)]
        fc_aligned  = [forecasts_raw[k] for k, m in enumerate(keep_mask) if m]
        ind_aligned = [indicator_vals[k] for k, m in enumerate(keep_mask) if m]

        cf     = np.mean(actuals[:cal_end] / (np.array(fc_aligned[:cal_end]) + epsilon))
        dl_cal = np.clip(np.array(fc_aligned) * cf, epsilon, None)
        del forecasts_raw, fc_aligned
        gc.collect()

        tgarch_merge = pd.DataFrame({'Date': dates_aligned}).merge(
            tgarch_clean[tgarch_clean['Sector'] == sector][['Date', 'TGARCH_Variance']],
            on='Date', how='inner'
        )
        if len(tgarch_merge) == 0:
            del tgarch_merge, dl_cal
            gc.collect()
            continue

        exact_dates  = set(tgarch_merge['Date'])
        keep2        = [d in exact_dates for d in dates_aligned]
        d_fin        = [d for d, m in zip(dates_aligned, keep2) if m]
        dl_fin       = dl_cal[keep2]
        ind_fin      = np.array(ind_aligned)[keep2]
        act_fin      = actuals[keep2]
        tg_fin       = tgarch_merge.set_index('Date').loc[d_fin, 'TGARCH_Variance'].values
        baseline_fin = baseline_sector_df.set_index('Date').loc[
            d_fin, 'Baseline_Hybrid_Variance'
        ].values

        del tgarch_merge, dl_cal, ind_aligned, dates_aligned, actuals, dates_out, indicator_vals
        gc.collect()

        tg_fin  = np.clip(tg_fin,  epsilon, None)
        dl_fin  = np.clip(dl_fin,  epsilon, None)
        act_fin = np.clip(act_fin, epsilon, None)

        n_fin     = len(act_fin)
        cal_end_f = max(int(n_fin * CAL_RATIO), 3)
        stk_end_f = max(int(n_fin * STACK_TRAIN_END), cal_end_f + 5)

        if stk_end_f >= n_fin or (n_fin - stk_end_f) < 5:
            del dl_fin, tg_fin, ind_fin, baseline_fin, act_fin
            gc.collect()
            continue

        (hybrid_enh, ew_dl, ew_g, ew_ind,
         cv_qlike, baseline_cv_qlike,
         cv_improvement, cv_winning_folds, cv_total_folds,
         trn_qlike, tst_qlike,
         fallback) = stacking_meta_learner_with_indicator(
            dl_fin, tg_fin, ind_fin, baseline_fin, act_fin,
            cal_end=cal_end_f, stack_end=stk_end_f, epsilon=epsilon
        )

        hybrid_full = np.asarray(hybrid_enh).copy()
        comparison  = pd.DataFrame({
            'Date': d_fin, 'Actual': act_fin, 'Enhanced': hybrid_full
        }).merge(
            baseline_sector_df[['Date', 'Baseline_Hybrid_Variance']],
            on='Date', how='inner'
        ).sort_values('Date').reset_index(drop=True)
        del hybrid_enh
        gc.collect()

        split_c   = max(int(len(comparison) * STACK_TRAIN_END), 3)
        comp_test = comparison.iloc[split_c:].copy()
        del comparison
        gc.collect()

        if len(comp_test) < 5:
            del comp_test, dl_fin, tg_fin, ind_fin, act_fin
            gc.collect()
            continue

        bq  = qlike_np(comp_test['Actual'].values,
                        comp_test['Baseline_Hybrid_Variance'].values, epsilon)
        br  = np.sqrt(np.mean(
            (comp_test['Actual'].values - comp_test['Baseline_Hybrid_Variance'].values) ** 2
        ))
        bm  = np.mean(np.abs(
            comp_test['Actual'].values - comp_test['Baseline_Hybrid_Variance'].values
        ))
        eq  = qlike_np(comp_test['Actual'].values, comp_test['Enhanced'].values, epsilon)
        er  = np.sqrt(np.mean(
            (comp_test['Actual'].values - comp_test['Enhanced'].values) ** 2
        ))
        em  = np.mean(np.abs(comp_test['Actual'].values - comp_test['Enhanced'].values))
        imp = (bq - eq) / bq * 100 if bq > 0 else 0.0

        enh_l  = qlike_series(comp_test['Actual'].values,
                               comp_test['Enhanced'].values, epsilon)
        base_l = qlike_series(comp_test['Actual'].values,
                               comp_test['Baseline_Hybrid_Variance'].values, epsilon)
        dm_s, dm_p, dm_md = diebold_mariano_test(base_l, enh_l, h=1)

        enhanced_hybrid_results.append({
            'Sector':                       sector,
            'Indicator':                    indicator,
            'Weight':                       weight,
            'Baseline_Hybrid_QLIKE':        baseline_row['Baseline_Hybrid_QLIKE'].values[0],
            'Baseline_QLIKE_Same_Dates':    round(bq, 6),
            'Baseline_RMSE_Same_Dates':     round(br, 6),
            'Baseline_MAE_Same_Dates':      round(bm, 6),
            'CV_QLIKE':                     round(cv_qlike, 6) if not np.isnan(cv_qlike) else np.nan,
            'Baseline_CV_QLIKE':            round(baseline_cv_qlike, 6) if not np.isnan(baseline_cv_qlike) else np.nan,
            'CV_Improvement_vs_Baseline_%': round(cv_improvement, 4) if np.isfinite(cv_improvement) else np.nan,
            'CV_Winning_Folds':             int(cv_winning_folds),
            'CV_Total_Folds':               int(cv_total_folds),
            'Indicator_Beta':               round(ew_ind, 4),
            'Indicator_Effect_Nonzero':     bool(np.isfinite(ew_ind) and abs(ew_ind) >= MIN_ABS_INDICATOR_BETA),
            'Base_Weight_Sum':              round(ew_dl + ew_g, 10) if np.isfinite(ew_dl) and np.isfinite(ew_g) else np.nan,
            'Meta_Learner':                 'Constrained_Convex_Base_Bounded_Indicator',
            'Passes_CV_Gate':               bool(
                np.isfinite(cv_improvement) and cv_improvement >= MIN_CV_IMPROVEMENT
                and cv_winning_folds >= MIN_WINNING_FOLDS
                and np.isfinite(ew_ind) and abs(ew_ind) >= MIN_ABS_INDICATOR_BETA
            ),
            'Passes_Conservative_CV_Gate':  bool(
                np.isfinite(cv_improvement) and cv_improvement >= MIN_CV_IMPROVEMENT
                and int(cv_total_folds) >= 2
                and int(cv_winning_folds) == int(cv_total_folds)
                and np.isfinite(ew_ind) and abs(ew_ind) >= MIN_ABS_INDICATOR_BETA
            ),
            'Train_Hybrid_QLIKE':           round(trn_qlike, 6),
            'Enhanced_Hybrid_QLIKE':        round(eq, 6),
            'Enhanced_Hybrid_RMSE':         round(er, 6),
            'Enhanced_Hybrid_MAE':          round(em, 6),
            'Improvement_%':                round(imp, 4),
            'Learned_Weight_DL':            round(ew_dl, 4) if np.isfinite(ew_dl) else np.nan,
            'Learned_Weight_GARCH':         round(ew_g,  4) if np.isfinite(ew_g) else np.nan,
            'Learned_Weight_Indicator':     round(ew_ind, 4),
            'Fallback_Used':                fallback,
            'DM_Stat':                      round(dm_s, 4) if not np.isnan(dm_s) else np.nan,
            'DM_PValue':                    round(dm_p, 4) if not np.isnan(dm_p) else np.nan,
            'DM_MeanLossDiff':              round(dm_md, 6),
            'DM_Significant_5pct':          bool(dm_p < 0.05) if not np.isnan(dm_p) else False,
            'DM_Significant_10pct':         bool(dm_p < 0.10) if not np.isnan(dm_p) else False,
            'N_Test_Obs':                   len(comp_test),
            'N_Predictions':                len(d_fin),
            'Code_Version':                 CODE_VERSION,
            'Reproducibility_Version':       REPRODUCIBILITY_VERSION,
            'Execution_Device':              GPU_NAME
        })

        dm_test_records.append({
            'Sector':      sector,
            'Indicator':   indicator,
            'Dates':       [str(d) for d in comp_test['Date'].tolist()],
            'Base_Losses': base_l.tolist(),
            'Enh_Losses':  enh_l.tolist(),
            'Loss_Diffs':  (base_l - enh_l).tolist()
        })

        ct_dates = set(pd.to_datetime(comp_test['Date']))
        temp_df  = pd.DataFrame({
            'Date':                     pd.to_datetime(d_fin),
            'Sector':                   sector,
            'Indicator':                indicator,
            'Actual_Variance':          act_fin,
            'Transformer_Forecast':     dl_fin,
            'TGARCH_Variance':          tg_fin,
            'Indicator_Value':          ind_fin,
            'Enhanced_Hybrid_Variance': hybrid_full,
            'Learned_Weight_DL':        ew_dl,
            'Learned_Weight_GARCH':     ew_g,
            'Learned_Weight_Indicator': ew_ind,
            'Indicator_Beta':           ew_ind,
            'Base_Weight_Sum':          (ew_dl + ew_g if np.isfinite(ew_dl) and np.isfinite(ew_g) else np.nan),
            'Meta_Learner':             'Constrained_Convex_Base_Bounded_Indicator',
            'Fallback_Used':            fallback,
            'Code_Version':             CODE_VERSION,
            'Reproducibility_Version': REPRODUCIBILITY_VERSION,
            'Execution_Device':        GPU_NAME
        })

        temp_df['Evaluation_Split'] = 'Calibration'
        temp_df.loc[cal_end_f:stk_end_f - 1, 'Evaluation_Split'] = 'Stacking_Train'
        temp_df.loc[stk_end_f:, 'Evaluation_Split'] = 'Final_Test'
        temp_df.loc[temp_df['Date'].isin(ct_dates), 'Evaluation_Split'] = 'Final_Test_Matched'

        enhanced_forecasts_all.append(temp_df)

        del temp_df, comp_test, hybrid_full, dl_fin, tg_fin, ind_fin, act_fin, base_l, enh_l, d_fin
        gc.collect()

    n_this = sum(1 for r in enhanced_hybrid_results if r['Sector'] == sector)
    print(f"\n   ✅ {sector} complete — {n_this} indicator results recorded")
    print(f"   📊 Overall progress: {sector_idx}/{n_sectors} sectors "
          f"({sector_idx/n_sectors*100:.0f}%)")
    print(f"   💾 Total results so far: {len(enhanced_hybrid_results)}")

    clear_memory()

    if enhanced_hybrid_results:
        pd.DataFrame(enhanced_hybrid_results).to_csv(CHECKPOINT_FILE, index=False)
        n_done = len(set(r['Sector'] for r in enhanced_hybrid_results))
        print(f"   💾 Checkpoint saved: {len(enhanced_hybrid_results)} results "
              f"({n_done} sectors done)")

    if enhanced_forecasts_all:
        pd.concat(enhanced_forecasts_all, ignore_index=True).to_csv(
            FORECAST_CHECKPOINT_FILE, index=False
        )

    if dm_test_records:
        dm_rows = []
        for rec in dm_test_records:
            for d, bl, el, diff in zip(
                rec['Dates'], rec['Base_Losses'], rec['Enh_Losses'], rec['Loss_Diffs']
            ):
                dm_rows.append({
                    'Sector': rec['Sector'], 'Indicator': rec['Indicator'],
                    'Date': d, 'Base_QLIKE': bl, 'Enh_QLIKE': el,
                    'Loss_Diff_dt': diff, 'Code_Version': CODE_VERSION, 'Reproducibility_Version': REPRODUCIBILITY_VERSION, 'Execution_Device': GPU_NAME
                })
        pd.DataFrame(dm_rows).to_csv(DM_LOSS_CHECKPOINT_FILE, index=False)

log_memory("After Enhanced Training")

# ================================================================
# 💾 SAVE ENHANCED RESULTS
# ================================================================
excluded_sectors = []
best_rows        = []

if enhanced_hybrid_results:
    enhanced_results_df = pd.DataFrame(enhanced_hybrid_results)
    enhanced_results_df.to_csv(
        'Enhanced_Stacking_Transformer_TGARCH_Results.csv', index=False
    )
    print(f"\n✅ Enhanced results: {len(enhanced_results_df)} rows")
    print(f"   Fallback count: {enhanced_results_df['Fallback_Used'].sum()}")

    def select_best(group):
        eligible = group[
            (group['Fallback_Used'] == False) &
            (group['CV_QLIKE'].notna()) &
            (group['Passes_CV_Gate'] == True)
        ]
        if len(eligible) == 0:
            return None
        return eligible.loc[eligible['CV_QLIKE'].idxmin()]

    for sector, grp in enhanced_results_df.groupby('Sector'):
        result = select_best(grp)
        if result is None:
            excluded_sectors.append(sector)
            print(f"   ℹ️  {sector}: no indicator passed the CV gate — baseline retained")
        else:
            best_rows.append(result)

    if not best_rows:
        print("❌ No valid enhanced models found for any sector.")
    else:
        best_per_sector = pd.DataFrame(best_rows).reset_index(drop=True)
        print(f"\n✅ Valid enhanced sectors: {len(best_per_sector)}")
        if excluded_sectors:
            print(f"   Baseline retained: {excluded_sectors}")

        best_per_sector.to_csv(
            'Best_Enhanced_Stacking_Transformer_TGARCH_Per_Sector.csv', index=False
        )

        sector_summary_enhanced = best_per_sector[[
            'Sector', 'Weight', 'Indicator',
            'CV_QLIKE', 'Baseline_CV_QLIKE',
            'CV_Improvement_vs_Baseline_%', 'CV_Winning_Folds', 'CV_Total_Folds',
            'Passes_CV_Gate', 'Passes_Conservative_CV_Gate', 'Train_Hybrid_QLIKE',
            'Baseline_QLIKE_Same_Dates', 'Enhanced_Hybrid_QLIKE',
            'Baseline_RMSE_Same_Dates',  'Enhanced_Hybrid_RMSE',
            'Baseline_MAE_Same_Dates',   'Enhanced_Hybrid_MAE',
            'DM_Stat', 'DM_PValue', 'DM_Significant_5pct', 'DM_Significant_10pct'
        ]].copy()
        sector_summary_enhanced.to_csv(
            'Sector_Summary_Metrics_Enhanced_Stacking_Transformer_TGARCH.csv', index=False
        )

        if enhanced_forecasts_all:
            all_enhanced_forecasts = pd.concat(enhanced_forecasts_all, ignore_index=True)
            all_enhanced_forecasts['Date'] = pd.to_datetime(
                all_enhanced_forecasts['Date'], dayfirst=True, errors='coerce'
            )
            best_ind_dict = best_per_sector.set_index('Sector')['Indicator'].to_dict()
            best_enhanced_forecasts = all_enhanced_forecasts[
                all_enhanced_forecasts.apply(
                    lambda row: best_ind_dict.get(row['Sector']) == row['Indicator'], axis=1
                )
            ].copy()
            best_enhanced_forecasts.to_csv(
                'Enhanced_Stacking_Transformer_TGARCH_Forecasts.csv', index=False
            )
            comp_test_only = best_enhanced_forecasts[
                best_enhanced_forecasts['Evaluation_Split'] == 'Final_Test_Matched'
            ].copy()
            comp_test_only.to_csv(
                'Enhanced_Stacking_CompTest_Final_Test_Matched.csv', index=False
            )
            print(f"✅ comp_test export: {len(comp_test_only)} rows")

        valid_weight_total           = best_per_sector['Weight'].sum()
        weighted_baseline_same_dates = (
            best_per_sector['Weight'] * best_per_sector['Baseline_QLIKE_Same_Dates']
        ).sum() / valid_weight_total
        weighted_enhanced_qlike      = (
            best_per_sector['Weight'] * best_per_sector['Enhanced_Hybrid_QLIKE']
        ).sum() / valid_weight_total
        overall_improvement          = (
            (weighted_baseline_same_dates - weighted_enhanced_qlike)
            / weighted_baseline_same_dates * 100
        ) if weighted_baseline_same_dates != 0 else 0.0
        weighted_baseline_rmse_same  = (
            best_per_sector['Weight'] * best_per_sector['Baseline_RMSE_Same_Dates']
        ).sum() / valid_weight_total
        weighted_enhanced_rmse       = (
            best_per_sector['Weight'] * best_per_sector['Enhanced_Hybrid_RMSE']
        ).sum() / valid_weight_total
        weighted_baseline_mae_same   = (
            best_per_sector['Weight'] * best_per_sector['Baseline_MAE_Same_Dates']
        ).sum() / valid_weight_total
        weighted_enhanced_mae        = (
            best_per_sector['Weight'] * best_per_sector['Enhanced_Hybrid_MAE']
        ).sum() / valid_weight_total

        # Full-market final strategy
        selected_lookup      = best_per_sector.set_index('Sector')
        market_strategy_rows = []

        for _, baseline_market_row in baseline_results.iterrows():
            market_sector = baseline_market_row['Sector']
            market_weight = float(baseline_market_row['Weight'])

            if market_sector in selected_lookup.index:
                selected_row       = selected_lookup.loc[market_sector]
                strategy_model     = 'Enhanced'
                strategy_indicator = selected_row['Indicator']
                comparator_qlike   = float(selected_row['Baseline_QLIKE_Same_Dates'])
                comparator_rmse    = float(selected_row['Baseline_RMSE_Same_Dates'])
                comparator_mae     = float(selected_row['Baseline_MAE_Same_Dates'])
                strategy_qlike     = float(selected_row['Enhanced_Hybrid_QLIKE'])
                strategy_rmse      = float(selected_row['Enhanced_Hybrid_RMSE'])
                strategy_mae       = float(selected_row['Enhanced_Hybrid_MAE'])
            else:
                strategy_model     = 'Baseline_Retained'
                strategy_indicator = ''
                comparator_qlike   = float(baseline_market_row['Baseline_Hybrid_QLIKE'])
                comparator_rmse    = float(baseline_market_row['Baseline_Hybrid_RMSE'])
                comparator_mae     = float(baseline_market_row['Baseline_Hybrid_MAE'])
                strategy_qlike     = comparator_qlike
                strategy_rmse      = comparator_rmse
                strategy_mae       = comparator_mae

            market_strategy_rows.append({
                'Sector':                    market_sector,
                'Weight':                    market_weight,
                'Final_Strategy_Model':      strategy_model,
                'Selected_Indicator':        strategy_indicator,
                'Comparator_Baseline_QLIKE': comparator_qlike,
                'Final_Strategy_QLIKE':      strategy_qlike,
                'Comparator_Baseline_RMSE':  comparator_rmse,
                'Final_Strategy_RMSE':       strategy_rmse,
                'Comparator_Baseline_MAE':   comparator_mae,
                'Final_Strategy_MAE':        strategy_mae,
                'Target_SHA256':             TARGET_SHA256,
                'Code_Version':              CODE_VERSION,
                'Reproducibility_Version':  REPRODUCIBILITY_VERSION,
                'Execution_Device':         GPU_NAME
            })

        market_wide_strategy       = pd.DataFrame(market_strategy_rows)
        market_weight_total        = market_wide_strategy['Weight'].sum()
        if market_weight_total <= 0:
            raise ValueError("Full-market sector weights sum to zero.")

        market_wide_strategy['Normalised_Weight'] = (
            market_wide_strategy['Weight'] / market_weight_total
        )
        market_wide_baseline_qlike = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Comparator_Baseline_QLIKE']).sum()
        )
        market_wide_strategy_qlike = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Final_Strategy_QLIKE']).sum()
        )
        market_wide_baseline_rmse  = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Comparator_Baseline_RMSE']).sum()
        )
        market_wide_strategy_rmse  = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Final_Strategy_RMSE']).sum()
        )
        market_wide_baseline_mae   = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Comparator_Baseline_MAE']).sum()
        )
        market_wide_strategy_mae   = float(
            (market_wide_strategy['Normalised_Weight']
             * market_wide_strategy['Final_Strategy_MAE']).sum()
        )
        market_wide_improvement    = (
            (market_wide_baseline_qlike - market_wide_strategy_qlike)
            / market_wide_baseline_qlike * 100
            if market_wide_baseline_qlike > 0 else 0.0
        )

        market_wide_strategy.to_csv('Market_Wide_Final_Strategy_Metrics.csv', index=False)

        # Conservative deployment robustness check
        # This keeps the main selected strategy unchanged and adds a second
        # validation-only deployment rule: enhanced models are retained only
        # if they win all chronological CV folds.
        conservative_best_rows = []
        conservative_strategy_rows = []

        def select_best_conservative(group):
            eligible = group[
                (group['Fallback_Used'] == False) &
                (group['CV_QLIKE'].notna()) &
                (group['Passes_Conservative_CV_Gate'] == True)
            ]
            if len(eligible) == 0:
                return None
            return eligible.loc[eligible['CV_QLIKE'].idxmin()]

        for _, baseline_market_row in baseline_results.iterrows():
            conservative_sector = baseline_market_row['Sector']
            conservative_weight = float(baseline_market_row['Weight'])
            sector_candidates = enhanced_results_df[
                enhanced_results_df['Sector'] == conservative_sector
            ]

            conservative_row = (
                select_best_conservative(sector_candidates)
                if len(sector_candidates) > 0 else None
            )

            if conservative_row is None:
                conservative_decision = 'Baseline retained'
                conservative_indicator = ''
                conservative_baseline_qlike = float(
                    baseline_market_row['Baseline_Hybrid_QLIKE']
                )
                conservative_final_qlike = conservative_baseline_qlike
                conservative_baseline_rmse = float(
                    baseline_market_row['Baseline_Hybrid_RMSE']
                )
                conservative_final_rmse = conservative_baseline_rmse
                conservative_baseline_mae = float(
                    baseline_market_row['Baseline_Hybrid_MAE']
                )
                conservative_final_mae = conservative_baseline_mae
                conservative_improvement = 0.0
            else:
                conservative_best_rows.append(conservative_row)
                conservative_decision = 'Enhanced retained'
                conservative_indicator = conservative_row['Indicator']
                conservative_baseline_qlike = float(
                    conservative_row['Baseline_QLIKE_Same_Dates']
                )
                conservative_final_qlike = float(
                    conservative_row['Enhanced_Hybrid_QLIKE']
                )
                conservative_baseline_rmse = float(
                    conservative_row['Baseline_RMSE_Same_Dates']
                )
                conservative_final_rmse = float(
                    conservative_row['Enhanced_Hybrid_RMSE']
                )
                conservative_baseline_mae = float(
                    conservative_row['Baseline_MAE_Same_Dates']
                )
                conservative_final_mae = float(
                    conservative_row['Enhanced_Hybrid_MAE']
                )
                conservative_improvement = (
                    (conservative_baseline_qlike - conservative_final_qlike)
                    / conservative_baseline_qlike * 100
                    if conservative_baseline_qlike > 0 else 0.0
                )

            conservative_strategy_rows.append({
                'Sector': conservative_sector,
                'Weight': conservative_weight,
                'Conservative_Decision': conservative_decision,
                'Conservative_Indicator': conservative_indicator,
                'Baseline_QLIKE': conservative_baseline_qlike,
                'Conservative_QLIKE': conservative_final_qlike,
                'Improvement_%': conservative_improvement,
                'Baseline_RMSE': conservative_baseline_rmse,
                'Conservative_RMSE': conservative_final_rmse,
                'Baseline_MAE': conservative_baseline_mae,
                'Conservative_MAE': conservative_final_mae,
                'Code_Version': CODE_VERSION,
                'Reproducibility_Version': REPRODUCIBILITY_VERSION,
                'Execution_Device': GPU_NAME
            })

        conservative_strategy = pd.DataFrame(conservative_strategy_rows)
        conservative_weight_total = conservative_strategy['Weight'].sum()
        conservative_strategy['Normalised_Weight'] = (
            conservative_strategy['Weight'] / conservative_weight_total
        )
        conservative_baseline_qlike = float(
            (conservative_strategy['Normalised_Weight']
             * conservative_strategy['Baseline_QLIKE']).sum()
        )
        conservative_final_qlike = float(
            (conservative_strategy['Normalised_Weight']
             * conservative_strategy['Conservative_QLIKE']).sum()
        )
        conservative_improvement = (
            (conservative_baseline_qlike - conservative_final_qlike)
            / conservative_baseline_qlike * 100
            if conservative_baseline_qlike > 0 else 0.0
        )
        conservative_strategy.to_csv(
            'Conservative_Deployment_Robustness_Check.csv', index=False
        )

        indicator_summary = enhanced_results_df[
            enhanced_results_df['Fallback_Used'] == False
        ].groupby('Indicator').agg(
            {'Improvement_%': ['mean', 'std', 'count'], 'DM_Significant_5pct': 'sum'}
        ).round(2)
        indicator_summary.columns = ['Mean_Improvement_%', 'Std', 'N_Sectors', 'N_DM_Sig_5pct']
        indicator_summary = indicator_summary.sort_values('Mean_Improvement_%', ascending=False)
        indicator_summary.to_csv(
            'Indicator_Performance_Summary_Stacking_Transformer_TGARCH.csv'
        )

        deployment_config = best_per_sector[[
            'Sector', 'Indicator', 'Learned_Weight_DL',
            'Learned_Weight_GARCH', 'Indicator_Beta',
            'Improvement_%', 'DM_Stat', 'DM_PValue', 'DM_Significant_5pct'
        ]].copy().rename(columns={
            'Indicator': 'Recommended_Indicator', 'Improvement_%': 'Improvement_vs_Baseline_%'
        })
        deployment_config['Base_Weight_Sum']              = (
            deployment_config['Learned_Weight_DL'] + deployment_config['Learned_Weight_GARCH']
        )
        deployment_config['Indicator_Adjustment_Lower_Bound'] = INDICATOR_ADJUSTMENT_LOW
        deployment_config['Indicator_Adjustment_Upper_Bound'] = INDICATOR_ADJUSTMENT_HIGH
        deployment_config['Weight'] = deployment_config['Sector'].map(sector_weights)
        deployment_config.to_csv(
            'Enhanced_Stacking_Transformer_TGARCH_Deployment_Configuration.csv', index=False
        )

        dm_loss_rows = []
        for rec in dm_test_records:
            for d, bl, el, diff in zip(
                rec['Dates'], rec['Base_Losses'], rec['Enh_Losses'], rec['Loss_Diffs']
            ):
                dm_loss_rows.append({
                    'Sector': rec['Sector'], 'Indicator': rec['Indicator'],
                    'Date': d, 'Base_QLIKE': bl, 'Enh_QLIKE': el,
                    'Loss_Diff_dt': diff, 'Code_Version': CODE_VERSION, 'Reproducibility_Version': REPRODUCIBILITY_VERSION, 'Execution_Device': GPU_NAME
                })
        pd.DataFrame(dm_loss_rows).to_csv(
            'DateLevel_QLIKE_LossDiffs_For_SPA_MCS.csv', index=False
        )
        print("✅ Date-level QLIKE loss diffs saved")

        n_sig5  = best_per_sector['DM_Significant_5pct'].sum()
        n_sig10 = best_per_sector['DM_Significant_10pct'].sum()
        n_total = len(best_per_sector)
        valid_sectors = list(best_per_sector['Sector'])
        all_sectors   = sorted(baseline_results['Sector'].unique())

        def _fmt_metric(value, width=12, decimals=6):
            if pd.isna(value): return f"{'N/A':>{width}}"
            value = float(value)
            if abs(value) >= 1000 or (0 < abs(value) < 1e-5):
                return f"{value:>{width}.4e}"
            return f"{value:>{width}.{decimals}f}"

        print(f"\n{'='*136}")
        print("BEST ENHANCED STACKING PER SECTOR")
        print("Selection: CV QLIKE + baseline-improvement gate | identical test dates")
        print(f"{'='*136}")
        print(f"{'Sector':<38}{'Indicator':<22}{'CV QLIKE':>12}{'Enhanced Q':>12}"
              f"{'Baseline Q':>12}{'Improve %':>11}{'DM p':>9}{'Sig':>6}"
              f"{'DL':>7}{'GARCH':>8}{'Beta':>7}")
        print("-"*136)
        for _, row in best_per_sector.sort_values('Improvement_%', ascending=False).iterrows():
            ind_label = row['Indicator'].replace('Sector_', '').replace('_Lag1', '')
            sig  = "**" if row['DM_Significant_5pct'] else ("*" if row['DM_Significant_10pct'] else "ns")
            cv_q = _fmt_metric(row['CV_QLIKE'], 12, 5)
            dm_p = f"{row['DM_PValue']:>9.4f}" if not pd.isna(row['DM_PValue']) else f"{'N/A':>9}"
            print(f"{row['Sector']:<38}{ind_label:<22}{cv_q}"
                  f"{_fmt_metric(row['Enhanced_Hybrid_QLIKE'], 12, 6)}"
                  f"{_fmt_metric(row['Baseline_QLIKE_Same_Dates'], 12, 6)}"
                  f"{row['Improvement_%']:>+10.2f}%{dm_p}{sig:>6}"
                  f"{row['Learned_Weight_DL']:>7.2f}{row['Learned_Weight_GARCH']:>8.2f}"
                  f"{row['Indicator_Beta']:>7.2f}")
        print(f"\n** p<0.05  * p<0.10  ns = not significant (DM, HLN)")

        print(f"\n{'='*70}")
        print("SELECTED ENHANCED SECTORS ONLY — IDENTICAL MATCHED DATES")
        print(f"{'='*70}")
        print(f"   Baseline QLIKE (same dates):  {weighted_baseline_same_dates:.6f}")
        print(f"   Enhanced QLIKE (same dates):  {weighted_enhanced_qlike:.6f}")
        print(f"   Improvement:                  {overall_improvement:+.2f}%")
        print(f"   Baseline RMSE (same dates):   {weighted_baseline_rmse_same:.6f}")
        print(f"   Enhanced RMSE (same dates):   {weighted_enhanced_rmse:.6f}")
        print(f"   Baseline MAE  (same dates):   {weighted_baseline_mae_same:.6f}")
        print(f"   Enhanced MAE  (same dates):   {weighted_enhanced_mae:.6f}")
        print(f"   DM p<0.05:                    {n_sig5}/{n_total} sectors")
        print(f"   DM p<0.10:                    {n_sig10}/{n_total} sectors")
        if excluded_sectors:
            print(f"   Excluded sectors:             {excluded_sectors}")
        print(f"\n   Note: Original baseline QLIKE={weighted_baseline_qlike:.6f} "
              f"(full period — not used in comparison)")

        print(f"\n{'='*70}")
        print("FULL-MARKET FINAL STRATEGY — ALL SECTORS")
        print("Selected sectors enhanced; all other sectors retain baseline")
        print(f"{'='*70}")
        print(f"   Comparator baseline QLIKE:    {market_wide_baseline_qlike:.6f}")
        print(f"   Final strategy QLIKE:          {market_wide_strategy_qlike:.6f}")
        print(f"   Improvement:                   {market_wide_improvement:+.2f}%")
        print(f"   Comparator baseline RMSE:     {market_wide_baseline_rmse:.6f}")
        print(f"   Final strategy RMSE:           {market_wide_strategy_rmse:.6f}")
        print(f"   Comparator baseline MAE:      {market_wide_baseline_mae:.6f}")
        print(f"   Final strategy MAE:            {market_wide_strategy_mae:.6f}")
        print(f"   Enhanced sectors:              {len(best_per_sector)}/{len(market_wide_strategy)}")
        print(f"   Normalised weight sum:         "
              f"{market_wide_strategy['Normalised_Weight'].sum():.10f}")

        print(f"\n{'='*70}")
        print("CONSERVATIVE DEPLOYMENT ROBUSTNESS CHECK")
        print("Enhanced retained only if it wins all chronological CV folds")
        print(f"{'='*70}")
        print(f"   Comparator baseline QLIKE:     {conservative_baseline_qlike:.6f}")
        print(f"   Conservative strategy QLIKE:   {conservative_final_qlike:.6f}")
        print(f"   Improvement:                   {conservative_improvement:+.2f}%")
        print(f"   Enhanced sectors retained:     "
              f"{(conservative_strategy['Conservative_Decision'] == 'Enhanced retained').sum()}/"
              f"{len(conservative_strategy)}")

        print("\n📊 INDICATOR PERFORMANCE (non-fallback):")
        print(indicator_summary.to_string())

# ================================================================
# 📊 Detailed Per-Sector Breakdown
# ================================================================
if enhanced_hybrid_results:
    print("\n" + "="*90)
    print("📊 DETAILED PER-SECTOR — sorted by CV QLIKE")
    print("="*90)
    for sector in sorted(baseline_results['Sector'].unique()):
        sec_df = enhanced_results_df[
            enhanced_results_df['Sector'] == sector
        ].sort_values('CV_QLIKE', na_position='last')
        if len(sec_df) == 0:
            continue
        print(f"\n{'='*90}")
        print(f"🔍 {sector}")
        print(f"{'='*90}")
        print(f"   {'Indicator':<28}{'FB':>4}{'CV QLIKE':>12}{'Enhanced Q':>12}"
              f"{'Baseline Q':>12}{'Improve %':>11}{'DM p':>9}{'DL':>7}{'GARCH':>8}{'Beta':>7}")
        print(f"   {'-'*110}")
        for _, row in sec_df.iterrows():
            ind_label = row['Indicator'].replace('Sector_', '').replace('_Lag1', '')
            cv_q = _fmt_metric(row['CV_QLIKE'], 12, 5)
            dm_p = f"{row['DM_PValue']:>9.4f}" if not pd.isna(row['DM_PValue']) else f"{'N/A':>9}"
            fb   = "FB" if row['Fallback_Used'] else ""
            sig  = "**" if row['DM_Significant_5pct'] else ("*" if row['DM_Significant_10pct'] else "")
            print(f"   {ind_label:<28}{fb:>4}{cv_q}"
                  f"{_fmt_metric(row['Enhanced_Hybrid_QLIKE'], 12, 6)}"
                  f"{_fmt_metric(row['Baseline_QLIKE_Same_Dates'], 12, 6)}"
                  f"{row['Improvement_%']:>+10.2f}%{dm_p}{sig:>3}"
                  f"{row['Learned_Weight_DL']:>7.2f}{row['Learned_Weight_GARCH']:>8.2f}"
                  f"{row['Indicator_Beta']:>7.2f}")

# ================================================================
# 📊 Visualizations
# ================================================================
if enhanced_hybrid_results and best_rows:
    all_results_combined = []
    for _, row in best_per_sector.iterrows():
        all_results_combined.append({
            'Sector': row['Sector'], 'Model': 'Baseline_Stacking',
            'QLIKE': row['Baseline_QLIKE_Same_Dates'],
            'RMSE':  row['Baseline_RMSE_Same_Dates'],
            'MAE':   row['Baseline_MAE_Same_Dates'],
            'Weight': row['Weight']
        })
        all_results_combined.append({
            'Sector': row['Sector'], 'Model': 'Enhanced_Stacking',
            'Indicator': row['Indicator'],
            'QLIKE': row['Enhanced_Hybrid_QLIKE'],
            'RMSE':  row['Enhanced_Hybrid_RMSE'],
            'MAE':   row['Enhanced_Hybrid_MAE'],
            'Weight': row['Weight']
        })
    combined_df = pd.DataFrame(all_results_combined)
    combined_df.to_csv(
        'Combined_Baseline_Enhanced_Stacking_Transformer_TGARCH_Results.csv', index=False
    )

    COLORS = {
        'baseline':  '#1E88E5', 'enhanced': '#D32F2F',
        'excellent': '#52B788', 'good':     '#95D5B2',
        'positive':  '#74C69D', 'neutral':  '#D9D9D9',
        'negative':  '#E5989B'
    }

    plot_df         = best_per_sector[[
        'Sector', 'Baseline_QLIKE_Same_Dates', 'Enhanced_Hybrid_QLIKE',
        'Improvement_%', 'DM_PValue', 'DM_Significant_5pct', 'DM_Significant_10pct'
    ]].copy()
    all_sectors     = plot_df['Sector'].tolist()
    baseline_qlikes = plot_df['Baseline_QLIKE_Same_Dates'].tolist()
    enhanced_qlikes = plot_df['Enhanced_Hybrid_QLIKE'].tolist()
    improvements    = plot_df['Improvement_%'].tolist()
    valid_sectors   = all_sectors

    # Visualization 1: Performance Comparison
    fig, ax = plt.subplots(figsize=(16, 9))
    x = np.arange(len(all_sectors))
    width = 0.38
    ax.bar(x - width/2, baseline_qlikes, width,
           label='Baseline Stacking (matched dates)',
           color=COLORS['baseline'], edgecolor='#2c3e50', linewidth=1.2)
    ax.bar(x + width/2, enhanced_qlikes, width,
           label='Enhanced Stacking (CV-selected, non-fallback)',
           color=[
               COLORS['excellent'] if imp > 5 else COLORS['good'] if imp > 1 else
               COLORS['positive']  if imp > 0 else COLORS['neutral']
               for imp in improvements
           ], edgecolor='#2c3e50', linewidth=1.2)
    for i, (b, e, imp) in enumerate(zip(baseline_qlikes, enhanced_qlikes, improvements)):
        if abs(imp) > 0.1:
            ax.text(i, max(b, e) + max(baseline_qlikes) * 0.02,
                    f'{imp:+.1f}%', ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color='#2c3e50')
    ax.set_xlabel('Sector', fontsize=13, fontweight='bold', color='#2c3e50')
    ax.set_ylabel('QLIKE — Identical Matched Dates (lower is better)',
                  fontsize=13, fontweight='bold', color='#2c3e50')
    ax.set_title(
        'Baseline vs Enhanced Stacking: Transformer + TGARCH\n'
        '(Baseline_QLIKE_Same_Dates vs Enhanced_QLIKE — identical observations)',
        fontsize=13, fontweight='bold', pad=20, color='#2c3e50')
    ax.set_xticks(x)
    ax.set_xticklabels(all_sectors, rotation=45, ha='right', fontsize=11, color='#2c3e50')
    ax.legend(loc='upper right', fontsize=12, framealpha=0.98)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#fafafa')
    plt.tight_layout()
    plt.savefig('Baseline_vs_Enhanced_Stacking_Transformer_TGARCH.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    display(Image('Baseline_vs_Enhanced_Stacking_Transformer_TGARCH.png'))
    plt.close()

    # Visualization 2: Improvement + DM significance
    fig, ax = plt.subplots(figsize=(17, 9))
    sorted_data  = sorted(
        zip(all_sectors, improvements,
            plot_df['DM_Significant_5pct'].tolist(),
            plot_df['DM_Significant_10pct'].tolist()),
        key=lambda x: x[1], reverse=True
    )
    s_sorted     = [x[0] for x in sorted_data]
    i_sorted     = [x[1] for x in sorted_data]
    sig5_sorted  = [x[2] for x in sorted_data]
    sig10_sorted = [x[3] for x in sorted_data]
    x_pos        = np.arange(len(s_sorted))
    c_sorted     = [
        '#2E8B57' if imp > 10 else '#66C28A' if imp > 5 else
        '#A8DDB5' if imp > 0  else '#D9D9D9' if imp == 0 else '#D97B7B'
        for imp in i_sorted
    ]
    ax.bar(x_pos, i_sorted, color=c_sorted, edgecolor='#2c3e50', linewidth=1.2)
    for i, (imp, s5, s10) in enumerate(zip(i_sorted, sig5_sorted, sig10_sorted)):
        va = 'bottom' if imp >= 0 else 'top'
        ax.text(i, imp + (0.35 if imp >= 0 else -0.35), f'{imp:+.1f}%',
                ha='center', va=va, fontsize=10, fontweight='bold', color='#2c3e50')
        sig_mark = "**" if s5 else ("*" if s10 else "")
        if sig_mark:
            ax.text(i, imp + (0.9 if imp >= 0 else -0.9), sig_mark,
                    ha='center', va='bottom', fontsize=12, color='darkred', fontweight='bold')
    ax.axhline(y=0, color='#2c3e50', linestyle='-', linewidth=1.5)
    ax.set_xlabel('Sector', fontsize=13, fontweight='bold', color='#2c3e50')
    ax.set_ylabel('QLIKE Improvement (%) — Identical Matched Dates',
                  fontsize=13, fontweight='bold', color='#2c3e50')
    ax.set_title(
        'QLIKE Improvement: Baseline → Enhanced Stacking\n'
        '(** p<0.05, * p<0.10 DM-HLN; Baseline_QLIKE_Same_Dates used)',
        fontsize=15, fontweight='bold', pad=18, color='#2c3e50')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(s_sorted, rotation=45, ha='right', fontsize=10, color='#2c3e50')
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#fafafa')
    plt.tight_layout()
    plt.savefig('Enhanced_Stacking_Transformer_TGARCH_Improvement.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    display(Image('Enhanced_Stacking_Transformer_TGARCH_Improvement.png'))
    plt.close()

    # Visualization 3: Sector Ranking
    fig, ax = plt.subplots(figsize=(14, 11))
    ranking_df = pd.DataFrame({
        'Sector': all_sectors, 'Baseline_QLIKE': baseline_qlikes,
        'Enhanced_QLIKE': enhanced_qlikes, 'Improvement_%': improvements
    }).sort_values('Baseline_QLIKE')
    y_pos = np.arange(len(ranking_df))
    ax.barh(y_pos - 0.2, ranking_df['Baseline_QLIKE'], 0.4,
            label='Baseline (matched dates)',
            color=COLORS['baseline'], edgecolor='#2c3e50', linewidth=1.1)
    ax.barh(y_pos + 0.2, ranking_df['Enhanced_QLIKE'], 0.4,
            label='Enhanced', color=[
                COLORS['excellent'] if imp > 5 else COLORS['good'] if imp > 1 else
                COLORS['positive']  if imp > 0 else COLORS['neutral']
                for imp in ranking_df['Improvement_%']
            ], edgecolor='#2c3e50', linewidth=1.1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ranking_df['Sector'], fontsize=10, color='#2c3e50')
    ax.set_xlabel('QLIKE — Identical Matched Dates (lower is better)',
                  fontsize=12, fontweight='bold', color='#2c3e50')
    ax.set_title('Sector Ranking: Stacking Models — Identical-Date QLIKE',
                 fontsize=15, fontweight='bold', pad=20, color='#2c3e50')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#fafafa')
    plt.tight_layout()
    plt.savefig('Sector_Ranking_Stacking_Transformer_TGARCH.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    display(Image('Sector_Ranking_Stacking_Transformer_TGARCH.png'))
    plt.close()

    # Visualization 4: Distribution
    valid_base = best_per_sector['Baseline_QLIKE_Same_Dates'].tolist()
    valid_enh  = best_per_sector['Enhanced_Hybrid_QLIKE'].tolist()
    fig, ax    = plt.subplots(figsize=(12, 8))
    bp = ax.boxplot(
        [valid_base, valid_enh],
        labels=['Baseline Stacking\n(matched dates)', 'Enhanced Stacking\n(non-fallback)'],
        patch_artist=True, showmeans=True,
        boxprops=dict(edgecolor='#2c3e50', linewidth=1.6),
        whiskerprops=dict(color='#2c3e50', linewidth=1.6),
        capprops=dict(color='#2c3e50', linewidth=1.6),
        medianprops=dict(color='#000000', linewidth=2.5),
        meanprops=dict(marker='D', markerfacecolor='#F4A261',
                       markeredgecolor='#2c3e50', markersize=8)
    )
    bp['boxes'][0].set_facecolor(COLORS['baseline']); bp['boxes'][0].set_alpha(0.85)
    bp['boxes'][1].set_facecolor(COLORS['enhanced']); bp['boxes'][1].set_alpha(0.85)
    bmed    = np.median(valid_base)
    emed    = np.median(valid_enh)
    med_imp = (bmed - emed) / bmed * 100 if bmed != 0 else 0.0
    ax.set_ylabel('QLIKE — Identical Matched Dates (lower is better)',
                  fontsize=12, fontweight='bold', color='#2c3e50')
    ax.set_title('QLIKE Distribution: Both from Baseline_QLIKE_Same_Dates',
                 fontsize=14, fontweight='bold', color='#2c3e50')
    ax.tick_params(axis='both', colors='#2c3e50', labelsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#fafafa')
    ax.text(0.02, 0.98, f'Median Improvement: {med_imp:+.2f}%',
            transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', color='#2c3e50',
            bbox=dict(boxstyle='round', facecolor='white',
                      alpha=0.95, edgecolor='#2c3e50', linewidth=1.5))
    plt.tight_layout()
    plt.savefig('Distribution_Stacking_Transformer_TGARCH.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    display(Image('Distribution_Stacking_Transformer_TGARCH.png'))
    plt.close()

    # Visualization 5: Dashboard
    pos_count = sum(1 for i in improvements if i > 0)
    neg_count = sum(1 for i in improvements if i < 0)
    neu_count = len(improvements) - pos_count - neg_count

    fig = plt.figure(figsize=(20, 12))
    fig.patch.set_facecolor('white')
    gs  = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3,
                   left=0.08, right=0.95, top=0.92, bottom=0.08)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.axis('off')
    for i, (title, value, color) in enumerate([
        ('Overall Improvement', f'{overall_improvement:+.2f}%',
         COLORS['excellent'] if overall_improvement > 0 else COLORS['neutral']),
        ('Baseline QLIKE\n(matched dates)', f'{weighted_baseline_same_dates:.4f}', COLORS['baseline']),
        ('Enhanced QLIKE\n(matched dates)', f'{weighted_enhanced_qlike:.4f}',       COLORS['enhanced']),
        ('Sectors Improved', f'{pos_count}/{len(valid_sectors)}',                   COLORS['excellent'])
    ]):
        xp = 0.05 + i * 0.25
        ax1.add_patch(plt.Rectangle(
            (xp, 0.2), 0.22, 0.6, transform=ax1.transAxes,
            facecolor=color, alpha=0.15, edgecolor=color, linewidth=2
        ))
        ax1.text(xp + 0.11, 0.65, title, transform=ax1.transAxes,
                 ha='center', va='center', fontsize=12, fontweight='bold', color='#2c3e50')
        ax1.text(xp + 0.11, 0.38, value, transform=ax1.transAxes,
                 ha='center', va='center', fontsize=20, fontweight='bold', color='#2c3e50')

    ax2 = fig.add_subplot(gs[1, 0])
    wedges, texts, autotexts = ax2.pie(
        [pos_count, neu_count, neg_count],
        labels=[f'Improved\n{pos_count}', f'Unchanged\n{neu_count}', f'Degraded\n{neg_count}'],
        colors=[COLORS['excellent'], COLORS['neutral'], COLORS['negative']],
        autopct='%1.0f%%', startangle=90, explode=(0.04, 0, 0),
        textprops=dict(fontsize=10, fontweight='bold', color='#2c3e50'),
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    for at in autotexts:
        at.set_color('white'); at.set_fontsize(12); at.set_fontweight('bold')
    ax2.set_title('Sector Outcomes (valid enhanced sectors)',
                  fontsize=14, fontweight='bold', color='#2c3e50', pad=20)

    ax3 = fig.add_subplot(gs[1, 1])
    top_n   = min(5, len(all_sectors))
    top_idx = np.argsort(improvements)[-top_n:][::-1]
    ax3.barh(np.arange(top_n), [improvements[i] for i in top_idx],
             color=COLORS['excellent'], edgecolor='#2c3e50', linewidth=1.5)
    ax3.set_yticks(np.arange(top_n))
    ax3.set_yticklabels([all_sectors[i] for i in top_idx],
                        fontsize=11, fontweight='bold', color='#2c3e50')
    ax3.set_xlabel('QLIKE Improvement (%)', fontsize=13, fontweight='bold', color='#2c3e50')
    ax3.set_title('Top 5 Sectors', fontsize=15, fontweight='bold', color='#2c3e50', pad=20)
    ax3.grid(axis='x', alpha=0.2, linestyle='--')
    ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
    ax3.set_facecolor('#fafafa')

    ax4 = fig.add_subplot(gs[2, :])
    if len(baseline_forecasts) > 0:
        ax4.plot(baseline_forecasts.groupby('Date')['Actual_Variance'].mean(),
                 label='Observed One-Day Squared-Return Variance Proxy',
                 linewidth=2.5, color='#000000', zorder=3)
        ax4.plot(baseline_forecasts.groupby('Date')['Baseline_Hybrid_Variance'].mean(),
                 label='Baseline Stacking', linewidth=2.5,
                 color=COLORS['baseline'], linestyle='--', zorder=2)
        if 'best_enhanced_forecasts' in locals():
            ax4.plot(
                best_enhanced_forecasts.groupby('Date')['Enhanced_Hybrid_Variance'].mean(),
                label='Enhanced Stacking', linewidth=2.5,
                color=COLORS['enhanced'], linestyle='-.', zorder=2)
    ax4.set_title('Average Market-Level Variance Paths',
                  fontsize=15, fontweight='bold', color='#2c3e50', pad=20)
    ax4.set_ylabel('One-Day Squared-Return Variance Proxy',
                   fontsize=12, fontweight='bold', color='#2c3e50')
    ax4.legend(fontsize=11, loc='best')
    ax4.grid(axis='y', alpha=0.2, linestyle='--')
    ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)
    ax4.set_facecolor('#fafafa')

    fig.suptitle(
        'STACKING TRANSFORMER-TGARCH HYBRID DASHBOARD\n'
        '(Matched-date metrics | CV-selected non-fallback | DM-HLN test)',
        fontsize=16, fontweight='bold', color='#2c3e50', y=0.97)
    plt.savefig('Stacking_Transformer_TGARCH_Dashboard.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    display(Image('Stacking_Transformer_TGARCH_Dashboard.png'))
    plt.close()

# ================================================================
# 💾 Diagnostics + Excel
# ================================================================
pd.DataFrame(training_diagnostics).to_csv(
    'Training_Diagnostics_Stacking_Transformer_TGARCH.csv', index=False
)

if enhanced_hybrid_results and best_rows:
    writer = pd.ExcelWriter(
        'Complete_Stacking_Transformer_TGARCH_Analysis.xlsx', engine='openpyxl'
    )
    summary_data = pd.DataFrame({
        'Metric': [
            'Code Version',
            'Weighted Baseline QLIKE — same matched dates',
            'Weighted Enhanced QLIKE — same matched dates',
            'Overall Improvement (%) — matched dates',
            'Weighted Baseline RMSE — same matched dates',
            'Weighted Enhanced RMSE — same matched dates',
            'Weighted Baseline MAE  — same matched dates',
            'Weighted Enhanced MAE  — same matched dates',
            'Original Baseline QLIKE (full period, reference only)',
            'Total Sectors', 'Valid Enhanced Sectors', 'Excluded Sectors',
            'Sectors Improved', 'Sectors Degraded', 'Sectors DM significant p<0.05',
            'Indicator Selection Criterion',
            'Calibration Period', 'Constrained CV Period', 'Final Test Period',
            'Fallback Trigger', 'Metric Alignment', 'Indicators Tested',
            'Transformer Training Strategy'
        ],
        'Value': [
            CODE_VERSION,
            weighted_baseline_same_dates, weighted_enhanced_qlike, overall_improvement,
            weighted_baseline_rmse_same,  weighted_enhanced_rmse,
            weighted_baseline_mae_same,   weighted_enhanced_mae,
            weighted_baseline_qlike,
            len(all_sectors), len(valid_sectors), len(excluded_sectors),
            pos_count, neg_count, n_sig5,
            'Time-series CV QLIKE using constrained convex base and bounded indicator adjustment (non-fallback only)',
            f'First {int(CAL_RATIO*100)}% of aligned observations',
            f'{int(CAL_RATIO*100)}% to {int(STACK_TRAIN_END*100)}%',
            f'{int(STACK_TRAIN_END*100)}% onward',
            'Numerical failure or invalid constrained prediction; corrected baseline retained',
            'Baseline QLIKE/RMSE/MAE from comp_test — identical dates',
            'EMA_10, EMA_14, EMA_20, SMA_10, SMA_20, SMA_50, BB_High, BB_Low, BB_Middle, BB_Width, BB_Pct, RSI_14, MACD, MACD_Signal, MACD_Diff, ATR_14 (all lagged by 1 day)',
            'Trained once per sector-indicator on the pre-test sample'
        ]
    })
    summary_data = pd.concat([
        summary_data,
        pd.DataFrame({
            'Metric': [
                'Common Actual Target SHA-256',
                'Full-Market Comparator Baseline QLIKE',
                'Full-Market Final Strategy QLIKE',
                'Full-Market Improvement (%)',
                'Full-Market Comparator Baseline RMSE',
                'Full-Market Final Strategy RMSE',
                'Full-Market Comparator Baseline MAE',
                'Full-Market Final Strategy MAE',
                'Conservative Comparator Baseline QLIKE',
                'Conservative Final Strategy QLIKE',
                'Conservative Improvement (%)',
                'Conservative Enhanced Sectors'
            ],
            'Value': [
                TARGET_SHA256,
                market_wide_baseline_qlike, market_wide_strategy_qlike, market_wide_improvement,
                market_wide_baseline_rmse,  market_wide_strategy_rmse,
                market_wide_baseline_mae,   market_wide_strategy_mae,
                conservative_baseline_qlike, conservative_final_qlike,
                conservative_improvement,
                int((conservative_strategy['Conservative_Decision'] == 'Enhanced retained').sum())
            ]
        })
    ], ignore_index=True)

    summary_data.to_excel(writer, sheet_name='Summary',              index=False)
    baseline_results.to_excel(writer, sheet_name='Baseline',         index=False)
    baseline_forecasts.to_excel(writer, sheet_name='Base_Forecasts', index=False)
    enhanced_results_df.to_excel(writer, sheet_name='Enhanced_All',  index=False)
    best_per_sector.to_excel(writer, sheet_name='Best_Per_Sector',   index=False)
    indicator_summary.to_excel(writer, sheet_name='Indicator_Summary')
    deployment_config.to_excel(writer, sheet_name='Deployment_Config',      index=False)
    market_wide_strategy.to_excel(writer, sheet_name='Market_Wide_Strategy', index=False)
    conservative_strategy.to_excel(writer, sheet_name='Conservative_Check', index=False)
    combined_df.to_excel(writer, sheet_name='Combined_Results',      index=False)
    pd.DataFrame(training_diagnostics).to_excel(
        writer, sheet_name='Training_Diagnostics', index=False
    )
    if 'best_enhanced_forecasts' in locals():
        best_enhanced_forecasts.to_excel(writer, sheet_name='Enhanced_Forecasts', index=False)
    if 'comp_test_only' in locals():
        comp_test_only.to_excel(writer, sheet_name='CompTest_Final_Test_Matched', index=False)
    writer.close()
    print("✅ Excel workbook saved")

# ================================================================
# 📥 Download All Files
# ================================================================
import time
from google.colab import files as colab_files

csv_files = [
    'Enhanced_Stacking_Transformer_TGARCH_Results.csv',
    'Best_Enhanced_Stacking_Transformer_TGARCH_Per_Sector.csv',
    'Enhanced_Stacking_Transformer_TGARCH_Forecasts.csv',
    'Enhanced_Stacking_CompTest_Final_Test_Matched.csv',
    'Sector_Summary_Metrics_Enhanced_Stacking_Transformer_TGARCH.csv',
    'Indicator_Performance_Summary_Stacking_Transformer_TGARCH.csv',
    'Enhanced_Stacking_Transformer_TGARCH_Deployment_Configuration.csv',
    'Combined_Baseline_Enhanced_Stacking_Transformer_TGARCH_Results.csv',
    'Training_Diagnostics_Stacking_Transformer_TGARCH.csv',
    'DateLevel_QLIKE_LossDiffs_For_SPA_MCS.csv',
    'Market_Wide_Final_Strategy_Metrics.csv',
    'Conservative_Deployment_Robustness_Check.csv',
    'Common_Actual_Variance_1Day_Squared_Return.csv'
]
png_files = [
    'Baseline_vs_Enhanced_Stacking_Transformer_TGARCH.png',
    'Enhanced_Stacking_Transformer_TGARCH_Improvement.png',
    'Sector_Ranking_Stacking_Transformer_TGARCH.png',
    'Distribution_Stacking_Transformer_TGARCH.png',
    'Stacking_Transformer_TGARCH_Dashboard.png'
]
all_files = csv_files + png_files + [
    'Complete_Stacking_Transformer_TGARCH_Analysis.xlsx',
    'Common_Actual_Variance_1Day_Squared_Return_SHA256.txt'
]

FINAL_BUNDLE  = 'Enhanced_Stacking_ALL_RESULTS.zip'
bundled_count = 0
with zipfile.ZipFile(FINAL_BUNDLE, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for f in all_files:
        if os.path.exists(f):
            bundle.write(f, arcname=f)
            bundled_count += 1
            print(f"   ✅ {f}")
        else:
            print(f"   ❌ NOT FOUND: {f}")

print(f"\n✅ Bundle: {bundled_count}/{len(all_files)} → {FINAL_BUNDLE}")

try:
    colab_files.download(FINAL_BUNDLE)
except Exception as e:
    print(f"⚠️ Bundle download failed: {e}")

for f in all_files:
    if os.path.exists(f):
        try:
            colab_files.download(f)
        except Exception as e:
            print(f"   ⚠️ {f} — {e}")
        time.sleep(1.5)

log_memory("Final")

print("\n" + "="*90)
print(f"✅ COMPLETE: ENHANCED STACKING  |  {CODE_VERSION}")
print("="*90)
print("\n🎯 All corrections applied:")
print("   ✅ Issue A1: Three-period split: calibration [:25%] → constrained CV [25%-65%] → test [65%:]")
print("   ✅ Issue A2: Overall weighted baseline uses matched dates")
print("   ✅ Issue A3: Baseline RMSE and MAE from comp_test")
print("   ✅ Issue B4: Explicit hybrid column detection")
print("   ✅ Issue B5: Visualizations use Baseline_QLIKE_Same_Dates")
print("   ✅ Issue B6: Code version guard; DM checkpoint; comp_test exported")
print("   ✅ One-day squared-return target created before volume/feature filtering")
print("   ✅ Common-target SHA-256 audit files exported")
print("   ✅ Target horizon aligned with one-step-ahead TGARCH-X forecasts")
print("   ✅ Full-market strategy includes baseline-retained sectors")
print("   ✅ Baseline meta-learner: constrained convex Transformer-TGARCH weights")
print("   ✅ Enhanced meta-layer: bounded multiplicative indicator adjustment")
print("   ✅ Efficiency: Transformer trained once per sector-indicator")
print("   ✅ Memory:    Aggressive cleanup after every indicator")
print("   ✅ REPRODUCIBILITY: deterministic T4 GPU + float32 + fixed per-build seed")
print("="*90)

✅ Deterministic T4-GPU environment variables set
🔄 LOADING PRE-COMPUTED BASELINE RESULTS
⚠️ Missing baseline files: ['Stacking_Transformer_TGARCH_Sector_Metrics.csv', 'Stacking_Transformer_TGARCH_Forecasts.csv']
📥 Upload ONLY the two baseline output CSVs now:
   - Stacking_Transformer_TGARCH_Sector_Metrics.csv
   - Stacking_Transformer_TGARCH_Forecasts.csv


KeyboardInterrupt: 

In [ ]:
# ================================================================
# REVIEWER-FOCUSED FORECAST COMPARISON
# Research question:
# "Do Technical Indicators Improve Hybrid Volatility Forecasting?"
#
# Primary:
#   1) Final locked strategy vs baseline using QLIKE
#   2) Stratified moving-block-bootstrap confidence interval
#   3) Sector-level HLN-corrected DM tests
#   4) Aggregate DM over selected sectors using original weights
#
# Secondary:
#   4) Hansen SPA test across all 16 indicator candidates
#
# Optional appendix:
#   5) Canonical MCS using arch.bootstrap.MCS
#
# Required input:
#   DateLevel_QLIKE_LossDiffs_For_SPA_MCS.csv
#
# Expected in memory after the enhanced model cell:
#   best_per_sector
#   baseline_results
#
# If unavailable, the script attempts to load:
#   Best_Enhanced_Stacking_Transformer_TGARCH_Per_Sector.csv
#   Stacking_Transformer_TGARCH_Sector_Metrics.csv
# ================================================================

import os
import time
import warnings
import subprocess
import sys
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

SEED = 42
B_BOOT = 5000
BLOCK_LENGTH = 5
DM_H = 5
ALPHA = 0.05
RUN_MCS_APPENDIX = False

np.random.seed(SEED)

print("=" * 100)
print("REVIEWER-FOCUSED FORECAST COMPARISON")
print("Question: Do technical indicators improve the hybrid volatility forecast?")
print("=" * 100)

# ================================================================
# 1. LOAD INPUTS
# ================================================================
LOSS_FILE = "DateLevel_QLIKE_LossDiffs_For_SPA_MCS.csv"
BEST_FILE = "Best_Enhanced_Stacking_Transformer_TGARCH_Per_Sector.csv"
BASELINE_METRICS_FILE = "Stacking_Transformer_TGARCH_Sector_Metrics.csv"

if not os.path.exists(LOSS_FILE):
    raise FileNotFoundError(
        f"Missing required file: {LOSS_FILE}"
    )

loss_df = pd.read_csv(LOSS_FILE)
required_cols = {
    "Sector", "Indicator", "Date",
    "Base_QLIKE", "Enh_QLIKE", "Loss_Diff_dt"
}
missing_cols = required_cols - set(loss_df.columns)
if missing_cols:
    raise ValueError(
        f"{LOSS_FILE} is missing columns: {sorted(missing_cols)}"
    )

loss_df["Date"] = pd.to_datetime(
    loss_df["Date"], errors="coerce"
)
loss_df = loss_df.dropna(
    subset=["Sector", "Indicator", "Date",
            "Base_QLIKE", "Enh_QLIKE"]
).copy()

for col in ["Base_QLIKE", "Enh_QLIKE", "Loss_Diff_dt"]:
    loss_df[col] = pd.to_numeric(
        loss_df[col], errors="coerce"
    )

loss_df = loss_df.dropna(
    subset=["Base_QLIKE", "Enh_QLIKE", "Loss_Diff_dt"]
).copy()

if loss_df.duplicated(
    ["Sector", "Indicator", "Date"]
).any():
    raise ValueError(
        "Duplicate Sector-Indicator-Date rows found in loss file."
    )

# Load selected enhanced models.
if "best_per_sector" in globals():
    selected_df = best_per_sector.copy()
elif os.path.exists(BEST_FILE):
    selected_df = pd.read_csv(BEST_FILE)
else:
    selected_df = pd.DataFrame(
        columns=["Sector", "Indicator"]
    )

if len(selected_df) > 0:
    if not {"Sector", "Indicator"}.issubset(
        selected_df.columns
    ):
        raise ValueError(
            "Selected-model table must contain Sector and Indicator."
        )
    selected_df = selected_df[
        ["Sector", "Indicator"]
    ].drop_duplicates("Sector")

selected_map = (
    selected_df.set_index("Sector")["Indicator"].to_dict()
    if len(selected_df) > 0 else {}
)

# Load sector weights.
if "baseline_results" in globals():
    weights_df = baseline_results[
        ["Sector", "Weight"]
    ].copy()
elif os.path.exists(BASELINE_METRICS_FILE):
    weights_df = pd.read_csv(
        BASELINE_METRICS_FILE
    )[["Sector", "Weight"]].copy()
else:
    weights_df = pd.DataFrame(
        {"Sector": sorted(loss_df["Sector"].unique())}
    )
    weights_df["Weight"] = 1.0 / len(weights_df)

weights_df["Weight"] = pd.to_numeric(
    weights_df["Weight"], errors="coerce"
)
weights_df = weights_df.dropna(
    subset=["Sector", "Weight"]
).drop_duplicates("Sector")

all_sectors = sorted(loss_df["Sector"].unique())
weights_df = pd.DataFrame(
    {"Sector": all_sectors}
).merge(
    weights_df, on="Sector", how="left"
)

if weights_df["Weight"].isna().any():
    missing_weight_sectors = weights_df.loc[
        weights_df["Weight"].isna(), "Sector"
    ].tolist()
    raise ValueError(
        "Missing sector weights for: "
        f"{missing_weight_sectors}"
    )

weights_df["Weight"] = (
    weights_df["Weight"] /
    weights_df["Weight"].sum()
)
weight_map = weights_df.set_index(
    "Sector"
)["Weight"].to_dict()

print(f"Loaded {len(loss_df)} date-level loss observations")
print(f"Sectors: {len(all_sectors)}")
print(f"Indicator candidates: {loss_df['Indicator'].nunique()}")
print(f"Enhanced models selected by CV gate: {len(selected_map)}")

# ================================================================
# 2. HELPERS
# ================================================================
def hln_dm_test(loss_base, loss_strategy, h=5):
    """
    Two-sided HLN-corrected Diebold-Mariano test.

    d_t = baseline loss - strategy loss.
    Positive mean(d_t) means the strategy is better.

    For a 5-day overlapping variance target, h=5 includes
    Bartlett-weighted autocovariances through lag 4.
    """
    base = np.asarray(loss_base, dtype=float)
    strategy = np.asarray(loss_strategy, dtype=float)

    mask = np.isfinite(base) & np.isfinite(strategy)
    d = base[mask] - strategy[mask]
    n = len(d)

    if n < max(10, h + 2):
        mean_d = np.mean(d) if n > 0 else np.nan
        return np.nan, np.nan, mean_d, n

    d_bar = float(np.mean(d))
    gamma0 = float(np.var(d, ddof=1))
    long_run_var = gamma0

    for lag in range(1, h):
        if n <= lag:
            break
        gamma_lag = np.cov(
            d[lag:], d[:-lag], ddof=1
        )[0, 1]
        long_run_var += (
            2.0 * (1.0 - lag / h) * gamma_lag
        )

    if (
        not np.isfinite(long_run_var)
        or long_run_var <= 1e-16
    ):
        return np.nan, np.nan, d_bar, n

    dm_raw = d_bar / np.sqrt(long_run_var / n)

    correction = (
        n + 1 - 2*h + h*(h - 1)/n
    ) / n

    if correction <= 0:
        return np.nan, np.nan, d_bar, n

    dm_hln = dm_raw * np.sqrt(correction)
    p_two = 2.0 * (
        1.0 - stats.t.cdf(
            abs(dm_hln), df=n - 1
        )
    )

    return dm_hln, p_two, d_bar, n


def moving_block_indices(
    n, block_length, rng
):
    """
    Circular moving-block bootstrap indices.
    """
    if n <= 0:
        return np.array([], dtype=int)

    n_blocks = int(
        np.ceil(n / block_length)
    )
    starts = rng.integers(
        0, n, size=n_blocks
    )

    idx = []
    for start in starts:
        idx.extend(
            (
                start +
                np.arange(block_length)
            ) % n
        )

    return np.asarray(
        idx[:n], dtype=int
    )


def block_bootstrap_mean_ci(
    values,
    block_length=5,
    reps=5000,
    alpha=0.05,
    seed=42
):
    """
    Percentile moving-block-bootstrap CI for mean(values).
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)

    if n < 5:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = np.empty(
        reps, dtype=float
    )

    for b in range(reps):
        idx = moving_block_indices(
            n, block_length, rng
        )
        boot_means[b] = np.mean(x[idx])

    lower = np.quantile(
        boot_means, alpha / 2
    )
    upper = np.quantile(
        boot_means, 1 - alpha / 2
    )

    # One-sided bootstrap probability against no improvement.
    p_one_sided = np.mean(
        boot_means <= 0
    )

    return float(lower), float(upper), float(p_one_sided)


def holm_adjust(p_values):
    """
    Holm step-down family-wise-error correction.
    Missing p-values remain missing.
    """
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(
        len(p), np.nan, dtype=float
    )

    valid_idx = np.where(
        np.isfinite(p)
    )[0]

    if len(valid_idx) == 0:
        return adjusted

    valid_p = p[valid_idx]
    order = np.argsort(valid_p)
    ordered_p = valid_p[order]
    m = len(ordered_p)

    ordered_adj = np.empty(
        m, dtype=float
    )

    running_max = 0.0
    for i, pval in enumerate(ordered_p):
        candidate = min(
            (m - i) * pval, 1.0
        )
        running_max = max(
            running_max, candidate
        )
        ordered_adj[i] = running_max

    reverse_order = np.empty_like(order)
    reverse_order[order] = np.arange(m)

    adjusted[valid_idx] = ordered_adj[
        reverse_order
    ]

    return adjusted


def sig_label(p):
    if pd.isna(p):
        return "n/a"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return "ns"

# ================================================================
# 3. BUILD THE LOCKED FINAL STRATEGY
# ================================================================
# For sectors selected by the training-only CV gate:
#     use the selected enhanced model.
#
# For all other sectors:
#     use the baseline.
#
# This avoids selecting models on the final test.
strategy_rows = []
sector_rows = []

for sector in all_sectors:
    sec = loss_df[
        loss_df["Sector"] == sector
    ].copy()

    # Baseline losses must be identical across indicator rows
    # for each sector-date.
    base_check = sec.groupby("Date")[
        "Base_QLIKE"
    ].agg(["min", "max"])

    if (
        (
            base_check["max"] -
            base_check["min"]
        ).abs() > 1e-12
    ).any():
        raise ValueError(
            f"{sector}: baseline losses differ across indicator rows."
        )

    baseline_series = sec.groupby(
        "Date"
    )["Base_QLIKE"].first().sort_index()

    selected_indicator = selected_map.get(
        sector
    )

    if selected_indicator is None:
        strategy_series = baseline_series.copy()
        model_used = "Baseline retained"
    else:
        selected_loss = sec[
            sec["Indicator"] == selected_indicator
        ].set_index("Date")[
            "Enh_QLIKE"
        ].sort_index()

        aligned = pd.concat(
            [
                baseline_series.rename(
                    "Baseline_Loss"
                ),
                selected_loss.rename(
                    "Strategy_Loss"
                )
            ],
            axis=1,
            join="inner"
        ).dropna()

        if len(aligned) == 0:
            raise ValueError(
                f"{sector}: no matched dates for selected indicator "
                f"{selected_indicator}."
            )

        baseline_series = aligned[
            "Baseline_Loss"
        ]
        strategy_series = aligned[
            "Strategy_Loss"
        ]
        model_used = selected_indicator

    sector_strategy = pd.DataFrame({
        "Date": baseline_series.index,
        "Sector": sector,
        "Weight": weight_map[sector],
        "Baseline_Loss": baseline_series.values,
        "Strategy_Loss": strategy_series.values
    })

    sector_strategy[
        "Loss_Diff"
    ] = (
        sector_strategy["Baseline_Loss"] -
        sector_strategy["Strategy_Loss"]
    )

    strategy_rows.append(
        sector_strategy
    )

    dm_stat, dm_p, mean_diff, n_obs = hln_dm_test(
        sector_strategy["Baseline_Loss"],
        sector_strategy["Strategy_Loss"],
        h=DM_H
    )

    ci_low, ci_high, boot_p = block_bootstrap_mean_ci(
        sector_strategy["Loss_Diff"],
        block_length=BLOCK_LENGTH,
        reps=B_BOOT,
        alpha=ALPHA,
        seed=SEED
    )

    baseline_mean = float(
        sector_strategy["Baseline_Loss"].mean()
    )
    strategy_mean = float(
        sector_strategy["Strategy_Loss"].mean()
    )
    improvement_pct = (
        mean_diff / baseline_mean * 100
        if baseline_mean != 0 else np.nan
    )

    sector_rows.append({
        "Sector": sector,
        "Weight": weight_map[sector],
        "Model_Used": model_used,
        "Enhanced_Selected": bool(
            selected_indicator is not None
        ),
        "N_Test": n_obs,
        "Baseline_QLIKE": baseline_mean,
        "Final_Strategy_QLIKE": strategy_mean,
        "Mean_Loss_Diff": mean_diff,
        "Improvement_Pct": improvement_pct,
        "DM_Stat_HLN": dm_stat,
        "DM_PValue": dm_p,
        "Bootstrap_CI_Lower": ci_low,
        "Bootstrap_CI_Upper": ci_high,
        "Bootstrap_OneSided_P": boot_p
    })

strategy_long = pd.concat(
    strategy_rows, ignore_index=True
)
sector_results = pd.DataFrame(
    sector_rows
)

# Apply Holm correction only to sectors where an enhanced model
# was actually selected and compared with the baseline.
tested_mask = (
    sector_results["Enhanced_Selected"]
    & sector_results["DM_PValue"].notna()
)

sector_results["DM_PValue_Holm"] = np.nan

if tested_mask.any():
    sector_results.loc[
        tested_mask,
        "DM_PValue_Holm"
    ] = holm_adjust(
        sector_results.loc[
            tested_mask,
            "DM_PValue"
        ].to_numpy(dtype=float)
    )

sector_results[
    "Significant_After_Holm_5pct"
] = (
    sector_results["DM_PValue_Holm"].notna()
    & (sector_results["DM_PValue_Holm"] < 0.05)
)

# ================================================================
# 4. MARKET-WIDE FINAL STRATEGY COMPARISON
# ================================================================
# Sector test windows do not fully overlap in calendar time.
# Therefore, a calendar-date market-wide DM series cannot be formed
# without discarding most observations.
#
# Instead, estimate the market-wide effect as the fixed-weighted
# average of sector-specific mean QLIKE losses:
#
#   Delta = sum_s w_s * mean_t(BaseLoss_st - StrategyLoss_st)
#
# Uncertainty is obtained with a stratified moving-block bootstrap:
# each sector is resampled in blocks, and sector weights remain fixed.
# This preserves within-sector serial dependence and uses all sectors.

def stratified_sector_block_bootstrap(
    strategy_panel,
    sector_weights,
    block_length=5,
    reps=5000,
    alpha=0.05,
    seed=42
):
    """
    Fixed-weight stratified moving-block bootstrap.

    Returns the bootstrap distribution of:
      weighted baseline QLIKE,
      weighted strategy QLIKE,
      weighted mean loss difference.

    Each sector is resampled separately in moving blocks, preserving
    within-sector dependence. Sector weights remain fixed.
    """
    rng = np.random.default_rng(seed)

    sectors = sorted(
        strategy_panel["Sector"].unique()
    )

    sector_arrays = {}
    for sector in sectors:
        sub = strategy_panel[
            strategy_panel["Sector"] == sector
        ][
            ["Baseline_Loss", "Strategy_Loss"]
        ].dropna()

        if len(sub) < 5:
            raise ValueError(
                f"{sector}: fewer than 5 paired loss observations."
            )

        sector_arrays[sector] = sub.to_numpy(
            dtype=float
        )

    boot_base = np.empty(reps, dtype=float)
    boot_strategy = np.empty(reps, dtype=float)
    boot_diff = np.empty(reps, dtype=float)

    for b in range(reps):
        base_total = 0.0
        strategy_total = 0.0

        for sector in sectors:
            arr = sector_arrays[sector]
            n = len(arr)

            idx = moving_block_indices(
                n, block_length, rng
            )
            sample = arr[idx]

            weight = float(
                sector_weights[sector]
            )

            base_total += (
                weight * sample[:, 0].mean()
            )
            strategy_total += (
                weight * sample[:, 1].mean()
            )

        boot_base[b] = base_total
        boot_strategy[b] = strategy_total
        boot_diff[b] = (
            base_total - strategy_total
        )

    lower = float(
        np.quantile(
            boot_diff, alpha / 2
        )
    )
    upper = float(
        np.quantile(
            boot_diff, 1 - alpha / 2
        )
    )

    # One-sided p-value for H0: no improvement or worse,
    # against H1: positive weighted loss difference.
    p_one_sided = float(
        np.mean(boot_diff <= 0)
    )

    return {
        "boot_base": boot_base,
        "boot_strategy": boot_strategy,
        "boot_diff": boot_diff,
        "ci_lower": lower,
        "ci_upper": upper,
        "p_one_sided": p_one_sided
    }


# Fixed-weight averages of sector-specific test means.
sector_means = strategy_long.groupby(
    "Sector"
).agg(
    Baseline_QLIKE=("Baseline_Loss", "mean"),
    Strategy_QLIKE=("Strategy_Loss", "mean"),
    N_Test=("Date", "nunique")
).reset_index()

sector_means["Weight"] = sector_means[
    "Sector"
].map(weight_map)

market_base_mean = float(
    np.sum(
        sector_means["Weight"] *
        sector_means["Baseline_QLIKE"]
    )
)

market_strategy_mean = float(
    np.sum(
        sector_means["Weight"] *
        sector_means["Strategy_QLIKE"]
    )
)

market_mean_diff = (
    market_base_mean -
    market_strategy_mean
)

market_improvement_pct = (
    market_mean_diff /
    market_base_mean * 100
    if market_base_mean != 0
    else np.nan
)

market_boot = stratified_sector_block_bootstrap(
    strategy_long,
    weight_map,
    block_length=BLOCK_LENGTH,
    reps=B_BOOT,
    alpha=ALPHA,
    seed=SEED
)


# ================================================================
# 4B. AGGREGATE DM TEST FOR THE ACTUAL INCREMENTAL STRATEGY
# ================================================================
# Baseline-retained sectors have exactly zero loss differential.
# Therefore, only the selected enhanced sectors contribute to the
# strategy's incremental loss difference. We preserve their original
# market weights and do not renormalize them.
selected_sectors_for_dm = sector_results.loc[
    sector_results["Enhanced_Selected"],
    "Sector"
].tolist()

aggregate_dm_stat = np.nan
aggregate_dm_p = np.nan
aggregate_dm_mean_diff = np.nan
aggregate_dm_n = 0
aggregate_dm_note = (
    "Not run: fewer than 10 common dates across selected sectors."
)

if len(selected_sectors_for_dm) > 0:
    selected_diff_pivot = (
        strategy_long[
            strategy_long["Sector"].isin(
                selected_sectors_for_dm
            )
        ]
        .pivot(
            index="Date",
            columns="Sector",
            values="Loss_Diff"
        )
        .dropna()
        .sort_index()
    )

    if len(selected_diff_pivot) >= 10:
        selected_weights = np.array(
            [
                weight_map[s]
                for s in selected_diff_pivot.columns
            ],
            dtype=float
        )

        aggregate_diff = (
            selected_diff_pivot.to_numpy(dtype=float)
            @ selected_weights
        )

        aggregate_dm_stat, aggregate_dm_p, \
        aggregate_dm_mean_diff, aggregate_dm_n = hln_dm_test(
            np.zeros(len(aggregate_diff), dtype=float),
            -aggregate_diff,
            h=DM_H
        )

        aggregate_dm_note = (
            "Run on selected enhanced sectors only, using original "
            "market weights; baseline-retained sectors contribute zero."
        )

market_summary = pd.DataFrame([{
    "Aggregation_Method": (
        "Fixed-weight sector means with stratified "
        "moving-block bootstrap"
    ),
    "N_Sectors": len(all_sectors),
    "N_Enhanced_Selected": len(selected_map),
    "Min_Sector_Test_N": int(
        sector_means["N_Test"].min()
    ),
    "Max_Sector_Test_N": int(
        sector_means["N_Test"].max()
    ),
    "Baseline_Weighted_QLIKE": market_base_mean,
    "Final_Strategy_Weighted_QLIKE": market_strategy_mean,
    "Mean_Loss_Diff": market_mean_diff,
    "Improvement_Pct": market_improvement_pct,
    "Bootstrap_CI_Lower": market_boot["ci_lower"],
    "Bootstrap_CI_Upper": market_boot["ci_upper"],
    "Bootstrap_OneSided_P": market_boot["p_one_sided"],
    "DM_Stat_HLN": aggregate_dm_stat,
    "DM_PValue_TwoSided": aggregate_dm_p,
    "DM_Mean_Loss_Diff_Selected_Sectors": aggregate_dm_mean_diff,
    "DM_N_Common_Selected_Sectors": aggregate_dm_n,
    "DM_Note": aggregate_dm_note
}])

# ================================================================
# 5. CANONICAL HANSEN SPA — SECONDARY FAMILY-LEVEL TEST
# ================================================================
# SPA asks:
# "Does at least one of the 16 indicator models beat the baseline
# after accounting for data snooping?"
#
# We use the maintained implementation in arch.bootstrap.SPA.
try:
    from arch.bootstrap import SPA, MCS
except ImportError:
    print("\nInstalling 'arch' for canonical SPA/MCS implementation...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "arch",
            "--quiet"
        ]
    )
    from arch.bootstrap import SPA, MCS

spa_rows = []

for sector in all_sectors:
    sec = loss_df[
        loss_df["Sector"] == sector
    ].copy()

    indicators = sorted(
        sec["Indicator"].unique()
    )

    benchmark = sec[
        sec["Indicator"] == indicators[0]
    ][["Date", "Base_QLIKE"]].set_index(
        "Date"
    ).rename(
        columns={"Base_QLIKE": "Baseline"}
    )

    candidate_series = []
    for indicator in indicators:
        s = sec[
            sec["Indicator"] == indicator
        ][["Date", "Enh_QLIKE"]].set_index(
            "Date"
        ).rename(
            columns={"Enh_QLIKE": indicator}
        )
        candidate_series.append(s)

    candidates = pd.concat(
        candidate_series,
        axis=1,
        join="inner"
    )

    aligned = benchmark.join(
        candidates,
        how="inner"
    ).dropna()

    if len(aligned) < 10:
        spa_rows.append({
            "Sector": sector,
            "N_Dates": len(aligned),
            "N_Indicators": len(indicators),
            "SPA_PValue_Consistent": np.nan,
            "SPA_PValue_Lower": np.nan,
            "SPA_PValue_Upper": np.nan,
            "Best_Indicator": "n/a",
            "Best_Mean_Loss_Diff": np.nan,
            "SPA_Significant_5pct": False
        })
        continue

    benchmark_loss = aligned[
        "Baseline"
    ].to_numpy()
    candidate_loss = aligned[
        indicators
    ].to_numpy()

    spa = SPA(
        benchmark_loss,
        candidate_loss,
        block_size=BLOCK_LENGTH,
        reps=2000,
        bootstrap="stationary",
        studentize=True,
        nested=False,
        seed=SEED
    )
    spa.compute()

    pvalues = spa.pvalues

    # arch returns lower / consistent / upper p-values.
    p_lower = float(
        pvalues.loc["lower"]
    )
    p_consistent = float(
        pvalues.loc["consistent"]
    )
    p_upper = float(
        pvalues.loc["upper"]
    )

    mean_diffs = (
        benchmark_loss[:, None] -
        candidate_loss
    ).mean(axis=0)

    best_idx = int(
        np.argmax(mean_diffs)
    )

    spa_rows.append({
        "Sector": sector,
        "N_Dates": len(aligned),
        "N_Indicators": len(indicators),
        "SPA_PValue_Consistent": p_consistent,
        "SPA_PValue_Lower": p_lower,
        "SPA_PValue_Upper": p_upper,
        "Best_Indicator": indicators[
            best_idx
        ],
        "Best_Mean_Loss_Diff": float(
            mean_diffs[best_idx]
        ),
        "SPA_Significant_5pct": bool(
            p_consistent < 0.05
        )
    })

spa_results = pd.DataFrame(
    spa_rows
)

# ================================================================
# 6. OPTIONAL CANONICAL MCS — APPENDIX ONLY
# ================================================================
mcs_results = pd.DataFrame()

if RUN_MCS_APPENDIX:
    mcs_rows = []

    for sector in all_sectors:
        sec = loss_df[
            loss_df["Sector"] == sector
        ].copy()

        indicators = sorted(
            sec["Indicator"].unique()
        )

        benchmark = sec[
            sec["Indicator"] == indicators[0]
        ][["Date", "Base_QLIKE"]].set_index(
            "Date"
        ).rename(
            columns={"Base_QLIKE": "Baseline"}
        )

        candidate_series = []
        for indicator in indicators:
            s = sec[
                sec["Indicator"] == indicator
            ][["Date", "Enh_QLIKE"]].set_index(
                "Date"
            ).rename(
                columns={"Enh_QLIKE": indicator}
            )
            candidate_series.append(s)

        losses = benchmark.join(
            pd.concat(
                candidate_series,
                axis=1,
                join="inner"
            ),
            how="inner"
        ).dropna()

        if len(losses) < 10:
            continue

        mcs = MCS(
            losses,
            size=0.10,
            reps=2000,
            block_size=BLOCK_LENGTH,
            method="R",
            bootstrap="stationary",
            seed=SEED
        )
        mcs.compute()

        included = list(
            mcs.included
        )

        mcs_rows.append({
            "Sector": sector,
            "N_Dates": len(losses),
            "N_In_MCS": len(included),
            "Baseline_In_MCS": (
                "Baseline" in included
            ),
            "MCS_Models": ", ".join(
                map(str, included)
            )
        })

    mcs_results = pd.DataFrame(
        mcs_rows
    )

# ================================================================
# 7. SAVE RESULTS
# ================================================================
strategy_long.to_csv(
    "Final_Strategy_DateLevel_Losses.csv",
    index=False
)
sector_results.to_csv(
    "Final_Strategy_Sector_Comparison.csv",
    index=False
)
market_summary.to_csv(
    "Final_Strategy_Market_Comparison.csv",
    index=False
)
spa_results.to_csv(
    "Hansen_SPA_Results.csv",
    index=False
)

if RUN_MCS_APPENDIX and len(mcs_results) > 0:
    mcs_results.to_csv(
        "MCS_Appendix_Results.csv",
        index=False
    )

# ================================================================
# 8. PRINT REVIEWER-READY OUTPUT
# ================================================================
print("\n" + "=" * 100)
print("A. PRIMARY MARKET-WIDE RESULT")
print("=" * 100)

r = market_summary.iloc[0]

print(
    "Aggregation:                  "
    f"{r['Aggregation_Method']}"
)
print(
    "Sector test observations:      "
    f"{int(r['Min_Sector_Test_N'])}–"
    f"{int(r['Max_Sector_Test_N'])} per sector"
)
print(f"Sectors:                       {int(r['N_Sectors'])}")
print(f"Enhanced sectors selected:     {int(r['N_Enhanced_Selected'])}")
print(f"Baseline weighted QLIKE:        {r['Baseline_Weighted_QLIKE']:.6f}")
print(f"Final strategy weighted QLIKE:  {r['Final_Strategy_Weighted_QLIKE']:.6f}")
print(f"Improvement:                    {r['Improvement_Pct']:+.2f}%")
print(f"Mean loss difference:           {r['Mean_Loss_Diff']:+.6f}")
print(
    "95% block-bootstrap CI:       "
    f"[{r['Bootstrap_CI_Lower']:+.6f}, "
    f"{r['Bootstrap_CI_Upper']:+.6f}]"
)
if pd.notna(r["DM_PValue_TwoSided"]):
    print(
        "Aggregate selected-sector DM: "
        f"{r['DM_Stat_HLN']:.4f}"
    )
    print(
        "Aggregate DM p-value:          "
        f"{r['DM_PValue_TwoSided']:.4f}"
    )
    print(
        "Common dates for aggregate DM: "
        f"{int(r['DM_N_Common_Selected_Sectors'])}"
    )
else:
    print("Aggregate selected-sector DM: Not run")

print(
    "DM note:                       "
    f"{r['DM_Note']}"
)
print(f"Bootstrap one-sided p-value:    {r['Bootstrap_OneSided_P']:.4f}")

if (
    r["Mean_Loss_Diff"] > 0
    and r["Bootstrap_CI_Lower"] > 0
):
    primary_conclusion = (
        "Technical indicators improve the final hybrid strategy."
    )
elif (
    r["Mean_Loss_Diff"] < 0
    and r["Bootstrap_CI_Upper"] < 0
):
    primary_conclusion = (
        "Technical indicators significantly worsen the final hybrid strategy."
    )
else:
    primary_conclusion = (
        "No statistically robust market-wide improvement from technical indicators."
    )

print(f"\nPrimary conclusion: {primary_conclusion}")

print("\n" + "=" * 100)
print("B. SECTOR-LEVEL LOCKED-STRATEGY RESULTS")
print("=" * 100)

display_cols = [
    "Sector",
    "Model_Used",
    "N_Test",
    "Baseline_QLIKE",
    "Final_Strategy_QLIKE",
    "Improvement_Pct",
    "DM_PValue",
    "DM_PValue_Holm"
]

sector_display = sector_results[
    display_cols
].sort_values("Sector").copy()

sector_display["DM_PValue"] = sector_display.apply(
    lambda row: (
        "Not tested"
        if not row["Model_Used"].startswith("Sector_")
        else (
            "n/a"
            if pd.isna(row["DM_PValue"])
            else f"{row['DM_PValue']:.4f}"
        )
    ),
    axis=1
)

sector_display["DM_PValue_Holm"] = sector_display.apply(
    lambda row: (
        "Not tested"
        if not row["Model_Used"].startswith("Sector_")
        else (
            "n/a"
            if pd.isna(row["DM_PValue_Holm"])
            else f"{row['DM_PValue_Holm']:.4f}"
        )
    ),
    axis=1
)

print(
    sector_display.to_string(
        index=False,
        formatters={
            "Baseline_QLIKE": "{:.6f}".format,
            "Final_Strategy_QLIKE": "{:.6f}".format,
            "Improvement_Pct": "{:+.2f}".format
        }
    )
)

print("\n" + "=" * 100)
print("C. SECONDARY HANSEN SPA RESULTS")
print("=" * 100)
print(
    spa_results[
        [
            "Sector",
            "N_Dates",
            "N_Indicators",
            "SPA_PValue_Consistent",
            "Best_Indicator",
            "Best_Mean_Loss_Diff"
        ]
    ].to_string(
        index=False,
        formatters={
            "SPA_PValue_Consistent": (
                lambda x: "n/a"
                if pd.isna(x)
                else f"{x:.4f}"
            ),
            "Best_Mean_Loss_Diff": (
                lambda x: "n/a"
                if pd.isna(x)
                else f"{x:+.6f}"
            )
        }
    )
)

print(
    "\nInterpretation: SPA tests whether at least one of the "
    "16 indicators beats the baseline after data-snooping correction."
)

print("\nSaved:")
print(" - Final_Strategy_DateLevel_Losses.csv")
print(" - Final_Strategy_Sector_Comparison.csv")
print(" - Final_Strategy_Market_Comparison.csv")
print(" - Hansen_SPA_Results.csv")

if RUN_MCS_APPENDIX and len(mcs_results) > 0:
    print(" - MCS_Appendix_Results.csv")

# ================================================================
# 9. DOWNLOAD RESULTS IN GOOGLE COLAB
# ================================================================
try:
    from google.colab import files as colab_files

    download_files = [
        "Final_Strategy_DateLevel_Losses.csv",
        "Final_Strategy_Sector_Comparison.csv",
        "Final_Strategy_Market_Comparison.csv",
        "Hansen_SPA_Results.csv"
    ]

    if RUN_MCS_APPENDIX and len(mcs_results) > 0:
        download_files.append(
            "MCS_Appendix_Results.csv"
        )

    print("\nDownloading result files...")
    for file_name in download_files:
        colab_files.download(
            file_name
        )
        time.sleep(1.0)

except Exception as exc:
    print(
        "\nColab downloads skipped: "
        f"{exc}"
    )

print("\n✅ Reviewer-focused comparison complete")
print("=" * 100)


REVIEWER-FOCUSED FORECAST COMPARISON
Question: Do technical indicators improve the hybrid volatility forecast?
Loaded 5744 date-level loss observations
Sectors: 10
Indicator candidates: 16
Enhanced models selected by CV gate: 9

A. PRIMARY MARKET-WIDE RESULT
Aggregation:                  Fixed-weight sector means with stratified moving-block bootstrap
Sector test observations:      17–38 per sector
Sectors:                       10
Enhanced sectors selected:     9
Baseline weighted QLIKE:        1.831632
Final strategy weighted QLIKE:  1.914409
Improvement:                    -4.52%
Mean loss difference:           -0.082778
95% block-bootstrap CI:       [-0.218087, +0.041109]
Aggregate selected-sector DM: Not run
DM note:                       Not run: fewer than 10 common dates across selected sectors.
Bootstrap one-sided p-value:    0.8958

Primary conclusion: No statistically robust market-wide improvement from technical indicators.

B. SECTOR-LEVEL LOCKED-STRATEGY RESULTS
         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Reviewer-focused comparison complete


In [ ]:
# ================================================================
# PER-SECTOR FINAL-STRATEGY VARIANCE FORECAST GRAPHS
# Corrected constrained-convex baseline + technical indicators
# ================================================================

from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------------
# Configuration
# ------------------------------------------------
EPSILON = 1e-8
SHOW_IN_NOTEBOOK = True
DOWNLOAD_ZIP = True

OUTPUT_DIR = Path("Sector_Final_Strategy_Forecast_Graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "enhanced_test": Path(
        "Enhanced_Stacking_CompTest_Final_Test_Matched.csv"
    ),
    "best_models": Path(
        "Best_Enhanced_Stacking_Transformer_TGARCH_Per_Sector.csv"
    ),
    "baseline_forecasts": Path(
        "Stacking_Transformer_TGARCH_Forecasts.csv"
    ),
    "baseline_metrics": Path(
        "Stacking_Transformer_TGARCH_Sector_Metrics.csv"
    ),
}

# ------------------------------------------------
# Validate input files
# ------------------------------------------------
missing_files = [
    str(path) for path in FILES.values() if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
        + "\nRun the corrected baseline and enhanced codes first."
    )


def require_columns(df, required, file_name):
    """Check that a dataframe contains the required columns."""
    missing = [column for column in required if column not in df.columns]

    if missing:
        raise ValueError(
            f"{file_name} is missing these columns: {missing}"
        )


def qlike_loss(actual, forecast, epsilon=EPSILON):
    """Calculate average QLIKE loss for variance forecasts."""
    actual = np.clip(
        np.asarray(actual, dtype=float), epsilon, None
    )
    forecast = np.clip(
        np.asarray(forecast, dtype=float), epsilon, None
    )

    ratio = actual / forecast
    return float(np.mean(ratio - np.log(ratio) - 1.0))


def safe_filename(text):
    """Create a filename-safe sector name."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(text)).strip("_")


# ------------------------------------------------
# Load corrected outputs
# ------------------------------------------------
enhanced_test = pd.read_csv(FILES["enhanced_test"])
best_models = pd.read_csv(FILES["best_models"])
baseline_forecasts = pd.read_csv(FILES["baseline_forecasts"])
baseline_metrics = pd.read_csv(FILES["baseline_metrics"])

require_columns(
    enhanced_test,
    [
        "Date",
        "Sector",
        "Indicator",
        "Actual_Variance",
        "Enhanced_Hybrid_Variance",
    ],
    FILES["enhanced_test"].name,
)

require_columns(
    best_models,
    [
        "Sector",
        "Indicator",
        "Weight",
        "Learned_Weight_DL",
        "Learned_Weight_GARCH",
        "Indicator_Beta",
    ],
    FILES["best_models"].name,
)

require_columns(
    baseline_forecasts,
    [
        "Date",
        "Sector",
        "Actual_Variance",
        "Hybrid_Forecast",
        "Evaluation_Split",
    ],
    FILES["baseline_forecasts"].name,
)

require_columns(
    baseline_metrics,
    [
        "Sector",
        "Weight",
        "Norm_Eff_Coef_Share_DL",
        "Norm_Eff_Coef_Share_GARCH",
    ],
    FILES["baseline_metrics"].name,
)

# ------------------------------------------------
# Prepare dates
# ------------------------------------------------
enhanced_test["Date"] = pd.to_datetime(
    enhanced_test["Date"], errors="coerce"
)

baseline_forecasts["Date"] = pd.to_datetime(
    baseline_forecasts["Date"], errors="coerce"
)

enhanced_test = enhanced_test.dropna(
    subset=["Date", "Sector"]
).copy()

baseline_forecasts = baseline_forecasts.dropna(
    subset=["Date", "Sector"]
).copy()

# Keep the locked baseline final-test observations.
baseline_final = baseline_forecasts.loc[
    baseline_forecasts["Evaluation_Split"].eq("Final_Test")
].copy()

if baseline_final.empty:
    raise ValueError(
        "No rows labelled 'Final_Test' were found in the baseline "
        "forecast file."
    )

# ------------------------------------------------
# Match enhanced forecasts with baseline forecasts
# on exact sector and date
# ------------------------------------------------
matched_enhanced = enhanced_test.merge(
    baseline_final[
        [
            "Date",
            "Sector",
            "Actual_Variance",
            "Hybrid_Forecast",
        ]
    ],
    on=["Date", "Sector"],
    how="left",
    suffixes=("", "_Baseline_File"),
    validate="many_to_one",
)

if matched_enhanced["Hybrid_Forecast"].isna().any():
    missing_matches = matched_enhanced.loc[
        matched_enhanced["Hybrid_Forecast"].isna(),
        ["Sector", "Date"],
    ]

    raise ValueError(
        "Some enhanced observations have no exact baseline match:\n"
        f"{missing_matches.head(10)}"
    )

# Confirm that both files use the same observed target.
target_match = np.allclose(
    matched_enhanced["Actual_Variance"].to_numpy(dtype=float),
    matched_enhanced[
        "Actual_Variance_Baseline_File"
    ].to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-12,
    equal_nan=False,
)

if not target_match:
    raise ValueError(
        "The baseline and enhanced files do not contain the same "
        "observed variance target on matched dates."
    )

matched_enhanced = matched_enhanced.rename(
    columns={
        "Hybrid_Forecast": "Baseline_Hybrid_Variance"
    }
)

# ------------------------------------------------
# Establish sector order
# ------------------------------------------------
baseline_metrics = baseline_metrics.sort_values(
    "Weight", ascending=False
).reset_index(drop=True)

sector_order = baseline_metrics["Sector"].tolist()
selected_sectors = set(best_models["Sector"])

summary_rows = []
generated_files = []

# ------------------------------------------------
# Generate one graph for each sector
# ------------------------------------------------
for sector in sector_order:

    baseline_row = baseline_metrics.loc[
        baseline_metrics["Sector"].eq(sector)
    ].iloc[0]

    market_weight = float(baseline_row["Weight"])

    if sector in selected_sectors:
        # Selected enhanced model: use identical matched dates.
        plot_data = matched_enhanced.loc[
            matched_enhanced["Sector"].eq(sector),
            [
                "Date",
                "Actual_Variance",
                "Baseline_Hybrid_Variance",
                "Enhanced_Hybrid_Variance",
            ],
        ].copy()

        selected_row = best_models.loc[
            best_models["Sector"].eq(sector)
        ].iloc[0]

        indicator = str(selected_row["Indicator"])
        weight_dl = float(selected_row["Learned_Weight_DL"])
        weight_tgarch = float(
            selected_row["Learned_Weight_GARCH"]
        )
        beta = float(selected_row["Indicator_Beta"])
        final_model = "Indicator-adjusted final strategy"

        plot_data = plot_data.rename(
            columns={
                "Enhanced_Hybrid_Variance":
                    "Final_Strategy_Variance"
            }
        )

    else:
        # No indicator passed the gate: retain the baseline.
        plot_data = baseline_final.loc[
            baseline_final["Sector"].eq(sector),
            [
                "Date",
                "Actual_Variance",
                "Hybrid_Forecast",
            ],
        ].copy()

        plot_data = plot_data.rename(
            columns={
                "Hybrid_Forecast":
                    "Baseline_Hybrid_Variance"
            }
        )

        plot_data["Final_Strategy_Variance"] = (
            plot_data["Baseline_Hybrid_Variance"]
        )

        indicator = "None—baseline retained"
        weight_dl = float(
            baseline_row["Norm_Eff_Coef_Share_DL"]
        )
        weight_tgarch = float(
            baseline_row["Norm_Eff_Coef_Share_GARCH"]
        )
        beta = np.nan
        final_model = "Constrained-convex baseline retained"

    plot_data = (
        plot_data
        .dropna(
            subset=[
                "Date",
                "Actual_Variance",
                "Baseline_Hybrid_Variance",
                "Final_Strategy_Variance",
            ]
        )
        .sort_values("Date")
        .reset_index(drop=True)
    )

    if plot_data.empty:
        print(f"Skipping {sector}: no valid observations.")
        continue

    # Metrics are calculated from the exact observations shown.
    baseline_qlike = qlike_loss(
        plot_data["Actual_Variance"],
        plot_data["Baseline_Hybrid_Variance"],
    )

    final_qlike = qlike_loss(
        plot_data["Actual_Variance"],
        plot_data["Final_Strategy_Variance"],
    )

    improvement = (
        (baseline_qlike - final_qlike)
        / baseline_qlike
        * 100.0
        if baseline_qlike > 0
        else np.nan
    )

    # ------------------------------------------------------------
    # Plot variance directly.
    # Do not take square roots because the model target is variance.
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 6.5))

    ax.plot(
        plot_data["Date"],
        plot_data["Actual_Variance"],
        linestyle="-",
        marker="o",
        markersize=3.5,
        linewidth=1.6,
        label="Observed one-day squared-return variance proxy",
    )

    ax.plot(
        plot_data["Date"],
        plot_data["Baseline_Hybrid_Variance"],
        linestyle="--",
        linewidth=2.0,
        label="Constrained-convex baseline",
    )

    final_label = (
        f"Final strategy ({indicator})"
        if sector in selected_sectors
        else "Final strategy (baseline retained)"
    )

    ax.plot(
        plot_data["Date"],
        plot_data["Final_Strategy_Variance"],
        linestyle="-.",
        linewidth=2.0,
        label=final_label,
    )

    # Main and secondary titles
    ax.set_title(
        f"One-Day Squared-Return Variance Forecasts: {sector}",
        fontsize=14,
        fontweight="bold",
        pad=30,
    )

    beta_text = (
        f" | β={beta:.2f}"
        if np.isfinite(beta)
        else ""
    )

    ax.text(
        0.5,
        1.025,
        (
            f"Market weight={market_weight:.4f} | "
            f"Indicator={indicator} | "
            f"w_DL={weight_dl:.2f} | "
            f"w_TGARCH-X={weight_tgarch:.2f}"
            f"{beta_text} | "
            f"QLIKE improvement={improvement:+.2f}%"
        ),
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=10.5,
        fontweight="bold",
    )

    # Compact statistics box
    statistics_text = (
        f"Baseline QLIKE: {baseline_qlike:.4f}\n"
        f"Final QLIKE: {final_qlike:.4f}\n"
        f"Observations: {len(plot_data)}\n"
        f"Model: {final_model}"
    )

    ax.text(
        0.015,
        0.97,
        statistics_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        bbox={
            "boxstyle": "round,pad=0.4",
            "facecolor": "white",
            "alpha": 0.85,
        },
    )

    ax.set_xlabel("Date", fontsize=11, fontweight="bold")
    ax.set_ylabel(
        "Variance (squared return)",
        fontsize=11,
        fontweight="bold",
    )

    locator = mdates.AutoDateLocator(
        minticks=5,
        maxticks=10,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    ax.grid(True, linestyle="--", alpha=0.25)
    ax.legend(loc="upper right", fontsize=9)
    ax.margins(x=0.01)

    fig.tight_layout()

    output_file = OUTPUT_DIR / (
        f"{safe_filename(sector)}_Final_Strategy_"
        f"Variance_Forecast.png"
    )

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    generated_files.append(str(output_file))

    if SHOW_IN_NOTEBOOK:
        plt.show()

    plt.close(fig)

    summary_rows.append(
        {
            "Sector": sector,
            "Market_Weight": market_weight,
            "Final_Model": final_model,
            "Selected_Indicator": indicator,
            "Weight_DL": weight_dl,
            "Weight_TGARCH_X": weight_tgarch,
            "Indicator_Beta": beta,
            "Baseline_QLIKE_Plot_Dates": baseline_qlike,
            "Final_QLIKE_Plot_Dates": final_qlike,
            "QLIKE_Improvement_%": improvement,
            "N_Observations": len(plot_data),
            "Graph_File": output_file.name,
        }
    )

# ------------------------------------------------
# Save graph summary
# ------------------------------------------------
graph_summary = pd.DataFrame(summary_rows)

summary_file = OUTPUT_DIR / "Sector_Graph_Metric_Summary.csv"
graph_summary.to_csv(summary_file, index=False)

print("\nGenerated sector graphs:")
for file_name in generated_files:
    print(f"  - {file_name}")

print(f"\nSummary saved to: {summary_file}")

# ------------------------------------------------
# Create ZIP archive
# ------------------------------------------------
zip_file = shutil.make_archive(
    base_name=str(OUTPUT_DIR),
    format="zip",
    root_dir=OUTPUT_DIR,
)

print(f"ZIP archive created: {zip_file}")

# Optional Colab download
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(zip_file)
    except ImportError:
        print(
            "Automatic download is available only in Google Colab."
        )


Generated sector graphs:
  - Sector_Final_Strategy_Forecast_Graphs/Banking_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Telecommunication_and_Technology_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Manufacturing_and_Allied_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Energy_and_Petroleum_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Insurance_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Construction_and_Allied_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Agriculture_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Commercial_and_services_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Investment_Final_Strategy_Variance_Forecast.png
  - Sector_Final_Strategy_Forecast_Graphs/Automobiles_and_Accessories_Final_Strategy_Variance_F

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>